# CASMI 2026 — specs_scratch R0 (library entropy + ICEBERG, own code)

**Generated by `scratch_pkg/tools/mk_notebook.py` from `scratch_pkg/drivers/r0_kaggle.py` — do not hand-edit.**
Spec: `specs/specs_scratch/`. The `casmi_scratch` package is embedded below and written to disk at start-up.

Settings: **GPU T4**, **Internet OFF**. Inputs: the competition data, **casmi26-iceberg** (the ICEBERG package of
the v18 zenith notebook; `casmi26-iceberg-offline` still works as a fallback and, if attached as well, its manifest
checks the checkpoint SHA1s), **eveliaveldrine/rdkit-wheels** (RDKit built for the image's Python), and the COCONUT
arrays (`coco_meta.pkl`, `coco_mass.npy`). With `pool_source="r1"` (R1a / R1b): **casmi26-scratch-pool** instead
of the COCONUT arrays. With `sim_model="glacier"` (the R1 GLACIER twins): also **casmi26-glacier**; GLACIER replaces
ICEBERG and simulates `[M+H]+` / `[M+Na]+` only (casmi26-iceberg stays attached: its wheels build the RDKit 2025.03 site).
With `class_vote=True` (R2): **casmi26-scratch-pool v2** (it carries `pool_npc.parquet`, NPClassifier labels).
With `fp_channel="own"` (R3): our FPNet export (e.g. **casmi-fpnet**: fp_single_s0/s1, fp_merged_m0/m1, fp_bits_sha1.txt) and the COCONUT arrays
dataset that holds `fp_bits.npy` + `coco_fp.npy` (6,930 bits; chosen by the checkpoints' sha1). With
`fp_channel="ahmed"`: **casmi26-fpnet-full1** (`fpnet_full1.pt`) and **casmi26-v2-pool** (`fp_bits.npy` 10,226 bits,
`pool_meta.parquet` + `pool_fp.npy` for the featuriser parity check). With `fp_banks=("ahmed", "prv")` (R4 family
fusion): the full1 inputs above plus **prvsiyan/casmi26-fp-models-v2** (`fp_single_s2.pt`, `fp_merged_m1.pt`) and the
COCONUT arrays dataset (**prvsiyan/coconut-casmi26-candidates**: `fp_bits.npy` 6,930 bits next to `coco_fp.npy`).
With `analog_channel="pssm"` / `"tani"` (R5, `specs/specs_scratch/pssm.md`): no extra dataset; analogue structures
come from train.parquet, and the PSSM background q is sampled from the pool at start-up unless an `analog_q.npy` is
attached.

R0 = the Track 3 simple-sim base (LB 0.289; 0.296 with MAX_PEAKS=100) ported with fixes: one `max_peaks` for the
library, MetFrag-lite and ICEBERG similarities; no globals; fast commit runs 20 molecules deep.

In [ ]:
CONFIG_OVERRIDES = dict(pool_source='r1', pool_drop_e180=True, sim_model='glacier', class_vote=True)          # e.g. dict(use_sim=False) for the no-simulation control
RUN_TAG = 'r2_gl'

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
INPUT = os.environ.get("CASMI_INPUT", "/kaggle/input")    # overridable for a local dry run
WORK = Path(os.environ.get("CASMI_WORK", "/kaggle/working"))
PKG_FILES = {'casmi_scratch/__init__.py': '"""casmi_scratch: our own CASMI 2026 pipeline (specs/specs_scratch).\n\nPure-python modules (numpy / pandas / pyarrow / numba; RDKit where a module says so). The ICEBERG model runs in\nits own venv through `resources/iceberg_worker.py`; this package only talks to it through files.\n"""\n__version__ = "0.1.0"\n', 'casmi_scratch/analog.py': '"""Analog library search with a shift-aware entropy similarity, and the NP class vote (R1 CV diagnostic, R2 candidate).\n\nA query peak matches a library peak directly (|dmz| <= tol) or shifted by the precursor difference\n(|q - (c + shift)| <= tol, shift = query neutral mass - library neutral mass): the GNPS modified-cosine idea applied\nto our entropy similarity. Pairs are assigned one-to-one, greedily by intensity product.\n\nTwo candidate sets (gate CV in specs/specs_scratch/gate_report.md):\n- "delta": library spectra at the query mass or one NP biosynthetic delta away (each +-ppm);\n- "open":  library spectra within +-200 Da.\n"""\nimport numpy as np\nfrom numba import njit, prange\n\nfrom .kernels import clean\nfrom .spectra import clean_spectrum\n\n# NP biosynthetic mass deltas (Da), fixed before the CV measurement\nNP_DELTAS = {\n    "CH2": 14.015650, "O": 15.994915, "H2O": 18.010565, "CH2O": 30.010565, "C2H2O acetyl": 42.010565,\n    "CO2": 43.989829, "C5H8 prenyl": 68.062600, "C3H2O3 malonyl": 86.000394, "C5H8O4 pentose": 132.042259,\n    "C9H6O2 coumaroyl": 146.036779, "C6H10O4 deoxyhexose": 146.057909, "C6H10O5 hexose": 162.052824,\n    "C6H8O6 glucuronide": 176.032088,\n}\nOPEN_DA = 200.0\n\n\n@njit(cache=True, fastmath=True)\ndef shift_entropy_sim(qmz, qp, cmz, cp, shift, tol):\n    """Entropy similarity with direct and precursor-shifted peak matches (both inputs cleaned, m/z ascending)."""\n    n = len(qmz)\n    m = len(cmz)\n    if n == 0 or m == 0:\n        return 0.0\n    use_shift = abs(shift) > tol\n    cap = 4 * (n + m) + 16\n    pi = np.empty(cap, np.int64)\n    pj = np.empty(cap, np.int64)\n    pw = np.empty(cap, np.float64)\n    k = 0\n    for pas in range(2 if use_shift else 1):\n        off = 0.0 if pas == 0 else shift\n        lo = 0\n        for i in range(n):\n            while lo < m and cmz[lo] + off < qmz[i] - tol:\n                lo += 1\n            j = lo\n            while j < m and cmz[j] + off <= qmz[i] + tol:\n                if k >= cap:\n                    break\n                pi[k] = i\n                pj[k] = j\n                # direct pairs win ties against shifted ones\n                pw[k] = qp[i] * cp[j] * (1.0 + 1e-6 if pas == 0 else 1.0)\n                k += 1\n                j += 1\n    uq = np.zeros(n, np.bool_)\n    uc = np.zeros(m, np.bool_)\n    mq = np.zeros(n, np.float64)        # matched partner intensity added to the query peak\n    if k > 0:\n        o = np.argsort(-pw[:k])\n        for t in range(k):\n            i = pi[o[t]]\n            j = pj[o[t]]\n            if uq[i] or uc[j]:\n                continue\n            uq[i] = True\n            uc[j] = True\n            mq[i] = cp[j]\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0:\n            SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0:\n            SB -= cp[x] * np.log(cp[x])\n    tot = 0.0\n    for x in range(n):\n        tot += qp[x] + mq[x]\n    for x in range(m):\n        if not uc[x]:\n            tot += cp[x]\n    if tot <= 0:\n        return 0.0\n    SAB = 0.0\n    for x in range(n):\n        v = (qp[x] + mq[x]) / tot\n        if v > 0:\n            SAB -= v * np.log(v)\n    for x in range(m):\n        if not uc[x]:\n            v = cp[x] / tot\n            if v > 0:\n                SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift(qmz, qp, qnm, cand, off, allmz, allin, lib_nm, tol, floor, topk, power, ent_weight):\n    """Shift-aware entropy similarity of one cleaned query (neutral mass qnm) against library spectra `cand`."""\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]\n        a = off[c]\n        b = off[c + 1]\n        if b <= a:\n            continue\n        cm, cp = clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0:\n            continue\n        out[k] = shift_entropy_sim(qmz, qp, cm, cp, qnm - lib_nm[c], tol)\n    return out\n\n\ndef candidates(lib, target, mode, ppm=10.0, open_da=OPEN_DA):\n    """Library spectrum indices for an analog search around target (\'delta\' or \'open\')."""\n    if mode == "open":\n        return lib.window(target, open_da)\n    assert mode == "delta", mode\n    parts = [lib.window(target, target * ppm / 1e6)]\n    for d in NP_DELTAS.values():\n        for t in (target - d, target + d):\n            parts.append(lib.window(t, t * ppm / 1e6))\n    return np.unique(np.concatenate(parts)) if parts else np.zeros(0, np.int64)\n\n\ndef analog_hits(lib, specs, target, cfg, mode, polarity=0, exclude=None, ppm=10.0, open_da=OPEN_DA):\n    """Best shift-aware similarity per library ik over a molecule\'s spectra -> {ik: sim}.\n\n    polarity: +1 / -1 keeps library spectra of that polarity only (0 = all). exclude: boolean mask over library\n    spectra (the own-structure ban).\n    """\n    cand = candidates(lib, target, mode, ppm, open_da)\n    if polarity and lib.pol is not None and len(cand):\n        cand = cand[lib.pol[cand] == polarity]\n    if exclude is not None and len(cand):\n        cand = cand[~exclude[cand]]\n    if len(cand) == 0:\n        return {}\n    agg = {}\n    for mz, it in specs:\n        qm, qp = clean_spectrum(mz, it, cfg)\n        if len(qm) == 0:\n            continue\n        sc = search_shift(qm, qp, float(target), cand, lib.off, lib.mz, lib.it, lib.nm, cfg.mz_tol, cfg.int_floor,\n                          cfg.max_peaks, cfg.int_power, cfg.ent_weight)\n        for c, s in zip(cand, sc):\n            kk = lib.ik[c]\n            if s > agg.get(kk, -1.0):\n                agg[kk] = float(s)\n    return agg\n\n\n# ---------------------------------------------------------------- class vote\n\ndef class_vote(hits, label_of, k=10, min_sim=0.5):\n    """Top-k hits with sim >= min_sim vote their structure\'s label, weighted by similarity; unlabelled hits abstain.\n\n    hits: {key: sim}; label_of: key -> label or None. -> (winning label or None, {label: weight}, n voters).\n    """\n    top = sorted(((s, key) for key, s in hits.items() if s >= min_sim), key=lambda x: (-x[0], x[1]))[:k]\n    w = {}\n    for s, key in top:\n        lab = label_of(key)\n        if isinstance(lab, str) and lab:\n            w[lab] = w.get(lab, 0.0) + s\n    if not w:\n        return None, {}, 0\n    win = sorted(w.items(), key=lambda x: (-x[1], x[0]))[0][0]\n    return win, w, sum(1 for s, key in top if isinstance(label_of(key), str) and label_of(key))\n\n\ndef vote_conf(w, win):\n    """Share of the voting weight that went to the winning label (0 when nothing voted)."""\n    tot = sum(w.values())\n    return float(w.get(win, 0.0) / tot) if tot > 0 and win is not None else 0.0\n\n\ndef vote_bonus(sc, sp, names, voted, conf, beta):\n    """R2 tier-2 bonus per candidate: beta * conf * P(candidate has the voted superclass).\n\n    sc / sp: the candidates\' NPClassifier top-k superclass codes (-1 = none) and probabilities. P is the probability\n    of the voted superclass among the candidate\'s top k (0 if absent).\n    """\n    sc, sp = np.atleast_2d(sc), np.atleast_2d(sp)\n    if voted is None or voted not in names or conf <= 0 or len(sc) == 0:\n        return np.zeros(len(sc), np.float32)\n    p = np.where(sc == names.index(voted), sp, 0.0).sum(axis=1)\n    return (beta * conf * p).astype(np.float32)\n\n\ndef label_match(pred, truth):\n    """A truth label may hold several NPClassifier values (\'A|B\' or \'A; B\'): a hit on any of them counts."""\n    if not isinstance(pred, str) or not isinstance(truth, str):\n        return False\n    parts = {p.strip() for p in truth.replace(";", "|").split("|")}\n    return pred.strip() in parts or pred == truth\n\n\ndef vote_report(preds, truths):\n    """Per-molecule predicted / truth labels -> coverage, purity, majority baseline (molecules with a truth label).\n\n    coverage = share with a vote; purity = share of voted molecules whose vote matches; baseline = share of voted\n    molecules carrying the most frequent truth label (always guessing it).\n    """\n    rows = [(p, t) for p, t in zip(preds, truths) if isinstance(t, str) and t]\n    n = len(rows)\n    voted = [(p, t) for p, t in rows if isinstance(p, str)]\n    if not voted:\n        return dict(n_with_truth=n, n_voted=0, coverage=0.0 if n else float("nan"), purity=float("nan"),\n                    baseline=float("nan"), majority=None)\n    first = [t.replace(";", "|").split("|")[0].strip() for _, t in voted]\n    vals, cnt = np.unique(first, return_counts=True)\n    maj = vals[cnt.argmax()]\n    return dict(n_with_truth=n, n_voted=len(voted), coverage=len(voted) / n,\n                purity=float(np.mean([label_match(p, t) for p, t in voted])),\n                baseline=float(np.mean([label_match(maj, t) for _, t in voted])), majority=str(maj))\n\n\n# ---------------------------------------------------------------- R5 analogue channel (PSSM / Tanimoto kernel)\n# Open (+-200 Da) analogue hits -> their structures\' raw Family-A bits -> one tier-2 RRF list over the candidates.\n# Design: specs/specs_scratch/pssm.md. Train structures are not scored by this list (they get the neutral term).\nMORGAN_R2 = 4096                    # the raw Family-A layout starts with Morgan r2 4096 (fpnet.Featuriser)\n\n\ndef molecule_analogs(lib, specs, pols, target, cfg, exclude=None):\n    """Open-window analogue hits of one molecule, each spectrum searched within its own polarity -> {key: best sim}."""\n    out = {}\n    for pol in sorted(set(pols)):\n        sp = [s for s, p in zip(specs, pols) if p == pol]\n        for k, s in analog_hits(lib, sp, target, cfg, "open", polarity=pol, exclude=exclude,\n                                open_da=cfg.analog_open_da).items():\n            if s > out.get(k, -1.0):\n                out[k] = s\n    return out\n\n\ndef analog_set(hits, s_min, m):\n    """{key: sim} -> the top m [(key, sim)] with sim >= s_min, best first (ties by key)."""\n    top = sorted(((s, k) for k, s in hits.items() if s >= s_min), key=lambda x: (-x[0], str(x[1])))[:m]\n    return [(k, s) for s, k in top]\n\n\ndef _logit(x):\n    return np.log(x) - np.log1p(-x)\n\n\ndef pssm_weights(X, S, q, alpha=1.0, clip=3.0, q_floor=1e-5):\n    """Spectrally weighted log-odds profile. X (m, D) analogue bits, S (m,) similarities, q (D,) background bit\n    frequency. p~ = (sum S^4 x + alpha q) / (sum S^4 + alpha) (prior = alpha background analogues, so a bit no\n    analogue carries stays near q); W = clip(logit p~ - logit q, +-clip)."""\n    w = np.asarray(S, np.float64) ** 4\n    q = np.clip(np.asarray(q, np.float64), q_floor, 1.0 - q_floor)\n    p = (w @ np.asarray(X, np.float64) + alpha * q) / (w.sum() + alpha)\n    return np.clip(_logit(p) - _logit(q), -clip, clip).astype(np.float32)\n\n\ndef tani_kernel(C, X, S):\n    """sum_i S_i^4 T(c, a_i) / sum_i S_i^4 per candidate. C (n, D), X (m, D) binary."""\n    C, X = np.asarray(C, np.float32), np.asarray(X, np.float32)\n    inter = C @ X.T\n    union = C.sum(1)[:, None] + X.sum(1)[None, :] - inter\n    T = np.where(union > 0, inter / np.maximum(union, 1.0), 0.0)\n    w = np.asarray(S, np.float64) ** 4\n    return (T @ w / w.sum()).astype(np.float32)\n\n\ndef train_smiles(train_path, keymap=None):\n    """Library structure key (train inchikey14 mapped through keymap) -> normalized_smiles (first row per ik14)."""\n    import pyarrow.parquet as pq\n    t = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles"]).to_pandas()\n    t = t.dropna().drop_duplicates("inchikey14")\n    km = keymap or {}\n    return {km.get(k, k): s for k, s in zip(t.inchikey14, t.normalized_smiles)}\n\n\ndef estimate_q(pool, feat, n=50_000, seed=0, mask=None, chunk=2_000):\n    """Background bit frequency q (float32 raw bits) over a seeded sample of the pool rows in mask (default: all)."""\n    idx = np.flatnonzero(np.ones(len(pool), bool) if mask is None else mask)\n    if len(idx) > n:\n        idx = np.sort(np.random.default_rng(seed).choice(idx, n, replace=False))\n    tot, cnt = np.zeros(feat.raw_bits, np.float64), 0\n    for a in range(0, len(idx), chunk):\n        for c in idx[a:a + chunk]:\n            r = feat.raw(pool.smiles[c])\n            if r is not None:\n                tot += r\n                cnt += 1\n    return (tot / max(cnt, 1)).astype(np.float32)\n\n\nclass AnalogBank:\n    """R5: analogue / candidate structures -> raw Family-A bits (packed, cached per key), background q, and the\n    per-molecule profile ("pssm": log-odds weights W; "tani": the top-k analogues\' Morgan r2 bits)."""\n\n    def __init__(self, smiles_of, q, cfg, feat=None):\n        if feat is None:\n            from .fpnet import A_RAW_BITS, Featuriser\n            feat = Featuriser("ahmed", np.arange(A_RAW_BITS))\n        self.feat, self.smiles_of, self.cfg, self._cache = feat, smiles_of, cfg, {}\n        self.q = None if q is None else np.asarray(q, np.float32)\n\n    def bits(self, keys, smiles):\n        """-> (raw bits (n, raw_bits) uint8, ok (n,) bool)."""\n        nb = self.feat.raw_bits\n        out, ok = np.zeros((len(keys), nb), np.uint8), np.zeros(len(keys), bool)\n        for i, (k, s) in enumerate(zip(keys, smiles)):\n            if k not in self._cache:\n                if len(self._cache) > 300_000:\n                    self._cache.clear()\n                r = self.feat.raw(s)\n                self._cache[k] = None if r is None else np.packbits(r)\n            p = self._cache[k]\n            if p is not None:\n                out[i], ok[i] = np.unpackbits(p)[:nb], True\n        return out, ok\n\n    def profile(self, hits, mode=None):\n        """{key: sim} -> profile dict, or None with fewer than analog_min_n analogues that have a structure."""\n        c = self.cfg\n        mode = mode or c.analog_channel\n        m = c.analog_m if mode == "pssm" else c.analog_k\n        top = [(k, s) for k, s in analog_set(hits, c.analog_smin, len(hits)) if self.smiles_of.get(k)]\n        X, ok = self.bits([k for k, _ in top[:2 * m]], [self.smiles_of[k] for k, _ in top[:2 * m]])\n        sel = np.flatnonzero(ok)[:m]\n        if len(sel) < c.analog_min_n:\n            return None\n        S = np.array([top[i][1] for i in sel], np.float64)\n        if mode == "pssm":\n            if self.q is None:\n                raise ValueError("pssm needs the background q")\n            return dict(mode=mode, n=len(sel), W=pssm_weights(X[sel], S, self.q, c.analog_alpha, c.analog_clip))\n        assert mode == "tani", mode\n        return dict(mode=mode, n=len(sel), X=X[sel, :MORGAN_R2], S=S)\n\n    def score(self, prof, keys, smiles):\n        """Candidate scores under a profile (NaN = no fingerprint)."""\n        out = np.full(len(keys), np.nan, np.float32)\n        if prof is None or not len(keys):\n            return out\n        C, ok = self.bits(keys, smiles)\n        if ok.any():\n            if prof["mode"] == "pssm":\n                out[ok] = C[ok].astype(np.float32) @ prof["W"]\n            else:\n                out[ok] = tani_kernel(C[ok, :MORGAN_R2], prof["X"], prof["S"])\n        return out\n', 'casmi_scratch/audit.py': '"""R1 coverage audit (gates R1.2, R1.3), the class-vote diagnostic (gate CV) and the R2 discrimination gate (R2.3)\non slice I.\n\nSlice I = the enveda-np-examples molecules of train (timsTOF natural products), one molecule per inchikey14.\nEach pool is audited in its own key space: R0 keys are plain inchikey14, R1 keys are scorer keys (the truth is\nmapped through the train keymap). Every library ban drops the query\'s own structure from the library.\n"""\nimport time\n\nimport numpy as np\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\n\nfrom .analog import analog_hits, class_vote, vote_report\nfrom .metfrag import frag_scores\nfrom .fpnet import molecule_spectra\nfrom .pipeline import analog_scores, class_bonus, features, molecule_target, plan_molecule, scored_range\nfrom .rank import rank_two_tier\nfrom .sim import rows_that_fit\n\nSLICE_I_LIB = "enveda-np-examples"\nSLICE_COLS = ["inchikey14", "normalized_smiles", "adduct", "precursor_mz", "ionization_mode", "collision_energy_ev",\n              "ms2_mzs", "ms2_normalized_intensities", "ingest_lib", "instrument_type"]\n\n\ndef load_slice(train_path, lib_name=SLICE_I_LIB):\n    """-> [(inchikey14, spectra frame)] for one ingest library."""\n    t = pq.read_table(train_path, columns=SLICE_COLS)\n    t = t.filter(pc.equal(t.column("ingest_lib").cast("string"), lib_name)).to_pandas()\n    return [(k, g.reset_index(drop=True)) for k, g in t.groupby("inchikey14")]\n\n\ndef window_keys(pool, target, cfg, scored=False):\n    lo, hi = scored_range(pool, target, cfg) if scored else pool.bounds(target, cfg.ppm_win)\n    return pool.keys[lo:hi]\n\n\ndef coverage(mols, pool, truth_of, cfg):\n    """Truth in the +-ppm_win window / in the scored range / in the pool at all, and window sizes."""\n    allkeys = set(pool.keys)\n    rows = []\n    for ik, sub in mols:\n        t = molecule_target(sub, cfg)\n        truth = truth_of(ik)\n        w = window_keys(pool, t, cfg) if np.isfinite(t) else np.zeros(0, object)\n        s = window_keys(pool, t, cfg, scored=True) if np.isfinite(t) else np.zeros(0, object)\n        rows.append(dict(ik=ik, in_pool=truth in allkeys, in_window=bool(np.isin(truth, w)),\n                         in_scored=bool(np.isin(truth, s)), n_window=len(w), n_scored=len(s)))\n    return rows\n\n\ndef summarize_coverage(rows):\n    a = lambda k: np.array([r[k] for r in rows])  # noqa: E731\n    return dict(n=len(rows), recall_window=float(a("in_window").mean()), recall_scored=float(a("in_scored").mean()),\n                in_pool=float(a("in_pool").mean()), window_median=float(np.median(a("n_window"))),\n                window_p95=float(np.percentile(a("n_window"), 95)), scored_median=float(np.median(a("n_scored"))))\n\n\ndef growth(rows0, rows1):\n    """Per-molecule window growth R1 / R0 (+-ppm_win window)."""\n    n0 = np.array([r["n_window"] for r in rows0], float)\n    n1 = np.array([r["n_window"] for r in rows1], float)\n    ratio = n1 / np.maximum(n0, 1.0)\n    return dict(ratio_median=float(np.median(ratio)), ratio_p95=float(np.percentile(ratio, 95)),\n                added_median=float(np.median(n1 - n0)), added_p95=float(np.percentile(n1 - n0, 95)),\n                recovered=int(sum(b["in_window"] and not a["in_window"] for a, b in zip(rows0, rows1))),\n                lost=int(sum(a["in_window"] and not b["in_window"] for a, b in zip(rows0, rows1))))\n\n\ndef own_ban(lib, key):\n    return lib.ik == key\n\n\ndef sim_budget_per_mol(cfg, n_test_mol=400):\n    """Simulation requests per test molecule that fit the time budget (R0\'s rows_that_fit, rate from Config)."""\n    rows = rows_that_fit(cfg.hard_budget_h * 3600.0 - cfg.pass2_s_per_mol * n_test_mol - 300.0, cfg.rate,\n                         cfg.startup_s)\n    return rows / n_test_mol\n\n\ndef truth_in_sim(mols, lib, pool, truth_of, cfg, req_per_mol, frag_fn=frag_scores, log=print):\n    """R1.3: share of molecules whose truth would be simulated, own spectra banned.\n\n    A molecule\'s request budget is spread over its (adduct, CE) conditions, so the truth is simulated iff its\n    priority rank < req_per_mol / n_conditions (an even split of the global budget; the real run shares it).\n    """\n    hit, n, t0 = [], 0, time.time()\n    for i, (ik, sub) in enumerate(mols):\n        t = molecule_target(sub, cfg)\n        if not np.isfinite(t):\n            continue\n        truth = truth_of(ik)\n        p = plan_molecule(sub, t, lib, pool, cfg.with_(use_sim=True), frag_fn=frag_fn,\n                          lib_exclude=own_ban(lib, truth))\n        keys = pool.keys[p["sim_idx"]]\n        j = np.flatnonzero(keys == truth)\n        n += 1\n        hit.append(bool(len(j)) and int(p["prio"][j[0]]) < req_per_mol / max(1, len(p["conds"])))\n        if i % 50 == 0:\n            log(f"  truth-in-sim {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n    return dict(n=n, req_per_mol=float(req_per_mol), truth_in_sim=float(np.mean(hit)) if hit else float("nan"))\n\n\ndef class_vote_diag(mols, lib, truth_of, label_of, cfg, modes=("delta", "open"), k=10, min_sim=0.5, log=print):\n    """Gate CV: per mode, vote purity vs the majority baseline and coverage, at superclass and pathway level.\n\n    label_of(key, level) -> label or None. The truth\'s own structure is banned from the library.\n    """\n    out = {}\n    for mode in modes:\n        preds = {"np_superclass": [], "np_pathway": []}\n        truths = {"np_superclass": [], "np_pathway": []}\n        n_hits, t0 = [], time.time()\n        for i, (ik, sub) in enumerate(mols):\n            t = molecule_target(sub, cfg)\n            truth = truth_of(ik)\n            pol = 1 if (sub.ionization_mode == "positive").mean() >= 0.5 else -1\n            specs = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]\n            hits = analog_hits(lib, specs, t, cfg, mode, polarity=pol, exclude=own_ban(lib, truth)) \\\n                if np.isfinite(t) else {}\n            n_hits.append(sum(1 for s in hits.values() if s >= min_sim))\n            for lvl in preds:\n                preds[lvl].append(class_vote(hits, lambda kk: label_of(kk, lvl), k, min_sim)[0])\n                truths[lvl].append(label_of(truth, lvl))\n            if i % 25 == 0:\n                log(f"  CV {mode} {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n        rep = {lvl: vote_report(preds[lvl], truths[lvl]) for lvl in preds}\n        sc = rep["np_superclass"]\n        rep["hits_ge_min_median"] = float(np.median(n_hits)) if n_hits else float("nan")\n        rep["pass"] = bool(sc["n_voted"] and sc["purity"] >= sc["baseline"] + 0.10 and sc["coverage"] >= 0.50)\n        out[mode] = rep\n    return out\n\n\ndef discrimination(mols, lib, pool, truth_of, vote_label, cfg, frag_fn=frag_scores, log=print):\n    """R2.3 per molecule, own structure banned from the library, no simulator (sv = 0), scored range.\n\n    -> rows: voted label, conf, the truth\'s P(voted) vs the window median, the truth\'s rank without / with the bonus,\n    whether the top 25 changed, and the tier-2 top-1 / top-2 fr gap.\n    """\n    c = cfg.with_(class_vote=True, use_sim=False)\n    rows, t0 = [], time.time()\n    for i, (ik, sub) in enumerate(mols):\n        t = molecule_target(sub, c)\n        if not np.isfinite(t):\n            continue\n        truth = truth_of(ik)\n        plan = plan_molecule(sub, t, lib, pool, c, frag_fn=frag_fn, lib_exclude=own_ban(lib, truth),\n                             vote_label=vote_label)\n        lo, hi = scored_range(pool, t, c)\n        cand = np.arange(lo, hi)\n        lv, sv, fr, ppm = features(plan, cand, pool, {}, c, frag_fn)\n        bonus = class_bonus(plan, cand, pool, c)\n        o0, o1 = rank_two_tier(lv, sv, fr, ppm, c.tau), rank_two_tier(lv, sv, fr, ppm, c.tau, bonus)\n        voted, conf = plan["vote"]\n        pv = bonus / (c.vote_beta * conf) if voted is not None and conf > 0 else np.zeros(len(cand))\n        j = np.flatnonzero(pool.keys[cand] == truth)\n        t2 = np.sort(fr[lv < c.tau])[::-1]\n        rows.append(dict(ik=ik, voted=voted, conf=conf, n_scored=len(cand), truth_in_scored=bool(len(j)),\n                         truth_p=float(pv[j[0]]) if len(j) else None,\n                         window_median_p=float(np.median(pv)) if len(pv) else None,\n                         rank0=int(np.flatnonzero(o0 == j[0])[0]) + 1 if len(j) else None,\n                         rank1=int(np.flatnonzero(o1 == j[0])[0]) + 1 if len(j) else None,\n                         changed=not np.array_equal(o0[:c.topn], o1[:c.topn]),\n                         fr_gap=float(t2[0] - t2[1]) if len(t2) > 1 else None))\n        if i % 25 == 0:\n            log(f"  R2.3 {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n    return rows\n\n\ndef summarize_discrimination(rows, topn=25):\n    """R2.3 verdict: (1) truth P(voted) > window median in >= 60% of voted molecules with the truth in range, and\n    (2) the bonus moves the truth up in more molecules than down."""\n    v = [r for r in rows if r["voted"] is not None and r["truth_in_scored"]]\n    above = [r["truth_p"] > r["window_median_p"] for r in v]\n    up = sum(r["rank1"] < r["rank0"] for r in v)\n    down = sum(r["rank1"] > r["rank0"] for r in v)\n    rr = lambda k: float(np.mean([1.0 / r[k] if r[k] <= topn else 0.0 for r in rows if r["truth_in_scored"]]))  # noqa\n    gaps = [r["fr_gap"] for r in rows if r["fr_gap"] is not None]\n    out = dict(n=len(rows), n_voted=sum(r["voted"] is not None for r in rows), n_voted_in_range=len(v),\n               truth_above_median=float(np.mean(above)) if above else float("nan"), up=up, down=down,\n               reach=float(np.mean([r["changed"] for r in rows])) if rows else float("nan"),\n               mrr_before=rr("rank0"), mrr_after=rr("rank1"),\n               fr_gap_median=float(np.median(gaps)) if gaps else float("nan"))\n    out["pass"] = bool(above and out["truth_above_median"] >= 0.60 and up > down)\n    return out\n\n\n# ---------------------------------------------------------------------------------------------- R3 FPNet channel\ndef load_cohorts(val_dir, cohorts=("C2", "C2_hard")):\n    """val_scripts/{val.parquet, val_key.pkl} -> [(molecule_id, inchikey14, query frame, ban_rows, ban_keys)].\n\n    C2 / C2_hard structures (C2_hard: their whole scaffold family) were banned from own-FPNet training; ban_rows\n    are train.parquet row indices (= library spectrum indices) to drop from the library for that query."""\n    import os\n    import pickle\n    key = [r for r in pickle.load(open(os.path.join(val_dir, "val_key.pkl"), "rb")) if r["cohort"] in cohorts]\n    q = pq.read_table(os.path.join(val_dir, "val.parquet")).to_pandas()\n    g = dict(list(q.groupby("molecule_id")))\n    return [(r["molecule_id"], r["inchikey14"], g[r["molecule_id"]].reset_index(drop=True),\n             list(r.get("ban_rows") or []), list(r.get("ban_keys") or [])) for r in key if r["molecule_id"] in g]\n\n\ndef ban_mask(lib, ban_rows, ban_keys, keymap=None):\n    m = np.zeros(len(lib), bool)\n    rows = np.asarray(ban_rows, np.int64)\n    m[rows[(rows >= 0) & (rows < len(lib))]] = True\n    keys = {keymap.get(k, k) if keymap else k for k in ban_keys}\n    if keys:\n        m |= np.isin(lib.ik, list(keys))\n    return m\n\n\ndef fp_discrimination(mols, lib, pool, truth_of, bank, cfg, bans=None, frag_fn=frag_scores, log=print):\n    """R3.audit per molecule: scored range, no simulator (sv = 0), class vote off; the truth\'s rank without the FP\n    channel (rank0) and with the RRF fusion (rank1). mols: [(mid, truth ik14, frame)]; bans: {mid: library mask}\n    (default: the truth\'s own structure)."""\n    c = cfg.with_(class_vote=False, use_sim=False, fp_channel=bank.kind)\n    rows, t0 = [], time.time()\n    for i, (mid, ik, sub) in enumerate(mols):\n        t = molecule_target(sub, c)\n        if not np.isfinite(t):\n            continue\n        truth = truth_of(ik)\n        excl = bans.get(mid) if bans is not None else own_ban(lib, truth)\n        plan = plan_molecule(sub, t, lib, pool, c, frag_fn=frag_fn, lib_exclude=excl)\n        z = bank.molecule_z(molecule_spectra(sub))\n        lo, hi = scored_range(pool, t, c)\n        cand = np.arange(lo, hi)\n        lv, sv, fr, ppm = features(plan, cand, pool, {}, c, frag_fn)\n        fz = bank.fz(z, [pool.keys[k] for k in cand], [pool.smiles[k] for k in cand]) if z is not None else None\n        o0 = rank_two_tier(lv, sv, fr, ppm, c.tau)\n        o1 = rank_two_tier(lv, sv, fr, ppm, c.tau, None, fz, c.fp_rrf_k) if fz is not None else o0\n        j = np.flatnonzero(pool.keys[cand] == truth)\n        fz_rank = None\n        if len(j) and fz is not None and np.isfinite(fz[j[0]]):\n            fz_rank = int(np.sum(np.nan_to_num(fz, nan=-np.inf) > fz[j[0]])) + 1\n        rows.append(dict(mid=mid, ik=ik, n_scored=len(cand), truth_in_scored=bool(len(j)), fp_ok=z is not None,\n                         tier1_top=bool(lv[o0[0]] >= c.tau) if len(o0) else False, fz_rank=fz_rank,\n                         rank0=int(np.flatnonzero(o0 == j[0])[0]) + 1 if len(j) else None,\n                         rank1=int(np.flatnonzero(o1 == j[0])[0]) + 1 if len(j) else None,\n                         changed=not np.array_equal(o0[:c.topn], o1[:c.topn])))\n        if i % 25 == 0:\n            log(f"  R3 {bank.kind} {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n    return rows\n\n\ndef summarize_fp(rows, topn=25, bar=0.02):\n    """R3.audit verdict: MRR@25 with FP - without >= bar and the truth moves up in more molecules than down."""\n    v = [r for r in rows if r["truth_in_scored"]]\n    rr = lambda k: float(np.mean([1.0 / r[k] if r[k] <= topn else 0.0 for r in v])) if v else float("nan")  # noqa\n    up = sum(r["rank1"] < r["rank0"] for r in v)\n    down = sum(r["rank1"] > r["rank0"] for r in v)\n    fr = [r["fz_rank"] for r in v if r["fz_rank"] is not None]\n    out = dict(n=len(rows), n_in_range=len(v), n_fp_ok=sum(r["fp_ok"] for r in rows), up=up, down=down,\n               reach=float(np.mean([r["changed"] for r in rows])) if rows else float("nan"),\n               mrr_before=rr("rank0"), mrr_after=rr("rank1"),\n               fz_only_mrr=float(np.mean([1.0 / x if x <= topn else 0.0 for x in fr])) if fr else float("nan"),\n               fz_top1=float(np.mean([x == 1 for x in fr])) if fr else float("nan"))\n    out["delta"] = out["mrr_after"] - out["mrr_before"]\n    out["pass"] = bool(v and out["delta"] >= bar and up > down)\n    return out\n\n\ndef fp_self_rank(bank, mols, pool, truth_of, cfg, n=5, k=5):\n    """R3.run smoke (b): for n molecules the bank trained on, is the truth\'s f.z in the top k of its +-ppm_win\n    window? -> (n_hit, rows). Catches a wrong bit order, adduct vocabulary or peak prep."""\n    rows = []\n    for ik, sub in mols:\n        if len(rows) >= n:\n            break\n        t = molecule_target(sub, cfg)\n        if not np.isfinite(t):\n            continue\n        truth = truth_of(ik)\n        cand = pool.window(t, cfg.ppm_win)\n        j = np.flatnonzero(pool.keys[cand] == truth)\n        if len(cand) < 10 or not len(j):\n            continue\n        z = bank.molecule_z(molecule_spectra(sub))\n        fz = bank.fz(z, [pool.keys[c] for c in cand], [pool.smiles[c] for c in cand])\n        if not np.isfinite(fz[j[0]]):\n            continue\n        rank = int(np.sum(np.nan_to_num(fz, nan=-np.inf) > fz[j[0]])) + 1\n        rows.append(dict(ik=ik, n_window=len(cand), rank=rank))\n    return sum(r["rank"] <= k for r in rows), rows\n\n\n# ---------------------------------------------------------------------------------------------- R5 analogue channel\nAN_MODES = ("none", "pssm", "tani")\n\n\ndef analog_discrimination(mols, lib, pool, truth_of, fp_bank, an_bank, cfg, vote_label=None, frag_fn=frag_scores,\n                          log=print):\n    """R5.audit per molecule: own structure banned from the library, no simulator (sv = 0), and the truth counted\n    as non-train (the analogue list scores it; every other train structure gets the neutral term), as for an NP\n    truth in the hidden test. The truth\'s rank under every arm: FP list off/on x analogue list none/pssm/tani x\n    class vote off/on (vote arms only with vote_label). mols: [(inchikey14, frame)]."""\n    c = cfg.with_(use_sim=False, class_vote=vote_label is not None, analog_channel="pssm")\n    rows, t0 = [], time.time()\n    for i, (ik, sub) in enumerate(mols):\n        t = molecule_target(sub, c)\n        if not np.isfinite(t):\n            continue\n        truth = truth_of(ik)\n        plan = plan_molecule(sub, t, lib, pool, c, frag_fn=frag_fn, lib_exclude=own_ban(lib, truth),\n                             vote_label=vote_label)\n        hits = plan.pop("an_hits") or {}\n        profs = {m: an_bank.profile(hits, m) for m in AN_MODES[1:]}\n        lo, hi = scored_range(pool, t, c)\n        cand = np.arange(lo, hi)\n        lv, sv, fr, ppm = features(plan, cand, pool, {}, c, frag_fn)\n        bonus = class_bonus(plan, cand, pool, c) if vote_label is not None else None\n        z = fp_bank.molecule_z(molecule_spectra(sub)) if fp_bank is not None else None\n        fz = fp_bank.fz(z, [pool.keys[k] for k in cand], [pool.smiles[k] for k in cand]) if z is not None else None\n        ans = {m: analog_scores(dict(an_prof=profs[m], an_free={truth}), cand, pool, c.with_(analog_channel=m),\n                                an_bank) for m in AN_MODES[1:]}\n        j = np.flatnonzero(pool.keys[cand] == truth)\n        row = dict(ik=ik, n_scored=len(cand), truth_in_scored=bool(len(j)), fp_ok=fz is not None,\n                   n_hits=len(hits), an_s=round(plan["an_s"], 2),\n                   tier1_top=bool(len(cand)) and bool(lv[rank_two_tier(lv, sv, fr, ppm, c.tau)[0]] >= c.tau))\n        for m in AN_MODES[1:]:\n            row[f"n_{m}"] = profs[m]["n"] if profs[m] is not None else 0\n            a = ans[m]\n            row[f"{m}_rank"] = (int(np.sum(np.nan_to_num(a, nan=-np.inf) > a[j[0]])) + 1\n                                if len(j) and a is not None and np.isfinite(a[j[0]]) else None)\n            row[f"{m}_scored"] = int(np.isfinite(a).sum()) if a is not None else 0\n        for fp_on in (False, True):\n            for m in AN_MODES:\n                for vote in (False, True):\n                    if vote and bonus is None:\n                        continue\n                    o = rank_two_tier(lv, sv, fr, ppm, c.tau, bonus if vote else None, fz if fp_on else None,\n                                      c.fp_rrf_k, None, ans.get(m), c.analog_w)\n                    row[f"fp{int(fp_on)}_{m}_v{int(vote)}"] = int(np.flatnonzero(o == j[0])[0]) + 1 if len(j) \\\n                        else None\n        rows.append(row)\n        if i % 25 == 0:\n            log(f"  R5 {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n    return rows\n\n\ndef summarize_analog(rows, topn=25, base="fp1_none_v1", bar=0.010):\n    """R5.audit: MRR@25 per arm (molecules with the truth in range), up / down vs base for each arm, coverage, the\n    analogue-only MRR, and the verdict per scorer: arm fp1_<mode>_v1 - base >= bar and up > down."""\n    v = [r for r in rows if r["truth_in_scored"]]\n    if not v:\n        return dict(n=len(rows), n_in_range=0)\n    if base not in v[0]:\n        base = base.replace("_v1", "_v0")\n    rr = lambda k: float(np.mean([1.0 / r[k] if r[k] <= topn else 0.0 for r in v]))  # noqa: E731\n    arms = sorted(k for k in v[0] if k.startswith("fp") and k[2] in "01")\n    out = dict(n=len(rows), n_in_range=len(v), base=base, mrr={k: rr(k) for k in arms}, up={}, down={},\n               an_s_median=float(np.median([r["an_s"] for r in rows])),\n               an_s_total=float(np.sum([r["an_s"] for r in rows])))\n    for k in arms:\n        out["up"][k] = sum(r[k] < r[base] for r in v)\n        out["down"][k] = sum(r[k] > r[base] for r in v)\n    for m in AN_MODES[1:]:\n        ranks = [r[f"{m}_rank"] for r in v if r[f"{m}_rank"] is not None]\n        out[f"{m}_coverage"] = float(np.mean([r[f"n_{m}"] > 0 for r in rows]))\n        out[f"{m}_n_median"] = float(np.median([r[f"n_{m}"] for r in rows]))\n        out[f"{m}_only_mrr"] = float(np.mean([1.0 / x if x <= topn else 0.0 for x in ranks])) if ranks else None\n        arm = base.replace("none", m)\n        out[f"{m}_delta"] = out["mrr"][arm] - out["mrr"][base]\n        out[f"{m}_pass"] = bool(out[f"{m}_delta"] >= bar and out["up"][arm] > out["down"][arm])\n    return out\n', 'casmi_scratch/chem.py': '"""Adduct physics and formula masses (no RDKit)."""\nimport re\n\nimport numpy as np\n\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nELECTRON = 0.00054857990\nPROTON = MASS["H"] - ELECTRON\nH2O = 2 * MASS["H"] + MASS["O"]\nNH4 = MASS["N"] + 4 * MASS["H"]\nFORMATE = MASS["C"] + 2 * MASS["H"] + 2 * MASS["O"]\nACETATE = 2 * MASS["C"] + 4 * MASS["H"] + 2 * MASS["O"]\n\n# adduct -> (n molecules, charge, mass delta): precursor m/z = (n * M + delta) / z\nADDUCTS = {\n    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - ELECTRON), "[M+Na]+": (1, 1, MASS["Na"] - ELECTRON),\n    "[M+K]+": (1, 1, MASS["K"] - ELECTRON), "[M-H2O+H]+": (1, 1, PROTON - H2O),\n    "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O), "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -ELECTRON),\n    "[M-H2O]+": (1, 1, -ELECTRON - H2O),\n    "[M+CH3OH+H]+": (1, 1, PROTON + MASS["C"] + 4 * MASS["H"] + MASS["O"]),\n    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS["C"] + 3 * MASS["H"] + MASS["N"]),\n    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),\n    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS["Cl"] + ELECTRON), "[M]-": (1, 1, ELECTRON),\n    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS["Na"] - 2 * PROTON),\n    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS["Na"] - ELECTRON), "[2M+NH4]+": (2, 1, NH4 - ELECTRON),\n    "[2M+K]+": (2, 1, MASS["K"] - ELECTRON), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),\n    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS["Na"] - 2 * PROTON),\n    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    """Vectorised precursor m/z -> neutral mass; NaN for an unknown adduct."""\n    mz = np.asarray(mz, np.float64)\n    out = np.full(len(mz), np.nan)\n    ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = ad == a\n        if m.any():\n            out[m] = (mz[m] * z - d) / n\n    return out\n\n\ndef precursor_of(mass, adduct):\n    """Neutral exact mass -> theoretical precursor m/z (singly charged, one molecule adducts)."""\n    n, z, d = ADDUCTS[adduct]\n    return (np.asarray(mass, np.float64) * n + d) / z\n\n\n_MONO = {"C": 12.0, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221, "S": 31.97207069,\n         "P": 30.97376151, "F": 18.99840320, "Cl": 34.96885271, "Br": 78.9183376, "I": 126.904468,\n         "Si": 27.9769265327, "Se": 79.9165196, "B": 11.0093055, "Na": 22.98976928, "K": 38.96370649,\n         "Fe": 55.9349375, "D": 2.0141017778, "As": 74.9215965, "Li": 7.0160034, "Sn": 119.9021947}\n_TOK = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    """Monoisotopic mass of a molecular formula string; NaN if unparseable or an exotic element."""\n    if not isinstance(f, str):\n        return np.nan\n    s, pos = 0.0, 0\n    for m in _TOK.finditer(f):\n        if m.start() != pos or m.group(1) not in _MONO:\n            return np.nan\n        pos = m.end()\n        s += _MONO[m.group(1)] * (int(m.group(2)) if m.group(2) else 1)\n    return s if pos == len(f) else np.nan\n\n\ndef query_target(nm, ppm_center):\n    """A molecule\'s neutral masses -> centred target mass (median, corrected for the timsTOF offset)."""\n    nm = np.asarray(nm, np.float64)\n    nm = nm[np.isfinite(nm)]\n    return float(np.median(nm)) / (1.0 + ppm_center / 1e6) if len(nm) else np.nan\n', 'casmi_scratch/config.py': '"""Every setting of the pipeline in one dataclass (replaces the notebooks\' global CFG)."""\nfrom dataclasses import asdict, dataclass, replace\n\nCOVERED_ADDUCTS = ("[M+H]+", "[M+NH4]+", "[M+Na]+", "[M-H]-", "[M+CH2O2-H]-", "[M+Cl]-")   # Track 3 G2b\n\n\n@dataclass(frozen=True)\nclass Config:\n    # --- candidate window (phase1 values) ---------------------------------------------------------------\n    ppm_win: float = 8.5\n    ppm_center: float = 1.45            # timsTOF precursor reads +1.45 ppm high\n    ppm_expand: tuple = (8.5, 20.0, 45.0, 80.0)\n    ppm_fallback: float = 30.0\n    backfill_ppm: float = 120.0\n    expand_backfill: bool = True\n    topn: int = 25\n\n    # --- candidate pool (R1) ------------------------------------------------------------------------------\n    pool_source: str = "r0"             # "r0": COCONUT + train built at runtime; "r1": casmi26-scratch-pool dataset\n    pool_drop_e180: bool = False        # R1b: drop structures whose only provenance is enveda-180\n\n    # --- spectrum cleaning: ONE max_peaks for library sim, MetFrag-lite and ICEBERG sim -------------------\n    int_floor: float = 0.002\n    max_peaks: int = 100\n    mz_tol: float = 0.01\n    int_power: float = 1.0\n    ent_weight: bool = True\n    prec_margin: float = 2.0            # predicted peaks above precursor + 2 Da are dropped\n\n    # --- library / MetFrag-lite -------------------------------------------------------------------------\n    struct_mass: bool = True            # library neutral mass from the label formula\n    max_bonds: int = 60\n    frag_workers: int = 4\n\n    # --- two-tier ranking ---------------------------------------------------------------------------------\n    tau: float = 0.75\n\n    # --- R2 class vote (NPClassifier superclass of analog library hits -> tier-2 bonus) -----------------------\n    class_vote: bool = False\n    vote_beta: float = 0.10             # pre-registered, never tuned\n    vote_k: int = 10\n    vote_min_sim: float = 0.5\n    vote_ppm: float = 10.0              # direct + NP delta-list windows\n\n    # --- R3 FPNet channel (f.z fused with sv by RRF inside tier 2, stages A + B) -------------------------------\n    fp_channel: str = ""                # "" (off) | "own" (our FPNet export, family P) | "ahmed" (fpnet_full1, family A)\n    fp_rrf_k: float = 3.0               # pre-registered, never tuned\n    # --- R4 FPNet family fusion: one f.z channel per bank, each its own RRF term (overrides fp_channel when set) ---\n    fp_banks: tuple = ()                # e.g. ("ahmed", "prv") = full1 (family A) + prvsiyan casmi26-fp-models-v2 (P)\n    fp_weights: tuple = ()              # RRF weight per bank (default 1.0 each); R4: (1.0, 0.6), never tuned\n    fp_stagec: bool = False             # R4: order the stage-C mass-nearest top-up by the fused FP key\n\n    def fp_kinds(self):\n        """The FPNet banks in use: fp_banks, else (fp_channel,) when set, else ()."""\n        return tuple(self.fp_banks) or ((self.fp_channel,) if self.fp_channel else ())\n\n    def fp_w(self):\n        k = self.fp_kinds()\n        w = tuple(float(x) for x in self.fp_weights) or (1.0,) * len(k)\n        if len(w) != len(k):\n            raise ValueError(f"fp_weights {self.fp_weights} do not match fp banks {k}")\n        return w\n\n    # --- R5 analogue channel (open +-200 Da analogues -> PSSM or Tanimoto kernel; a tier-2 RRF list) ----------\n    analog_channel: str = ""            # "" (off) | "pssm" | "tani"; specs/specs_scratch/pssm.md\n    analog_open_da: float = 200.0\n    analog_smin: float = 0.5\n    analog_m: int = 100                 # PSSM: top analogues (deduplicated by structure)\n    analog_k: int = 20                  # Tanimoto kernel: top analogues\n    analog_alpha: float = 1.0           # pseudocount: background analogues in the profile\n    analog_clip: float = 3.0            # |W_j| cap\n    analog_w: float = 1.0               # RRF weight, pre-registered, never tuned\n    analog_min_n: int = 3               # fewer analogues -> no list for that molecule\n    analog_q_n: int = 50_000            # pool sample for the background q when no analog_q.npy is attached\n    analog_exclude_train: bool = True   # D4: F_TRAIN candidates get the neutral term in the list; False = score all\n\n    # --- simulator (ICEBERG, or GLACIER for the R1 GLACIER twins) -------------------------------------------\n    use_sim: bool = True\n    sim_model: str = "iceberg"          # "iceberg" | "glacier" (casmi26-glacier: [M+H]+ / [M+Na]+ only)\n    gl_batch: int = 32                  # GLACIER max batch (the runner also caps B * N_atoms^2 <= bs * 1600)\n    gl_batch_retry: int = 8\n    sim_tier1: bool = False\n    merge_rule: str = "max"\n    covered: tuple = COVERED_ADDUCTS\n    ce_mode: str = "ev"\n    instrument: str = "QTOF"\n    ce_default: float = 40.0\n    batch: int = 128\n    batch_retry: int = 32\n    chunk: int = 20_000\n    sparse_k: int = 100\n    rate: float = 36.6                  # requests/s on one T4 (replaced by the measured rate)\n    startup_s: float = 10.4\n    hard_budget_h: float = 7.5\n    pass2_s_per_mol: float = 1.0\n    min_ok_fast: float = 0.99\n\n    # --- run mode -----------------------------------------------------------------------------------------\n    allow_fast: bool = True\n    fast_dev_molecules: int = 20\n    public_n_mol: int = 400\n    public_n_spec: int = 1213\n    force_full: bool = False\n\n    def with_(self, **kw):\n        return replace(self, **kw)\n\n    def as_dict(self):\n        return asdict(self)\n', 'casmi_scratch/fpnet.py': '"""R3: spectrum -> fingerprint channel (FPNet). Candidate score f.z = candidate bits . predicted logits.\n\nTwo checkpoint families with different architectures and bit sets; never mix them:\n- "own" (family P): build/fpnet_train.py / prvsiyan\'s phase1 FPNet (6 layers, adduct + instrument embeddings,\n  128 peaks). Raw bits Morgan r2 4096 + r3 4096 + RDKitFP(2048, maxPath 6) + MACCS 167 = 10,407; fp_bits.npy\n  (next to coco_fp.npy) selects 6,930. Models whose basename contains "merged" score the merged pseudo-spectrum,\n  the rest score each spectrum; z = mean(single-model per-spectrum mean, merged-model merged logits).\n  Copied from the zenith Engine 2 `pv_fp.molecule_logits`.\n- "prv" (family P, R4): prvsiyan\'s casmi26-fp-models-v2 (fp_single_s2, fp_merged_m1), the zenith Engine 2 models\n  themselves; same model, featuriser and fp_bits.npy as "own", found by dataset path (no fp_bits_sha1.txt).\n- "ahmed" (family A): Ahmed Berat Ozer\'s casmi26-v4b engine FPNet (casmi/fpnet.py + casmi/chem.py, vendored here\n  with attribution; 8 layers, ADDUCT_LIST embedding, ce_known / merged / mode inputs, 160 peaks). Raw bits\n  ecfp4 4096, ecfp6 4096, fcfp4 2048, rdk6 2048, ap 2048, tt 2048, maccs 167 = 16,507; casmi26-v2-pool\'s\n  fp_bits.npy selects 10,226. z = 0.5 (mean per-spectrum logits + mean per-polarity merged logits), at most 16\n  spectra, fp16 autocast on T4. Copied from the zenith PubChem runner `molecule_logits`.\n\ntorch / RDKit are imported lazily so the rest of the package (and the tests) never need them.\n"""\nimport hashlib\nimport math\nimport os\n\nimport numpy as np\n\n# ---------------------------------------------------------------------------------------------- shared helpers\n_TORCH = {}\n\n\ndef _torch():\n    if not _TORCH:\n        import torch\n        import torch.nn as nn\n        import torch.nn.functional as F\n        _TORCH.update(torch=torch, nn=nn, F=F)\n    return _TORCH["torch"], _TORCH["nn"], _TORCH["F"]\n\n\ndef bits_sha1(bits):\n    return hashlib.sha1(np.ascontiguousarray(bits).tobytes()).hexdigest()\n\n\ndef _ce_scalar(v):\n    """(mean |CE|, mean CE, n) of a list-like collision energy; NaN / 0 when absent. Family P uses |CE| (zenith\n    parse_ce), family A the signed mean (casmi/build.py ce_mean)."""\n    try:\n        a = np.atleast_1d(np.asarray(v, float)) if v is not None else np.zeros(0)\n        a = a[np.isfinite(a)]\n        return (float(np.abs(a).mean()), float(a.mean()), int(len(a))) if len(a) else (np.nan, np.nan, 0)\n    except Exception:\n        return np.nan, np.nan, 0\n\n\ndef molecule_spectra(sub):\n    """One molecule\'s rows -> spectrum dicts (raw peaks; both families clean them their own way)."""\n    out = []\n    has_instr = "instrument_type" in sub.columns\n    for r in sub.itertuples():\n        ce, ce_signed, ce_n = _ce_scalar(r.collision_energy_ev)\n        out.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                        prec=float(r.precursor_mz), adduct=str(r.adduct),\n                        mode=1 if r.ionization_mode == "positive" else -1,\n                        instr=str(r.instrument_type) if has_instr else None, ce=ce, ce_signed=ce_signed,\n                        ce_n=ce_n))\n    return out\n\n\ndef _merge_peaks(pairs, mz_tol=0.005, chain=False):\n    """Collapse several (mz, it) lists into one (each normalised to base peak 1; max wins on near-duplicates).\n    chain=False: pairwise neighbour rule (family P `merge_peaks`); chain=True: family A `merge_spectra`."""\n    pairs = [(np.asarray(a, float), np.asarray(b, float)) for a, b in pairs if len(a)]\n    pairs = [(a, b) for a, b in pairs if b.max() > 0] if chain else pairs\n    if not pairs:\n        return np.zeros(0), np.zeros(0)\n    mz = np.concatenate([a for a, _ in pairs])\n    it = np.concatenate([b / max(float(b.max()), 1e-9) for _, b in pairs])\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    keep = np.ones(len(mz), bool)\n    last = 0\n    for j in range(1, len(mz)):\n        ref = last if chain else j - 1\n        if mz[j] - mz[ref] < mz_tol:\n            if it[j] >= it[ref]:\n                keep[ref] = False\n                last = j\n            else:\n                keep[j] = False\n        else:\n            last = j\n    return mz[keep], it[keep]\n\n\ndef _sin_emb_cls(hi):\n    torch, nn, _ = _torch()\n\n    class SinEmb(nn.Module):\n        def __init__(self, dim, lo=-2.0, hi=hi, power=1.0):\n            super().__init__()\n            n = dim // 2\n            wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\n            self.register_buffer("inv", (2 * math.pi) / wav)\n\n        def forward(self, x):\n            a = x.unsqueeze(-1) * self.inv\n            return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\n    return SinEmb\n\n\ndef _block_cls():\n    torch, nn, F = _torch()\n\n    class Block(nn.Module):\n        def __init__(self, d, h, drop):\n            super().__init__()\n            self.h = h\n            self.n1 = nn.LayerNorm(d)\n            self.qkv = nn.Linear(d, 3 * d)\n            self.o = nn.Linear(d, d)\n            self.n2 = nn.LayerNorm(d)\n            self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\n            self.drop = nn.Dropout(drop)\n\n        def forward(self, x, pad):\n            B, N, D = x.shape\n            y = self.n1(x)\n            q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n            a = F.scaled_dot_product_attention(q, k, v, attn_mask=(~pad)[:, None, None, :])\n            x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\n            return x + self.drop(self.ff(self.n2(x)))\n\n    return Block\n\n\ndef _pool_head(x, pad):\n    cls = x[:, 0]\n    msk = (~pad[:, 1:]).float().unsqueeze(-1)\n    mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\n    return cls, mean\n\n\n# ---------------------------------------------------------------------------------------------- family P ("own")\nP_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",\n             "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",\n             "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",\n             "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]\nP_AD_IX = {a: i for i, a in enumerate(P_ADDUCTS)}\nP_INSTR = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]\nP_RAW_BITS = 4096 + 4096 + 2048 + 167\nP_KINDS = ("own", "prv")    # family P banks: our casmi-fpnet export, prvsiyan casmi26-fp-models-v2\n\n\ndef p_instr_family(s):\n    if s is None:\n        return 4\n    t = str(s).lower()\n    if "timstof" in t:\n        return 0\n    if "orbitrap" in t or "qft" in t or "ftms" in t or "hybrid ft" in t or "itft" in t or "exactive" in t:\n        return 1\n    if "tof" in t:\n        return 2\n    if "trap" in t or "qq" in t:\n        return 3\n    return 4\n\n\ndef p_prep_peaks(mz, inten, prec_mz, max_peaks=128, floor=1e-3, win=50.0, per_win=8):\n    """Verbatim family-P peak prep (<= prec + 1.5, floor, at most 8 per 50 Da, top 128, sqrt intensity)."""\n    mz = np.asarray(mz, np.float64)\n    it = np.asarray(inten, np.float64)\n    if len(mz) == 0:\n        return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = mz <= prec_mz + 1.5\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0:\n        return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    mx = it.max()\n    if mx <= 0:\n        return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = it >= floor * mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        order = np.argsort(-it)\n        bucket = (mz // win).astype(np.int64)\n        cnt, sel = {}, []\n        for i in order:\n            b = bucket[i]\n            c = cnt.get(b, 0)\n            if c < per_win:\n                cnt[b] = c + 1\n                sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            chosen = set(sel.tolist())\n            rest = np.array([i for i in order if i not in chosen])\n            if len(rest):\n                sel = np.concatenate([sel, rest[:max_peaks - len(sel)]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    return mz.astype(np.float32), np.sqrt(it / it.max()).astype(np.float32)\n\n\ndef p_model(nbits, d=512, layers=6, heads=8, drop=0.1):\n    torch, nn, _ = _torch()\n    SinEmb, Block = _sin_emb_cls(3.2), _block_cls()\n\n    class FPNetP(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.d = d\n            self.mz_emb, self.nl_emb = SinEmb(d), SinEmb(d)\n            self.pk = nn.Linear(2 * d + 1, d)\n            self.prec_emb = SinEmb(d)\n            self.ad = nn.Embedding(len(P_ADDUCTS), d)\n            self.ins = nn.Embedding(len(P_INSTR), d)\n            self.gl = nn.Linear(d + 3, d)\n            self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n            self.norm = nn.LayerNorm(d)\n            self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n\n        def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n            B = mz.shape[0]\n            nl = (prec[:, None] - mz).clamp(min=0)\n            p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n            g = self.gl(torch.cat([self.prec_emb(prec), (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                                   torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\n            x = torch.cat([g.unsqueeze(1), p], 1)\n            pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n            for b in self.blocks:\n                x = b(x, pad)\n            x = self.norm(x)\n            return self.head(torch.cat(_pool_head(x, pad), -1))\n\n    return FPNetP()\n\n\n# ---------------------------------------------------------------------------------------------- family A ("ahmed")\nA_TEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",\n                  "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]\n# key order of casmi/chem.py ADDUCTS (insertion order defines the embedding rows)\nA_ADDUCT_KEYS = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M+Li]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M-H2O]+",\n                 "[M]+", "[M+2H]2+", "[M+3H]3+", "[M+H+Na]2+", "[M+HNa]2+", "[M+2Na]2+", "[M+2Na-H]+", "[M+Ca]2+",\n                 "[M+CH3OH+H]+", "[M+CH3CN+H]+", "[M+CH3CN+Na]+", "[M+C2H4N]+", "[M+CH5O]+", "[M-H2N]+",\n                 "[M-NH3+H]+", "[M-H5O3]+", "[M-CH3]+", "[M+H-CH3]+", "[M-H]-", "[M-2H]-", "[M-2H]2-",\n                 "[M-H2]2-", "[M]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+HCOO]-", "[M+C2H4O2-H]-", "[M+CH3COO]-",\n                 "[M+Cl]-", "[M+Br]-", "[M+Na-2H]-", "[M+K-2H]-", "[M+Na-H2]-", "[M+K-H2]-", "[M-CH3]-",\n                 "[M-C3H6NO2]-", "[M+C2F3O2]-", "[M+CF3COO]-", "[2M+H]+", "[2M+Na]+", "[2M+K]+", "[2M+NH4]+",\n                 "[2M+Ca]2+", "[2M-H]-", "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[2M+Cl]-", "[3M+H]+",\n                 "[3M+Na]+", "[3M-H]-"]\nA_ADDUCTS = A_TEST_ADDUCTS + [a for a in A_ADDUCT_KEYS if a not in A_TEST_ADDUCTS] + ["<unk>"]\nA_AD_IX = {a: i for i, a in enumerate(A_ADDUCTS)}\nA_RAW_BITS = 4096 + 4096 + 2048 + 2048 + 2048 + 2048 + 167\nA_MAX_SPECTRA = 16\n\n\ndef a_prep_peaks(mz, it, prec_mz, max_peaks=160, floor=1e-3):\n    """Verbatim family-A peak prep (<= prec + 2, floor, top 160 by intensity, sqrt intensity)."""\n    mz = np.asarray(mz, np.float64)\n    it = np.asarray(it, np.float64)\n    if len(mz) == 0:\n        return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = (mz <= prec_mz + 2.0) & (mz > 0)\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0 or it.max() <= 0:\n        return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    it = it / it.max()\n    keep = it >= floor\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        sel = np.argsort(-it)[:max_peaks]\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    return mz[o].astype(np.float32), np.sqrt(it[o]).astype(np.float32)\n\n\ndef a_model(nbits, d=512, layers=8, heads=8, drop=0.1, n_elem=10):\n    torch, nn, _ = _torch()\n    SinEmb, Block = _sin_emb_cls(3.3), _block_cls()\n\n    class FPNetA(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.mz_emb, self.nl_emb = SinEmb(d), SinEmb(d)\n            self.int_emb = nn.Linear(2, d)\n            self.pk = nn.Linear(3 * d, d)\n            self.prec_emb = SinEmb(d)\n            self.ad = nn.Embedding(len(A_ADDUCTS), d)\n            self.ce_emb = SinEmb(d // 4, lo=0.0, hi=2.2)\n            self.gl = nn.Linear(d + d // 4 + 4, d)\n            self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n            self.norm = nn.LayerNorm(d)\n            self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n            self.elem_head = nn.Sequential(nn.Linear(2 * d, 256), nn.GELU(), nn.Linear(256, n_elem))\n\n        def forward(self, mz, it, pad, prec, ad, ce, ce_known, merged, mode):\n            B = mz.shape[0]\n            nl = (prec[:, None] - mz).clamp(min=0)\n            p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl),\n                                   self.int_emb(torch.stack([it, it * it], -1))], -1))\n            g = self.gl(torch.cat([self.prec_emb(prec), self.ce_emb(ce / 10.0 + 1.0), ce_known.unsqueeze(-1),\n                                   merged.unsqueeze(-1), mode.unsqueeze(-1),\n                                   torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad)\n            x = torch.cat([g.unsqueeze(1), p], 1)\n            pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n            for b in self.blocks:\n                x = b(x, pad)\n            x = self.norm(x)\n            return self.head(torch.cat(_pool_head(x, pad), -1))\n\n    return FPNetA()\n\n\n# ---------------------------------------------------------------------------------------------- featurisers\nclass Featuriser:\n    """SMILES -> selected binary bits (uint8, n x nbits) for one family; RDKit generators built once."""\n\n    def __init__(self, kind, bits):\n        self.kind, self.bits = kind, np.asarray(bits, np.int64)\n        self.nbits = len(self.bits)\n        self.raw_bits = P_RAW_BITS if kind in P_KINDS else A_RAW_BITS\n        if self.bits.max() >= self.raw_bits:\n            raise ValueError(f"fp_bits indexes past the {kind} raw layout ({self.bits.max()} >= {self.raw_bits})")\n        from rdkit import Chem, RDLogger\n        from rdkit.Chem import MACCSkeys, rdFingerprintGenerator as g\n        RDLogger.DisableLog("rdApp.*")\n        self._chem, self._maccs = Chem, MACCSkeys\n        if kind in P_KINDS:\n            self._gens = [g.GetMorganGenerator(radius=2, fpSize=4096), g.GetMorganGenerator(radius=3, fpSize=4096),\n                          g.GetRDKitFPGenerator(fpSize=2048, maxPath=6)]\n        else:\n            self._gens = [g.GetMorganGenerator(radius=2, fpSize=4096), g.GetMorganGenerator(radius=3, fpSize=4096),\n                          g.GetMorganGenerator(radius=2, fpSize=2048,\n                                               atomInvariantsGenerator=g.GetMorganFeatureAtomInvGen()),\n                          g.GetRDKitFPGenerator(fpSize=2048, maxPath=6), g.GetAtomPairGenerator(fpSize=2048),\n                          g.GetTopologicalTorsionGenerator(fpSize=2048)]\n\n    def raw(self, smi):\n        try:\n            m = self._chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n            if m is None:\n                return None\n            parts = [gen.GetFingerprintAsNumPy(m).astype(np.uint8) for gen in self._gens]\n            parts.append(np.array(list(self._maccs.GenMACCSKeys(m)), dtype=np.uint8))\n            return np.concatenate(parts)\n        except Exception:\n            return None\n\n    def __call__(self, smiles):\n        """-> (bits (n, nbits) uint8, ok (n,) bool)."""\n        out = np.zeros((len(smiles), self.nbits), np.uint8)\n        ok = np.zeros(len(smiles), bool)\n        for i, s in enumerate(smiles):\n            r = self.raw(s)\n            if r is not None:\n                out[i], ok[i] = r[self.bits], True\n        return out, ok\n\n\ndef featuriser_parity(feat, smiles, packed_ref):\n    """Rows where our bits equal a precomputed packed reference (coco_fp.npy / pool_fp.npy rows)."""\n    bits, ok = feat(smiles)\n    ref = np.unpackbits(np.asarray(packed_ref, np.uint8), axis=1)[:, :feat.nbits]\n    return int(sum(bool(o) and np.array_equal(b, r) for b, r, o in zip(bits, ref, ok)))\n\n\n# ---------------------------------------------------------------------------------------------- banks\nclass FPBank:\n    """Loaded checkpoints of one family + its featuriser. `molecule_z(spectra)` -> logits (nbits,) or None;\n    `fz(z, keys, smiles)` -> f.z per candidate (NaN when the SMILES does not parse)."""\n\n    def __init__(self, kind, models, merged, feat, device, sha1=None, paths=()):\n        self.kind, self.models, self.merged, self.feat, self.device = kind, models, merged, feat, device\n        self.nbits, self.sha1, self.paths = feat.nbits, sha1, list(paths)\n        self._cache = {}\n\n    def describe(self):\n        return dict(kind=self.kind, nbits=self.nbits, fp_bits_sha1=self.sha1, device=self.device,\n                    models=[os.path.basename(p) for p in self.paths],\n                    merged=[os.path.basename(p) for p, m in zip(self.paths, self.merged) if m])\n\n    # -- logits\n    def _run(self, nets, items, args_fn, autocast=False):\n        torch, _, _ = _torch()\n        items = [x for x in items if len(x["mz"])]\n        if not items or not nets:\n            return None\n        B, N = len(items), max(len(x["mz"]) for x in items)\n        mz = np.zeros((B, N), np.float32)\n        it = np.zeros((B, N), np.float32)\n        pad = np.ones((B, N), bool)\n        for i, x in enumerate(items):\n            n = len(x["mz"])\n            mz[i, :n], it[i, :n], pad[i, :n] = x["mz"], x["it"], False\n        T = lambda a, dt=None: torch.as_tensor(np.asarray(a, dt), device=self.device)  # noqa: E731\n        args = (T(mz), T(it), T(pad)) + args_fn(items, T)\n        use_ac = autocast and self.device == "cuda"\n        dtype = torch.float16 if use_ac and torch.cuda.get_device_capability()[0] < 8 else torch.bfloat16\n        with torch.no_grad(), torch.autocast("cuda", dtype=dtype, enabled=use_ac):\n            return np.stack([n(*args).float().cpu().numpy() for n in nets]).mean(0)   # (B, nbits)\n\n    def molecule_z(self, spectra):\n        if not spectra:\n            return None\n        return self._z_own(spectra) if self.kind in P_KINDS else self._z_ahmed(spectra)\n\n    def _z_own(self, spectra):\n        single = [m for m, g in zip(self.models, self.merged) if not g]\n        merged = [m for m, g in zip(self.models, self.merged) if g]\n        f32 = np.float32\n\n        def args(items, T):\n            return (T([x["prec"] for x in items], f32), T([P_AD_IX.get(x["adduct"], P_AD_IX["<unk>"]) for x in items]),\n                    T([p_instr_family(x["instr"]) for x in items]), T([x["ce"] for x in items], f32),\n                    T([x["mode"] for x in items], f32))\n\n        out = []\n        items = []\n        for s in spectra:\n            mz, it = self._clip(s)\n            pm, pi = p_prep_peaks(mz, it, s["prec"])\n            items.append(dict(mz=pm, it=pi, prec=s["prec"], adduct=s["adduct"], instr=s["instr"],\n                              ce=s["ce"] if np.isfinite(s["ce"]) else 25.0, mode=float(s["mode"])))\n        za = self._run(single, items, args)\n        if za is not None:\n            out.append(za.mean(0))\n        if merged:\n            mz, it = _merge_peaks([self._clip(s) for s in spectra])\n            pm = float(np.median([s["prec"] for s in spectra]))\n            a, b = p_prep_peaks(mz, it, pm)\n            zb = self._run(merged, [dict(mz=a, it=b, prec=pm, adduct=spectra[0]["adduct"], instr=spectra[0]["instr"],\n                                         ce=25.0, mode=float(np.mean([s["mode"] for s in spectra])))], args)\n            if zb is not None:\n                out.append(zb.mean(0))\n        return np.mean(out, axis=0).astype(np.float32) if out else None\n\n    @staticmethod\n    def _clip(s):\n        """Zenith Engine 2 input: peaks <= prec + 2, base-peak normalised."""\n        k = s["mz"] <= s["prec"] + 2.0\n        a, b = s["mz"][k], s["it"][k]\n        return a, (b / b.max() if len(b) and b.max() > 0 else b)\n\n    def _z_ahmed(self, spectra):\n        spectra = spectra[:A_MAX_SPECTRA]\n        f32 = np.float32\n\n        def args(items, T):\n            return (T([x["prec"] for x in items], f32), T([x["adduct_ix"] for x in items]),\n                    T([x["ce"] for x in items], f32), T([x["ce_known"] for x in items], f32),\n                    T([x["n_merged"] / 4.0 for x in items], f32), T([x["mode"] for x in items], f32))\n\n        items_s = []\n        for s in spectra:\n            pm, pi = a_prep_peaks(s["mz"], s["it"], s["prec"])\n            ok = np.isfinite(s["ce_signed"])\n            items_s.append(dict(mz=pm, it=pi, prec=s["prec"], adduct_ix=A_AD_IX.get(s["adduct"], A_AD_IX["<unk>"]),\n                                ce=s["ce_signed"] if ok else 0.0, ce_known=1.0 if ok else 0.0,\n                                n_merged=min(max(1, s["ce_n"]), 8), mode=float(s["mode"])))\n        items_m = []\n        for md in (1, -1):\n            grp = [s for s in spectra if s["mode"] == md]\n            if not grp:\n                continue\n            mm, ii = _merge_peaks([(s["mz"], s["it"]) for s in grp], chain=True)\n            prec = float(np.median([s["prec"] for s in grp]))\n            pm, pi = a_prep_peaks(mm, ii, prec)\n            ces = [s["ce_signed"] for s in grp if np.isfinite(s["ce_signed"])]\n            ads = [s["adduct"] for s in grp]\n            ad = max(ads, key=ads.count)\n            items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=A_AD_IX.get(ad, A_AD_IX["<unk>"]),\n                                ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                                n_merged=min(sum(max(1, s["ce_n"]) for s in grp), 8), mode=float(md)))\n        zs = self._run(self.models, items_s, args, autocast=True)\n        zm = self._run(self.models, items_m, args, autocast=True)\n        if zs is None:\n            return None\n        return (0.5 * (zs.mean(0) + (zm.mean(0) if zm is not None else zs.mean(0)))).astype(np.float32)\n\n    # -- candidate scores\n    def fz(self, z, keys, smiles):\n        out = np.full(len(keys), np.nan, np.float32)\n        if z is None or not len(keys):\n            return out\n        miss = [i for i, k in enumerate(keys) if k not in self._cache]\n        if miss:\n            if len(self._cache) > 300_000:\n                self._cache.clear()\n            bits, ok = self.feat([smiles[i] for i in miss])\n            for j, i in enumerate(miss):\n                self._cache[keys[i]] = np.packbits(bits[j]) if ok[j] else None\n        z = np.asarray(z, np.float32)\n        for i, k in enumerate(keys):\n            p = self._cache.get(k)\n            if p is not None:\n                out[i] = float(np.unpackbits(p)[:self.nbits] @ z)\n        return out\n\n    def release(self):\n        """Free the GPU after pass 1 (z is cached in the plans; f.z is numpy)."""\n        self.models = []\n        try:\n            torch, _, _ = _torch()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n        except Exception:\n            pass\n\n\ndef load_bank(kind, paths, bits_path, device=None):\n    """kind "own": family P checkpoints (fp_*.pt; \'merged\' in the basename -> merged model), fp_bits sha1 asserted\n    against each checkpoint\'s fp_bits_sha1 when present. kind "prv": prvsiyan\'s family-P set (same model and\n    featuriser, no sha1). kind "ahmed": family A checkpoints (fpnet_*.pt)."""\n    torch, _, _ = _torch()\n    if kind not in P_KINDS + ("ahmed",):\n        raise ValueError(kind)\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    bits = np.load(bits_path)\n    sha = bits_sha1(bits)\n    paths = sorted(paths)\n    if not paths:\n        raise FileNotFoundError(f"no {kind} FPNet checkpoints")\n    models, merged = [], []\n    for p in paths:\n        ck = torch.load(p, map_location="cpu", weights_only=False)\n        if int(ck["nbits"]) != len(bits):\n            raise ValueError(f"{p}: nbits {ck[\'nbits\']} != fp_bits {len(bits)}")\n        if kind in P_KINDS:\n            if ck.get("fp_bits_sha1") not in (None, sha):\n                raise ValueError(f"{p}: fp_bits sha1 {ck[\'fp_bits_sha1\']} != {sha}")\n            net = p_model(int(ck["nbits"]), d=int(ck["d"]), layers=int(ck["layers"]))\n        else:\n            if ck.get("arch", "fpnet") != "fpnet":\n                raise ValueError(f"{p}: arch {ck.get(\'arch\')} not supported")\n            n_ad = ck["model"]["ad.weight"].shape[0]\n            if n_ad != len(A_ADDUCTS):\n                raise ValueError(f"{p}: adduct embedding {n_ad} != vendored ADDUCT_LIST {len(A_ADDUCTS)}")\n            net = a_model(int(ck["nbits"]), d=int(ck["d"]), layers=int(ck["layers"]), heads=int(ck.get("heads", 8)))\n        net.load_state_dict(ck["model"])\n        models.append(net.to(device).eval())\n        merged.append(kind in P_KINDS and "merged" in os.path.basename(p))\n    if kind in P_KINDS and not (any(merged) and not all(merged)):\n        raise ValueError(f"{kind} bank needs at least one single and one merged checkpoint")\n    return FPBank(kind, models, merged, Featuriser(kind, bits), device, sha1=sha, paths=paths)\n\n\n# ---------------------------------------------------------------------------------------------- dataset discovery\nOWN_CKPT_GLOB = "**/fp_*.pt"                 # our FPNet export: fp_single_s0/s1, fp_merged_m0/m1 + fp_bits_sha1.txt\nAHMED_CKPT_GLOB = "**/fpnet_full1.pt"        # casmi26-fpnet-full1\nPRV_DATASET = "casmi26-fp-models-v2"         # prvsiyan: fp_single_s2.pt, fp_merged_m1.pt (no fp_bits_sha1.txt)\n\n\ndef own_ckpt_paths(input_dir):\n    """Our FPNet export, found by content, not by dataset name: the folder whose fp_*.pt sit next to\n    fp_bits_sha1.txt (written by build/fpnet_train.py; prvsiyan\'s casmi26-fp-models-v2 has no such file)."""\n    import glob\n    dirs = {}\n    for p in glob.glob(os.path.join(input_dir, OWN_CKPT_GLOB), recursive=True):\n        dirs.setdefault(os.path.dirname(p), []).append(p)\n    ours = sorted((d for d in dirs if os.path.exists(os.path.join(d, "fp_bits_sha1.txt"))), key=len)\n    return sorted(dirs[ours[0]]) if ours else []\n\n\ndef find_bank_files(kind, input_dir, bits_dirs=None):\n    """-> (checkpoint paths, fp_bits path, parity reference dir). The fp_bits.npy is chosen by content, not by\n    dataset name: for "own" the one whose sha1 equals the checkpoints\' fp_bits_sha1 (it sits next to coco_fp.npy);\n    for "prv" the one next to coco_fp.npy whose length equals the checkpoint\'s nbits (as zenith\'s Engine 2);\n    for "ahmed" the one whose length equals the checkpoint\'s nbits (casmi26-v2-pool, next to pool_fp.npy).\n    bits_dirs: where to look for fp_bits.npy (default: input_dir, recursively)."""\n    import glob\n    torch, _, _ = _torch()\n    if kind == "own":\n        paths = own_ckpt_paths(input_dir)\n    elif kind == "prv":\n        paths = sorted(p for p in glob.glob(os.path.join(input_dir, OWN_CKPT_GLOB), recursive=True)\n                       if PRV_DATASET in p)                       # as zenith Engine 2\n    else:\n        paths = sorted(glob.glob(os.path.join(input_dir, AHMED_CKPT_GLOB), recursive=True))[:1]\n    if not paths:\n        raise FileNotFoundError(f"{kind}: no FPNet checkpoints under {input_dir} -- attach "\n                                + {"own": "our FPNet export (fp_*.pt + fp_bits_sha1.txt, e.g. casmi-fpnet)",\n                                   "prv": f"prvsiyan/{PRV_DATASET} (fp_single_s2.pt, fp_merged_m1.pt)"}.get(\n                                    kind, "casmi26-fpnet-full1 (fpnet_full1.pt)"))\n    ck = torch.load(paths[0], map_location="cpu", weights_only=False)\n    want_sha, want_n = ck.get("fp_bits_sha1"), int(ck["nbits"])\n    txt = os.path.join(os.path.dirname(paths[0]), "fp_bits_sha1.txt")\n    if kind == "own" and not want_sha and os.path.exists(txt):\n        want_sha = open(txt).read().strip()\n    cands = [h for d in (bits_dirs or [input_dir])\n             for h in glob.glob(os.path.join(str(d), "**/fp_bits.npy"), recursive=True)]\n    for b in sorted(cands, key=len):\n        bits = np.load(b)\n        if len(bits) != want_n or (kind == "own" and want_sha and bits_sha1(bits) != want_sha):\n            continue\n        if kind == "prv" and not os.path.exists(os.path.join(os.path.dirname(b), "coco_fp.npy")):\n            continue\n        return paths, b, os.path.dirname(b)\n    raise FileNotFoundError(f"{kind}: no fp_bits.npy with nbits {want_n}" + (f", sha1 {want_sha}" if want_sha else "")\n                            + " -- attach the COCONUT arrays dataset with fp_bits.npy + coco_fp.npy (own, prv) or "\n                            "casmi26-v2-pool (ahmed)")\n\n\ndef parity_reference(kind, ref_dir, n=50, seed=0):\n    """(smiles, packed bits) of n random precomputed rows next to fp_bits.npy: coco_meta.pkl + coco_fp.npy (own),\n    pool_meta.parquet + pool_fp.npy (ahmed). None when the files are absent."""\n    rng = np.random.default_rng(seed)\n    if kind in P_KINDS:\n        import pickle\n        meta, fp = os.path.join(ref_dir, "coco_meta.pkl"), os.path.join(ref_dir, "coco_fp.npy")\n        if not (os.path.exists(meta) and os.path.exists(fp)):\n            return None\n        smiles = np.asarray(pickle.load(open(meta, "rb"))["smiles"], dtype=object)\n    else:\n        import pandas as pd\n        meta, fp = os.path.join(ref_dir, "pool_meta.parquet"), os.path.join(ref_dir, "pool_fp.npy")\n        if not (os.path.exists(meta) and os.path.exists(fp)):\n            return None\n        smiles = pd.read_parquet(meta, columns=["smiles"]).smiles.to_numpy(dtype=object)\n    packed = np.load(fp, mmap_mode="r")\n    if len(packed) != len(smiles):\n        return None\n    idx = np.sort(rng.choice(len(smiles), size=min(n, len(smiles)), replace=False))\n    return list(smiles[idx]), np.asarray(packed[idx])\n', 'casmi_scratch/kernels.py': '"""Numba kernels: spectrum cleaning and spectral entropy similarity (phase1, verbatim logic)."""\nimport numpy as np\nfrom numba import njit, prange\n\n\n@njit(cache=True, fastmath=True)\ndef clean(mz, it, floor, topk, power, ent_weight):\n    """Drop peaks below floor * max, keep the top-k by intensity (m/z order kept), normalise to sum 1,\n    and entropy-weight low-entropy spectra."""\n    n = len(mz)\n    if n == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx:\n            mx = it[i]\n    if mx <= 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx\n    c = 0\n    for i in range(n):\n        if it[i] >= thr:\n            c += 1\n    idx = np.empty(c, np.int64)\n    j = 0\n    for i in range(n):\n        if it[i] >= thr:\n            idx[j] = i\n            j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c):\n            v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk):\n            k2[i] = idx[o[i]]\n        k2.sort()\n        idx = k2\n        c = topk\n    om = np.empty(c, np.float32)\n    oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]\n        v = it[idx[i]] ** power\n        oi[i] = v\n        s += v\n    if s > 0:\n        for i in range(c):\n            oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0:\n                S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S\n            s2 = 0.0\n            for i in range(c):\n                oi[i] = oi[i] ** w\n                s2 += oi[i]\n            if s2 > 0:\n                for i in range(c):\n                    oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0\n    j = 0\n    n = len(qmz)\n    m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0:\n            SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0:\n            SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0\n    tot = 0.0\n    buf = np.empty(n + m, np.float64)\n    b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol:\n            buf[b] = qp[i]\n            i += 1\n        elif d > tol:\n            buf[b] = cp[j]\n            j += 1\n        else:\n            buf[b] = qp[i] + cp[j]\n            i += 1\n            j += 1\n        b += 1\n    while i < n:\n        buf[b] = qp[i]\n        i += 1\n        b += 1\n    while j < m:\n        buf[b] = cp[j]\n        j += 1\n        b += 1\n    for x in range(b):\n        tot += buf[x]\n    if tot <= 0:\n        return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0:\n            SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    """Entropy similarity of one cleaned query against library spectra `cand` (cleaned on the fly)."""\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]\n        a = off[c]\n        b = off[c + 1]\n        if b <= a:\n            continue\n        cm, cp = clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0:\n            continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n', 'casmi_scratch/library.py': '"""Reference spectral library (train.parquet) and the Class-1 direct match `lv`."""\nimport time\nfrom dataclasses import dataclass, replace\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom .chem import formula_mass, neutral_mass\nfrom .kernels import search\nfrom .spectra import clean_spectrum\n\n\n@dataclass\nclass Library:\n    off: np.ndarray      # (n + 1,) peak offsets\n    mz: np.ndarray       # flat m/z\n    it: np.ndarray       # flat intensities\n    nm: np.ndarray       # neutral mass per spectrum\n    ik: np.ndarray       # inchikey14 per spectrum (object)\n    order: np.ndarray    # spectrum indices sorted by nm (finite first)\n    snm: np.ndarray      # nm[order]\n    n_ok: int            # number of finite nm\n    pol: np.ndarray = None   # +1 / -1 / 0 per spectrum, from the adduct\'s charge sign (analog search)\n\n    def __len__(self):\n        return len(self.off) - 1\n\n    def window(self, target, tol_da):\n        snm = self.snm[:self.n_ok]\n        lo = np.searchsorted(snm, target - tol_da, "left")\n        hi = np.searchsorted(snm, target + tol_da, "right")\n        return self.order[lo:hi]\n\n\ndef build_library(off, mz, it, prec, adduct, ik, formula=None, struct_mass=True):\n    """Library from flat arrays. With struct_mass, the label formula\'s exact mass replaces the precursor-derived\n    neutral mass wherever it parses (the base\'s TIER-1 fix)."""\n    nm = neutral_mass(prec, adduct)\n    if struct_mass and formula is not None:\n        fm = {f: formula_mass(f) for f in set(formula.tolist())}\n        nm_s = np.array([fm.get(f, np.nan) for f in formula], np.float64)\n        nm = np.where(np.isfinite(nm_s), nm_s, nm)\n    ok = np.isfinite(nm)\n    order = np.argsort(np.where(ok, nm, 1e18), kind="mergesort")\n    return Library(off=np.asarray(off, np.int64), mz=np.asarray(mz, np.float32), it=np.asarray(it, np.float32),\n                   nm=nm, ik=np.asarray(ik, dtype=object), order=order, snm=nm[order], n_ok=int(ok.sum()),\n                   pol=adduct_polarity(adduct))\n\n\ndef adduct_polarity(adduct):\n    """\'[M+H]+\' -> 1, \'[M-H]-\' -> -1, anything else -> 0."""\n    ad = np.asarray(adduct, dtype=object)\n    return np.array([1 if isinstance(a, str) and a.endswith("+") else -1 if isinstance(a, str) and a.endswith("-")\n                     else 0 for a in ad], np.int8)\n\n\ndef remap_keys(lib, keymap):\n    """A Library view whose ik are mapped through keymap (train inchikey14 -> R1 scorer_key); unmapped keys kept.\n    Peaks are shared, not copied."""\n    u, inv = np.unique(lib.ik.astype(str), return_inverse=True)\n    mapped = np.array([keymap.get(k, k) for k in u], dtype=object)\n    return replace(lib, ik=mapped[inv])\n\n\ndef load_library(path, struct_mass=True, rows=None):\n    """train.parquet -> Library. rows: optional row indices to keep (validation bans)."""\n    t0 = time.time()\n    t = pq.read_table(path, columns=["inchikey14", "adduct", "precursor_mz", "molecular_formula",\n                                     "ms2_mzs", "ms2_normalized_intensities"])\n    if rows is not None:\n        t = t.take(pa.array(np.asarray(rows, np.int64)))\n    mzc = t.column("ms2_mzs").combine_chunks()\n    itc = t.column("ms2_normalized_intensities").combine_chunks()\n    lib = build_library(\n        off=mzc.offsets.to_numpy().astype(np.int64),\n        mz=mzc.values.to_numpy(zero_copy_only=False), it=itc.values.to_numpy(zero_copy_only=False),\n        prec=t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64),\n        adduct=np.asarray(t.column("adduct").cast(pa.string()).to_pylist(), dtype=object),\n        ik=np.asarray(t.column("inchikey14").cast(pa.string()).to_pylist(), dtype=object),\n        formula=np.asarray(t.column("molecular_formula").cast(pa.string()).to_pylist(), dtype=object),\n        struct_mass=struct_mass)\n    # `.values` ignores a slice offset and `.offsets` index into it, so the pair stays consistent\n    print(f"[library] {len(lib):,} spectra, {lib.n_ok:,} with a neutral mass ({time.time() - t0:.1f}s)", flush=True)\n    return lib\n\n\ndef lib_sim(lib, specs, target, cfg, exclude=None):\n    """Class 1: best entropy similarity per ik14 over library spectra within +-ppm_win of target.\n\n    specs: [(mz, it)] of one molecule. exclude: optional boolean mask over library spectra to skip.\n    """\n    cand = lib.window(target, target * cfg.ppm_win / 1e6)\n    if exclude is not None and len(cand):\n        cand = cand[~exclude[cand]]\n    if len(cand) == 0:\n        return {}\n    agg = {}\n    for mz, it in specs:\n        qm, qp = clean_spectrum(mz, it, cfg)\n        if len(qm) == 0:\n            continue\n        sc = search(qm, qp, cand, lib.off, lib.mz, lib.it, cfg.mz_tol, cfg.int_floor, cfg.max_peaks,\n                    cfg.int_power, cfg.ent_weight)\n        for c, s in zip(cand, sc):\n            k = lib.ik[c]\n            if s > agg.get(k, -1.0):\n                agg[k] = float(s)\n    return agg\n', 'casmi_scratch/metfrag.py': '"""MetFrag-lite `fr`: share of (sqrt) query intensity explained by 1-2 bond-break fragments. Needs RDKit."""\nfrom multiprocessing import Pool as MPool\n\nimport numpy as np\n\nfrom .chem import ELECTRON, MASS\nfrom .kernels import clean\n\nH_ATOM = MASS["H"]\nPROTON = H_ATOM - ELECTRON\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    m = Chem.MolFromSmiles(smi)\n    if m is None:\n        return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = MASS.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM\n    if (w == 0).any():\n        return None\n    return w, [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()], n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop:\n            continue\n        adj[a].append(b)\n        adj[b].append(a)\n    seen = np.zeros(n, bool)\n    comps = []\n    for s in range(n):\n        if seen[s]:\n            continue\n        stack, cur = [s], [s]\n        seen[s] = True\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]:\n                    seen[v] = True\n                    stack.append(v)\n                    cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_bonds=60, max_breaks=2):\n    g = mol_graph(smi)\n    if g is None:\n        return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds:\n        return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0:\n        return 0.0\n    ion = np.sort(np.concatenate([frag_mass + dh * H_ATOM + (PROTON if mode > 0 else -PROTON) for dh in h_shifts]))\n    w = np.sqrt(np.asarray(peak_int, float))\n    tot = w.sum()\n    if tot <= 0:\n        return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef _frag_job(args):\n    smi, max_bonds = args\n    try:\n        return fragment_masses(smi, max_bonds)\n    except Exception:\n        return np.zeros(0)\n\n\ndef frag_scores(cand_smiles, specs, mode, cfg):\n    """fr per candidate: best explained share over the molecule\'s spectra. mode > 0 positive, < 0 negative."""\n    if not len(cand_smiles):\n        return np.zeros(0, np.float32)\n    jobs = [(s, cfg.max_bonds) for s in cand_smiles]\n    if cfg.frag_workers > 1:\n        with MPool(cfg.frag_workers) as mp:\n            frags = mp.map(_frag_job, jobs, chunksize=8)\n    else:\n        frags = [_frag_job(j) for j in jobs]\n    peaks = []\n    for mz, it in specs:\n        m2, i2 = clean(np.asarray(mz, np.float32), np.asarray(it, np.float32), cfg.int_floor, cfg.max_peaks, 1.0, False)\n        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n    out = np.zeros(len(cand_smiles), np.float32)\n    for j, f in enumerate(frags):\n        out[j] = max((explain_score(f, a, b, mode=mode, tol=cfg.mz_tol) for a, b in peaks), default=0.0)\n    return out\n', 'casmi_scratch/np_labels.py': '"""NP labels for every pool row: exact NP-table match, else the nearest labelled NP-table structure (kNN), plus Ertl\nNP-likeness. Ported from build/np_signals_core.py (Track 4b: morgan_fp, knn_label, np_scores).\n\nlabel_source: 0 none (neutral), 1 table (exact scorer_key match), 2 knn (Morgan r2 2048, Tanimoto >= thr).\n"""\nimport os\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom .pool_build import run_chunked\n\nLABELS = ["np_pathway", "np_superclass", "np_class"]\nKNN_THR = 0.7\n_G = {}\n\n\ndef morgan_fp(smiles, nbits=2048):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog("rdApp.*")\n    gen = _G.setdefault("gen", rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=nbits))\n    m = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\n    return None if m is None else gen.GetFingerprint(m)\n\n\ndef _init_bank(bank_smiles):\n    fps = [morgan_fp(s) for s in bank_smiles]\n    _G["bank_idx"] = np.array([i for i, f in enumerate(fps) if f is not None], np.int64)\n    _G["bank"] = [f for f in fps if f is not None]\n\n\ndef _knn_chunk(smis):\n    """-> [(bank row index or -1, best Tanimoto)] against the bank set up by _init_bank."""\n    from rdkit import DataStructs\n    out = []\n    for s in smis:\n        fp = morgan_fp(s)\n        if fp is None or not _G["bank"]:\n            out.append((-1, 0.0))\n            continue\n        sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, _G["bank"]))\n        j = int(sims.argmax())\n        out.append((int(_G["bank_idx"][j]), float(sims[j])))\n    return out\n\n\ndef knn_nearest(query_smiles, bank_smiles, ckpt_dir, chunk=20_000, workers=8, log=print):\n    """Nearest bank structure per query (checkpointed) -> DataFrame(bank_j, sim) in query order."""\n    return run_chunked(list(query_smiles), _knn_chunk, ckpt_dir, "knn", ["bank_j", "sim"], chunk, workers,\n                       initializer=_init_bank, initargs=(list(bank_smiles),), log=log)\n\n\ndef assign_labels(keys, smiles, table, nearest=None, thr=KNN_THR):\n    """Pool keys -> label frame (index key): LABELS, label_source, knn_sim.\n\n    table: NP-table metadata indexed by recomputed key (LABELS + extra columns). nearest: optional function\n    (query smiles list, bank smiles list) -> DataFrame(bank_j, sim); without it, keys outside the table stay unlabelled.\n    """\n    keys = pd.Index(np.asarray(keys, dtype=object), name="key")\n    out = pd.DataFrame(index=keys, columns=LABELS, dtype=object)\n    out["label_source"] = np.int8(0)\n    out["knn_sim"] = np.float32(np.nan)\n    hit = keys.isin(table.index)\n    t = table.reindex(keys[hit])\n    has = t.np_superclass.notna().to_numpy() | t.np_pathway.notna().to_numpy()\n    out.loc[keys[hit][has], LABELS] = t.loc[has, LABELS].to_numpy()\n    out.loc[keys[hit][has], "label_source"] = np.int8(1)\n\n    bank = table[table.np_superclass.notna() | table.np_pathway.notna()]\n    need = out.label_source.to_numpy() == 0\n    if nearest is not None and need.any() and len(bank):\n        q = np.asarray(smiles, dtype=object)[need]\n        nn = nearest(list(q), list(bank["smiles"]))\n        j, s = nn.bank_j.to_numpy(np.int64), nn.sim.to_numpy(np.float64)\n        ok = (j >= 0) & (s >= thr)\n        idx = keys[need]\n        out.loc[idx[ok], LABELS] = bank.iloc[j[ok]][LABELS].to_numpy()\n        out.loc[idx[ok], "label_source"] = np.int8(2)\n        out.loc[idx, "knn_sim"] = np.where(j >= 0, s, np.nan).astype(np.float32)\n    return out\n\n\n# ---------------------------------------------------------------- NP-likeness (Ertl, RDKit Contrib)\n\ndef _np_model():\n    if "np_m" not in _G:\n        from rdkit.Chem import RDConfig\n        p = os.path.join(RDConfig.RDContribDir, "NP_Score")\n        if p not in sys.path:\n            sys.path.append(p)\n        import npscorer\n        _G["np_f"], _G["np_m"] = npscorer, npscorer.readNPModel()\n    return _G["np_f"], _G["np_m"]\n\n\ndef _np_chunk(smis):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    f, m = _np_model()\n    out = []\n    for s in smis:\n        mol = Chem.MolFromSmiles(s) if isinstance(s, str) else None\n        v = np.nan\n        if mol is not None:\n            try:\n                v = float(f.scoreMol(mol, m))\n            except Exception:\n                pass\n        out.append((v,))\n    return out\n\n\ndef np_scores(smiles, ckpt_dir, chunk=50_000, workers=8, log=print):\n    """Ertl NP-likeness per SMILES (NaN where RDKit cannot parse), checkpointed -> float32 array."""\n    return run_chunked(list(smiles), _np_chunk, ckpt_dir, "npscore", ["np_score"], chunk, workers,\n                       log=log).np_score.to_numpy(np.float32)\n', 'casmi_scratch/npc.py': '"""NPClassifier labels for the pool (R2): fingerprint, model loading, top-k decoding, pool_npc.parquet I/O.\n\nNPClassifier = Kim et al., J. Nat. Prod. 2021, github.com/mwang87/NP-Classifier (MIT). Weights: Zenodo 5068687\n(`model.zip` -> NP_classifier_{pathway,superclass,class}_V1.hdf5, Keras / TF 2.3). Label names:\n`Classifier/dict/index_v1.json` ({"Pathway"|"Superclass"|"Class": {name: output index}}). Inputs, as the repo\'s\n`fingerprint_handler.calculate_fingerprint(smiles, 2)`: `input_2048` = radius-0 counted Morgan bits, `input_4096` =\nradius-1 and radius-2 counted bits (explicit H, 2048 bits each, only environments of exactly that radius).\nLabels are computed once on Colab; Kaggle only reads pool_npc.parquet (no TensorFlow there).\n\nThe V1 weights were trained on the Morgan bits of RDKit <= 2021.03; RDKit 2021.09 changed them, and on today\'s bits\nthe net outputs noise (R2.0 FAIL, gate_report). Real labelling therefore takes its fingerprints from\n`fp_subprocess` (resources/npc_fp_worker.py under rdkit-pypi 2021.3.5). `predict` without an explicit fp_fn refuses a\nnewer RDKit; tests and the LR fallback (trained and applied on the same bits) call `fingerprints` directly.\n"""\nimport json\nimport os\nimport subprocess\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nHEADS = ("pathway", "superclass", "class")\nINDEX_KEY = dict(pathway="Pathway", superclass="Superclass", **{"class": "Class"})\nNPC_FILE = "pool_npc.parquet"\nNSC = 3                                  # superclasses kept per structure\nFP_RDKIT = "2021.3.5"                    # rdkit-pypi release with the Morgan bits the V1 weights were trained on\nFP_PYTHON = "3.9"                        # newest CPython with rdkit-pypi 2021.3.5 wheels\nFP_WORKER = Path(__file__).resolve().parent / "resources" / "npc_fp_worker.py"\n\n\ndef fingerprint(smiles, radi=2):\n    """-> (formula (2048,), binary (2048*radi,)) int32 counts, or None if RDKit cannot parse.\n\n    Same arithmetic as NP-Classifier\'s calculate_fingerprint: bit i of radius r counts the environments of exactly\n    radius r that set it.\n    """\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import rdMolDescriptors\n    RDLogger.DisableLog("rdApp.*")\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\n    if mol is None or mol.GetNumAtoms() == 0:\n        return None\n    mol = Chem.AddHs(mol)\n    formula, binary = np.zeros(2048, np.int32), np.zeros(2048 * radi, np.int32)\n    for r in range(radi + 1):\n        bi = {}\n        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=r, bitInfo=bi, nBits=2048)\n        for i in fp.GetOnBits():\n            n = sum(1 for e in bi[i] if e[1] == r)\n            if n == 0:\n                continue\n            if r == 0:\n                formula[i] = n\n            else:\n                binary[2048 * (r - 1) + i] = n\n    return formula, binary\n\n\ndef fingerprints(smiles):\n    """List of SMILES -> (F (n, 2048), B (n, 4096), ok (n,)); rows RDKit cannot parse are zero and ok=False."""\n    n = len(smiles)\n    F, B, ok = np.zeros((n, 2048), np.float32), np.zeros((n, 4096), np.float32), np.zeros(n, bool)\n    for j, s in enumerate(smiles):\n        fp = fingerprint(s)\n        if fp is not None:\n            F[j], B[j], ok[j] = fp[0], fp[1], True\n    return F, B, ok\n\n\ndef label_names(index_json):\n    """index_v1.json -> {head: [name per output index]} (an index named twice keeps its first name)."""\n    d = json.load(open(index_json)) if isinstance(index_json, (str, os.PathLike)) else index_json\n    out = {}\n    for head in HEADS:\n        m = d[INDEX_KEY[head]]\n        names = [None] * (max(m.values()) + 1)\n        for name, i in sorted(m.items(), key=lambda x: (x[1], x[0])):\n            if names[i] is None:\n                names[i] = name\n        out[head] = names\n    return out\n\n\ndef load_models(model_dir):\n    """{head: Keras model} from the three NP_classifier_*_V1.hdf5 files (tf_keras first: they are Keras 2 files)."""\n    try:\n        import tf_keras as K\n    except ImportError:\n        from tensorflow import keras as K\n    files = dict(pathway="NP_classifier_pathway_V1.hdf5", superclass="NP_classifier_superclass_V1.hdf5",\n                 **{"class": "NP_classifier_class_V1.hdf5"})\n    return {h: K.models.load_model(os.path.join(model_dir, f), compile=False) for h, f in files.items()}\n\n\ndef model_inputs(model, F, B):\n    """Feed the two fingerprints in the model\'s own input order (matched on width)."""\n    by_w = {2048: F, 4096: B}\n    return [by_w[int(t.shape[-1])] for t in model.inputs]\n\n\ndef topk(P, names, k):\n    """P (n, m) probabilities -> (labels (n, k) object, probs (n, k) float32), best first."""\n    P = np.asarray(P, np.float32)\n    k = min(k, P.shape[1])\n    idx = np.argsort(-P, axis=1, kind="stable")[:, :k]\n    lab = np.asarray(names, dtype=object)[idx]\n    return lab, np.take_along_axis(P, idx, axis=1)\n\n\ndef decode(preds, names, ok):\n    """{head: P} -> label frame: sc1..3 / sp1..3 (superclass top 3), pathway / pathway_p, class / class_p."""\n    sc, sp = topk(preds["superclass"], names["superclass"], NSC)\n    out = {f"sc{i + 1}": sc[:, i] for i in range(sc.shape[1])}\n    out.update({f"sp{i + 1}": sp[:, i] for i in range(sp.shape[1])})\n    for head in ("pathway", "class"):\n        lab, p = topk(preds[head], names[head], 1)\n        out[head], out[f"{head}_p"] = lab[:, 0], p[:, 0]\n    df = pd.DataFrame(out)\n    df.loc[~ok, [c for c in df.columns if not c.endswith("_p") and not c.startswith("sp")]] = None\n    df.loc[~ok, [c for c in df.columns if c.endswith("_p") or c.startswith("sp")]] = np.nan\n    return df\n\n\ndef rdkit_bits_ok(version=None):\n    """True iff this RDKit gives the Morgan bits NP-Classifier V1 was trained on (<= 2021.03)."""\n    if version is None:\n        import rdkit\n        version = rdkit.__version__\n    return tuple(int(x) for x in version.split(".")[:2]) <= (2021, 3)\n\n\ndef local_fingerprints_checked(smiles):\n    """In-process fingerprints, only on an RDKit with the old bits (else the labels are noise)."""\n    if not rdkit_bits_ok():\n        import rdkit\n        raise RuntimeError(f"RDKit {rdkit.__version__} has post-2021.09 Morgan bits; NP-Classifier needs "\n                           f"fp_subprocess (rdkit-pypi {FP_RDKIT}) or an explicit fp_fn")\n    return fingerprints(smiles)\n\n\ndef fp_env_cmd(python=FP_PYTHON):\n    """Command prefix that runs the fingerprint worker in a uv-managed Python with rdkit-pypi FP_RDKIT.\n    pandas / tqdm only for the repo\'s fingerprint_handler (parity check)."""\n    return ["uv", "run", "--no-project", "--python", python, "--with", f"rdkit-pypi=={FP_RDKIT}", "--with", "numpy<2",\n            "--with", "pandas", "--with", "tqdm", "python", "-I", str(FP_WORKER)]\n\n\ndef fp_subprocess(smiles, cmd=None, workers=None, ref_dir=None, workdir=None):\n    """SMILES -> (F (n, 2048) f32, B (n, 4096) f32, ok (n,), meta {rdkit, parity}) from the worker in its own env."""\n    cmd = list(cmd or fp_env_cmd())\n    smiles = ["" if not isinstance(x, str) else x.replace("\\n", "") for x in smiles]\n    with tempfile.TemporaryDirectory(dir=workdir) as d:\n        inp, out = os.path.join(d, "in.txt"), os.path.join(d, "out.npz")\n        with open(inp, "w", encoding="utf-8") as fh:\n            fh.write("\\n".join(smiles) + "\\n")\n        args = cmd + ["--inp", inp, "--out", out, "--workers", str(workers or os.cpu_count() or 1)]\n        if ref_dir:\n            args += ["--ref", str(ref_dir)]\n        r = subprocess.run(args, capture_output=True, text=True)\n        if r.returncode != 0 or not os.path.exists(out):\n            raise RuntimeError(f"npc_fp_worker failed ({r.returncode}):\\n{r.stdout[-2000:]}\\n{r.stderr[-3000:]}")\n        with np.load(out) as z:\n            assert len(z["ok"]) == len(smiles), (len(z["ok"]), len(smiles))\n            return (z["F"].astype(np.float32), z["B"].astype(np.float32), z["ok"].astype(bool),\n                    dict(rdkit=str(z["rdkit"]), parity=int(z["parity"])))\n\n\ndef fp_fn_subprocess(**kw):\n    """fp_fn for predict / label_all: fingerprints from the old-RDKit worker."""\n    return lambda smiles: fp_subprocess(smiles, **kw)[:3]\n\n\ndef predict(smiles, models, names, fp_fn=None, batch=4096):\n    """SMILES -> decoded label frame (one row per SMILES, in order). fp_fn(smiles) -> (F, B, ok); default = in-process\n    fingerprints, which refuse a new RDKit (pass fp_fn_subprocess() for real labels)."""\n    F, B, ok = (fp_fn or local_fingerprints_checked)(list(smiles))\n    F, B = np.asarray(F, np.float32), np.asarray(B, np.float32)\n    preds = {h: m.predict(model_inputs(m, F, B), batch_size=batch, verbose=0) for h, m in models.items()}\n    return decode(preds, names, np.asarray(ok, bool))\n\n\ndef label_all(smiles, models, names, ckpt_dir, fp_fn=None, chunk=20_000, log=print):\n    """Every SMILES -> decoded label frame, one checkpoint parquet per chunk (a rerun resumes)."""\n    from .pool_build import run_chunked\n    cols = [f"sc{i + 1}" for i in range(NSC)] + [f"sp{i + 1}" for i in range(NSC)] + \\\n        ["pathway", "pathway_p", "class", "class_p"]\n\n    def fn(s):\n        return list(predict(s, models, names, fp_fn)[cols].itertuples(index=False, name=None))\n    return run_chunked(list(smiles), fn, ckpt_dir, "npc", cols, chunk, workers=1, log=log)\n\n\n# ---------------------------------------------------------------- pool_npc.parquet\n\ndef write_npc(pool_dir, labels):\n    """labels: frame aligned row-for-row with pool_key.npy (mass order) -> pool_npc.parquet (with a key column)."""\n    keys = np.load(os.path.join(pool_dir, "pool_key.npy")).astype(str)\n    assert len(labels) == len(keys), (len(labels), len(keys))\n    df = labels.reset_index(drop=True).copy()\n    df.insert(0, "key", keys)\n    for c in df.columns:\n        if c.startswith("sp") or c.endswith("_p"):\n            df[c] = df[c].astype(np.float32)\n    path = os.path.join(pool_dir, NPC_FILE)\n    pq.write_table(pa.Table.from_pandas(df, preserve_index=False), path + ".tmp")\n    os.replace(path + ".tmp", path)\n    return path\n\n\ndef load_npc(pool_dir):\n    """pool_npc.parquet -> (superclass codes (n, 3) int16 with -1 = none, probs (n, 3) float32, names list),\n    aligned with pool_key.npy. The file\'s key column must match pool_key.npy exactly."""\n    t = pq.read_table(os.path.join(pool_dir, NPC_FILE)).to_pandas()\n    keys = np.load(os.path.join(pool_dir, "pool_key.npy")).astype(str)\n    assert len(t) == len(keys) and np.array_equal(t.key.to_numpy().astype(str), keys), "pool_npc misaligned"\n    cols = [c for c in (f"sc{i + 1}" for i in range(NSC)) if c in t]\n    names = sorted({v for c in cols for v in t[c].dropna().unique()})\n    code = {n: i for i, n in enumerate(names)}\n    sc = np.full((len(t), len(cols)), -1, np.int16)\n    for j, c in enumerate(cols):\n        sc[:, j] = t[c].map(code).fillna(-1).to_numpy(np.int16)\n    sp = np.nan_to_num(t[[f"sp{i + 1}" for i in range(len(cols))]].to_numpy(np.float32), nan=0.0)\n    return sc, sp, names\n\n\n# ---------------------------------------------------------------- fallback (R2.0 FAIL): linear model on table labels\n\ndef lr_fit(smiles, labels, max_iter=200, seed=0):\n    """Multinomial logistic regression on the NPClassifier fingerprint (both blocks, log1p counts)."""\n    from sklearn.linear_model import SGDClassifier\n    F, B, ok = fingerprints(list(smiles))\n    X = np.log1p(np.hstack([F, B]))[ok]\n    y = np.asarray(labels, dtype=object)[ok]\n    clf = SGDClassifier(loss="log_loss", max_iter=max_iter, random_state=seed, n_jobs=-1)\n    clf.fit(X, y)\n    return clf\n\n\ndef lr_predict(clf, smiles):\n    """-> label frame with sc1..3 / sp1..3 like `decode` (pathway / class not available)."""\n    F, B, ok = fingerprints(list(smiles))\n    P = clf.predict_proba(np.log1p(np.hstack([F, B])))\n    sc, sp = topk(P, list(clf.classes_), NSC)\n    df = pd.DataFrame({**{f"sc{i + 1}": sc[:, i] for i in range(sc.shape[1])},\n                       **{f"sp{i + 1}": sp[:, i] for i in range(sp.shape[1])}})\n    df.loc[~ok, [f"sc{i + 1}" for i in range(sc.shape[1])]] = None\n    df.loc[~ok, [f"sp{i + 1}" for i in range(sp.shape[1])]] = np.nan\n    return df\n', 'casmi_scratch/pipeline.py': '"""R0 pipeline: per molecule, centred window -> lv (library), fr (MetFrag-lite), sv (ICEBERG) -> two-tier ranking.\n\nPass 1 plans every deep molecule and collects simulation requests; `sim.simulate` runs them under a deadline;\npass 2 scores and fills 25 slots (stage A tight window, B wider tiers append-only, C mass-nearest top-up).\nEverything is injected (library, pool, worker, frag function), so the validation harness runs the same code.\n"""\nimport time\n\nimport numpy as np\n\nfrom .analog import analog_hits, analog_set, class_vote, molecule_analogs, vote_bonus, vote_conf\nfrom .chem import neutral_mass, query_target\nfrom .library import lib_sim\nfrom .fpnet import molecule_spectra\nfrom .metfrag import frag_scores\nfrom .pool import F_TRAIN\nfrom .rank import Slots, fp_rrf, rank_two_tier\nfrom .sim import build_requests, sim_adducts, sim_priority, simulate\nfrom .spectra import ce_tuple, prep, spectrum_ces, spectrum_sv\n\n\ndef scored_range(pool, target, cfg):\n    """Contiguous pool range scored by stages A + B: +-ppm_win (+-ppm_fallback if empty), widened through\n    ppm_expand until >= topn candidates."""\n    lo, hi = pool.bounds(target, cfg.ppm_win)\n    if hi == lo:\n        lo, hi = pool.bounds(target, cfg.ppm_fallback)\n    if hi - lo < cfg.topn:\n        for ppm in cfg.ppm_expand[1:]:\n            a, b = pool.bounds(target, ppm)\n            lo, hi = min(lo, a), max(hi, b)\n            if hi - lo >= cfg.topn:\n                break\n    return lo, hi\n\n\ndef molecule_target(sub, cfg):\n    nm = neutral_mass(sub.precursor_mz.to_numpy(np.float64), sub.adduct.to_numpy(dtype=object))\n    return query_target(nm, cfg.ppm_center)\n\n\ndef molecule_vote(lib, specs, target, ion_mode, cfg, vote_label, lib_exclude=None):\n    """R2: NPClassifier superclass voted by the delta-list analog library hits -> (label or None, confidence)."""\n    hits = analog_hits(lib, specs, target, cfg, "delta", polarity=1 if ion_mode >= 0 else -1, exclude=lib_exclude,\n                       ppm=cfg.vote_ppm)\n    win, w, _ = class_vote(hits, vote_label.get, cfg.vote_k, cfg.vote_min_sim)\n    return win, vote_conf(w, win)\n\n\ndef plan_molecule(sub, target, lib, pool, cfg, sim_tier1=False, frag_fn=frag_scores, lib_exclude=None,\n                  vote_label=None):\n    """Pass 1 for one molecule: lv, fr on the scored range; covered query spectra; which candidates to simulate;\n    the class vote (R2, when vote_label is given)."""\n    specs = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]\n    lib_hits = lib_sim(lib, specs, target, cfg, exclude=lib_exclude)\n    ion_mode = float(np.mean([1.0 if m == "positive" else -1.0 for m in sub.ionization_mode]))\n    vote = molecule_vote(lib, specs, target, ion_mode, cfg, vote_label, lib_exclude)         if cfg.class_vote and vote_label is not None else (None, 0.0)\n    an_hits, an_s = None, 0.0\n    if cfg.analog_channel:                  # R5: open analogue hits (top few hundred kept; the profile is built later)\n        ta = time.time()\n        pols = [1 if m == "positive" else -1 for m in sub.ionization_mode]\n        an_hits = dict(analog_set(molecule_analogs(lib, specs, pols, target, cfg, lib_exclude), cfg.analog_smin,\n                                  4 * max(cfg.analog_m, cfg.analog_k)))\n        an_s = time.time() - ta\n    lo, hi = scored_range(pool, target, cfg)\n    cand = np.arange(lo, hi)\n    lv = np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32)\n    fr = frag_fn([pool.smiles[c] for c in cand], specs, ion_mode, cfg) if len(cand) else np.zeros(0, np.float32)\n    ppm = np.abs(pool.mass[cand] / target - 1.0) * 1e6\n\n    qs, n_ce_default, adducts = [], 0, sim_adducts(cfg)\n    for r in sub.itertuples():\n        if r.adduct not in adducts:\n            continue\n        ces, defaulted = spectrum_ces(ce_tuple(r.collision_energy_ev), cfg.merge_rule, cfg.ce_default)\n        n_ce_default += defaulted\n        q = prep(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz), cfg)\n        if len(q[0]):\n            qs.append((r.adduct, ces, q))\n    conds = sorted({(a, c) for a, ces, _ in qs for c in ces})\n\n    valid = np.array([k is not None and len(str(k)) >= 14 for k in pool.keys[cand]], bool)\n    to_sim = valid & ((lv < cfg.tau) | sim_tier1) & bool(cfg.use_sim)\n    return dict(target=target, lib_hits=lib_hits, ion_mode=ion_mode, specs=specs,\n                fr={int(c): float(f) for c, f in zip(cand, fr)}, qs=qs, conds=conds, n_cand=len(cand),\n                n_ce_default=n_ce_default, sim_idx=cand[to_sim], vote=vote, an_hits=an_hits, an_s=an_s,\n                prio=sim_priority(lv[to_sim], fr[to_sim], ppm[to_sim]))\n\n\ndef features(plan, idx, pool, sim, cfg, frag_fn=frag_scores):\n    """lv, sv, fr, ppm for pool indices idx (fr computed now for indices pass 1 did not score)."""\n    lv = np.array([plan["lib_hits"].get(pool.keys[c], 0.0) for c in idx], np.float32)\n    miss = [int(c) for c in idx if int(c) not in plan["fr"]]\n    if miss:\n        plan["fr"].update(zip(miss, map(float, frag_fn([pool.smiles[c] for c in miss], plan["specs"],\n                                                       plan["ion_mode"], cfg))))\n    fr = np.array([plan["fr"][int(c)] for c in idx], np.float32)\n    ppm = np.abs(pool.mass[idx] / plan["target"] - 1.0) * 1e6\n    sv = np.zeros(len(idx), np.float32)\n    for j, c in enumerate(idx):\n        k = pool.keys[c]\n        for adduct, ces, q in plan["qs"]:\n            sv[j] = max(sv[j], spectrum_sv(q, [sim.get((k, adduct, ce)) for ce in ces], cfg))\n    return lv, sv, fr, ppm\n\n\ndef class_bonus(plan, idx, pool, cfg):\n    """R2 tier-2 bonus for pool indices idx (None when the vote is off)."""\n    if not cfg.class_vote or pool.npc is None:\n        return None\n    sc, sp, names = pool.npc\n    return vote_bonus(sc[idx], sp[idx], names, *plan.get("vote", (None, 0.0)), cfg.vote_beta)\n\n\ndef as_banks(fp_bank):\n    """None | one FPBank | a list of them -> list (R3 passes one bank, R4 one per family)."""\n    return [] if fp_bank is None else (list(fp_bank) if isinstance(fp_bank, (list, tuple)) else [fp_bank])\n\n\ndef fp_scores(plan, idx, pool, cfg, fp_bank):\n    """R3/R4 f.z for pool indices idx: a list with one array per bank (NaN where a bank has no logits or the SMILES\n    does not parse), or None when the channel is off or no bank has logits for the molecule."""\n    zs = plan.get("fpz")\n    if not cfg.fp_kinds() or fp_bank is None or not zs or all(z is None for z in zs):\n        return None\n    keys, smis = [pool.keys[c] for c in idx], [pool.smiles[c] for c in idx]\n    return [b.fz(z, keys, smis) for b, z in zip(as_banks(fp_bank), zs)]\n\n\ndef analog_scores(plan, idx, pool, cfg, analog_bank):\n    """R5 analogue score for pool indices idx (NaN = not scored: train structures when cfg.analog_exclude_train,\n    unless exempted in plan["an_free"], and structures without a fingerprint), or None when the channel is off or the\n    molecule has fewer than analog_min_n analogues."""\n    prof = plan.get("an_prof")\n    if not cfg.analog_channel or analog_bank is None or prof is None or not len(idx):\n        return None\n    keys = [pool.keys[c] for c in idx]\n    s = analog_bank.score(prof, keys, [pool.smiles[c] for c in idx]).astype(np.float64)\n    if cfg.analog_exclude_train and pool.flags is not None:\n        free = plan.get("an_free") or set()\n        train = (np.asarray(pool.flags[idx], np.int64) & F_TRAIN) != 0\n        s[train & ~np.array([k in free for k in keys], bool)] = np.nan\n    return s\n\n\ndef rank_molecule(plan, pool, sim, cfg, frag_fn=frag_scores, fp_bank=None, analog_bank=None):\n    """Pass 2 for one deep molecule -> (Slots, diag dict, stage-A features)."""\n    target, slots = plan["target"], Slots(cfg.topn)\n    cand = pool.window(target, cfg.ppm_win)\n    if len(cand) == 0:\n        cand = pool.window(target, cfg.ppm_fallback)\n    diag, stage_a = None, None\n    if len(cand):\n        lv, sv, fr, ppm = features(plan, cand, pool, sim, cfg, frag_fn)\n        bonus = class_bonus(plan, cand, pool, cfg)\n        fz = fp_scores(plan, cand, pool, cfg, fp_bank)\n        an = analog_scores(plan, cand, pool, cfg, analog_bank)\n        order = rank_two_tier(lv, sv, fr, ppm, cfg.tau, bonus, fz, cfg.fp_rrf_k, cfg.fp_w(), an, cfg.analog_w)\n        slots.take(pool, cand[order])\n        diag = dict(n_cand=len(cand), n_covered_spec=len(plan["qs"]), n_cond=len(plan["conds"]),\n                    lv_max=float(lv.max()), sv_max=float(sv.max()), fr_max=float(fr.max()),\n                    top_is_tier1=bool(lv[order[0]] >= cfg.tau))\n        if bonus is not None:\n            plain = rank_two_tier(lv, sv, fr, ppm, cfg.tau)\n            diag.update(voted=plan["vote"][0] is not None, bonus_max=float(bonus.max()),\n                        reordered=not np.array_equal(order[:cfg.topn], plain[:cfg.topn]),\n                        vote_first=bool(plan["vote"][0] is not None and sv.max() <= 0 and bonus.max() > 0))\n        if cfg.fp_kinds():\n            nofp = rank_two_tier(lv, sv, fr, ppm, cfg.tau, bonus)\n            diag.update(fp_ok=fz is not None and any(bool(np.isfinite(f).any()) for f in fz),\n                        fp_reordered=fz is not None and not np.array_equal(order[:cfg.topn], nofp[:cfg.topn]))\n            if fz is not None and len(fz) > 1:          # R4: per family, does dropping it change the top 25?\n                w = cfg.fp_w()\n                diag.update(fp_ok_by=[bool(np.isfinite(f).any()) for f in fz],\n                            fp_reordered_by=[not np.array_equal(order[:cfg.topn], rank_two_tier(\n                                lv, sv, fr, ppm, cfg.tau, bonus, fz[:i] + fz[i + 1:], cfg.fp_rrf_k,\n                                w[:i] + w[i + 1:])[:cfg.topn]) for i in range(len(fz))])\n        if cfg.analog_channel:\n            noan = rank_two_tier(lv, sv, fr, ppm, cfg.tau, bonus, fz, cfg.fp_rrf_k, cfg.fp_w())\n            diag.update(an_ok=an is not None and bool(np.isfinite(an).any()),\n                        an_scored=int(np.isfinite(an).sum()) if an is not None else 0,\n                        an_reordered=not np.array_equal(order[:cfg.topn], noan[:cfg.topn]))\n        stage_a = (cand, lv, sv, fr, ppm)\n    if cfg.expand_backfill and not slots.full:\n        for tier in cfg.ppm_expand[1:]:\n            wide = pool.window(target, tier)\n            new = wide[~np.isin(wide, cand)]\n            if len(new):\n                slots.take(pool, new[rank_two_tier(*features(plan, new, pool, sim, cfg, frag_fn), cfg.tau,\n                                                   class_bonus(plan, new, pool, cfg),\n                                                   fp_scores(plan, new, pool, cfg, fp_bank), cfg.fp_rrf_k,\n                                                   cfg.fp_w(), analog_scores(plan, new, pool, cfg, analog_bank),\n                                                   cfg.analog_w)])\n            if slots.full:\n                break\n    if not slots.full:\n        n0 = len(slots.keys)\n        fp_fn = (lambda idx: fp_scores(plan, idx, pool, cfg, fp_bank)) if cfg.fp_stagec and cfg.fp_kinds() else None\n        fill_mass_nearest(slots, pool, target, cfg.backfill_ppm, fp_fn, cfg.fp_w(), cfg.fp_rrf_k)\n        if diag is not None:\n            diag["stagec_slots"] = len(slots.keys) - n0\n    return slots, diag, stage_a\n\n\ndef fill_mass_nearest(slots, pool, target, ppm, fp_fn=None, fp_w=None, rrf_k=3.0):\n    """Stage C top-up: mass-nearest within +-ppm. R4: with fp_fn (pool indices -> list of f.z or None) the window is\n    ordered by the fused FP key sum w_i / (k + rank(f.z_i)) first, |delta mass| second."""\n    wide = pool.window(target, ppm)\n    if not len(wide):\n        return\n    dm = np.abs(pool.mass[wide] - target)\n    fz = fp_fn(wide) if fp_fn is not None else None\n    order = np.lexsort((dm, -fp_rrf(fz, fp_w, rrf_k))) if fz is not None else np.argsort(dm, kind="stable")\n    slots.take(pool, wide[order])\n\n\ndef run(mols, lib, pool, cfg, call=None, deep=None, t0=None, strict=False, frag_fn=frag_scores, lib_exclude=None,\n        vote_label=None, fp_bank=None, analog_bank=None, analog_free=None, log=lambda s: print(s, flush=True)):\n    """Rank every molecule.\n\n    mols: [(molecule_id, sub_df)]. deep: molecule ids that get the full engine (None = all); the rest get\n    mass-nearest +-25 ppm candidates. call: IcebergWorker (None = no simulation). lib_exclude: optional\n    {molecule_id: boolean mask over library spectra} (validation bans). vote_label: {library key: superclass} for\n    the R2 class vote (cfg.class_vote; the pool must carry NPClassifier labels). fp_bank: fpnet.FPBank for the R3\n    FPNet channel (cfg.fp_channel), or a list of banks aligned with cfg.fp_banks (R4 family fusion); logits are\n    computed in pass 1, then the GPU is released before simulation. analog_bank: analog.AnalogBank for the R5\n    analogue channel (cfg.analog_channel); analog_free: optional {molecule_id: keys} that the analogue list scores\n    even though they are train structures (validation: the held-out truth).\n    Returns ({molecule_id: [25 smiles]}, report dict, plans).\n    """\n    t0 = time.time() if t0 is None else t0\n    times, plans, out, diags, stage_a = {}, {}, {}, {}, {}\n    sim_tier1 = cfg.sim_tier1 or strict\n    if cfg.class_vote and (vote_label is None or pool.npc is None):\n        raise ValueError("class_vote needs vote_label and a pool loaded with_npc=True")\n    banks = as_banks(fp_bank)\n    if len(banks) != len(cfg.fp_kinds()):\n        raise ValueError(f"fp banks {[b.kind for b in banks]} do not match cfg.fp_kinds() {cfg.fp_kinds()}")\n    if cfg.analog_channel and analog_bank is None:\n        raise ValueError(f"analog_channel={cfg.analog_channel!r} needs an analog_bank")\n\n    t = time.time()\n    for i, (mid, sub) in enumerate(mols):\n        target = molecule_target(sub, cfg)\n        if not np.isfinite(target) or (deep is not None and mid not in deep):\n            plans[mid] = dict(target=target, deep=False)\n            continue\n        excl = None if lib_exclude is None else lib_exclude.get(mid)\n        plans[mid] = dict(plan_molecule(sub, target, lib, pool, cfg, sim_tier1, frag_fn, excl, vote_label),\n                          deep=True)\n        if cfg.analog_channel:\n            ta = time.time()\n            plans[mid]["an_prof"] = analog_bank.profile(plans[mid].pop("an_hits") or {})\n            plans[mid]["an_free"] = set((analog_free or {}).get(mid, ()))\n            times["analog_s"] = times.get("analog_s", 0.0) + plans[mid]["an_s"] + time.time() - ta\n        if banks:\n            tf, spectra = time.time(), molecule_spectra(sub)\n            plans[mid]["fpz"] = [b.molecule_z(spectra) for b in banks]\n            times["fp_z_s"] = times.get("fp_z_s", 0.0) + time.time() - tf\n        if i % 50 == 0:\n            log(f"  pass 1: {i + 1}/{len(mols)} | {time.time() - t0:.0f}s")\n    deep_ids = [m for m, p in plans.items() if p["deep"]]\n    times["pass1_s"] = round(time.time() - t, 1)\n    if "analog_s" in times:\n        times["analog_s"] = round(times["analog_s"], 1)\n    if banks:\n        times["fp_z_s"] = round(times.get("fp_z_s", 0.0), 1)\n        for b in banks:\n            b.release()\n\n    req = build_requests(({"key": pool.keys[p["sim_idx"]], "smiles": pool.smiles[p["sim_idx"]],\n                           "mass": pool.mass[p["sim_idx"]], "prio": p["prio"], "conds": p["conds"]}\n                          for p in (plans[m] for m in deep_ids)), cfg.instrument, cfg.ce_mode)\n    log(f"[pass 1] {len(deep_ids):,} deep molecules in {times[\'pass1_s\']:.0f}s; {len(req):,} simulation requests")\n\n    t = time.time()\n    if call is not None and len(req):\n        reserve = cfg.pass2_s_per_mol * len(deep_ids) + 300.0\n        sim, sim_stats = simulate(req, call, t0 + cfg.hard_budget_h * 3600.0 - reserve, cfg, strict=strict, log=log)\n    else:\n        sim, sim_stats = {}, dict(n_requests=len(req), n_predicted=0, sim_ok_share=float("nan"), sim_errors={},\n                                  failed_rows=0, capped_rows=len(req) if call is None else 0)\n    times["sim_s"] = round(time.time() - t, 1)\n\n    t = time.time()\n    for mid, _ in mols:\n        p = plans[mid]\n        if p["deep"]:\n            slots, diag, sa = rank_molecule(p, pool, sim, cfg, frag_fn, fp_bank, analog_bank)\n            if diag is not None:\n                diags[mid], stage_a[mid] = diag, sa\n        else:\n            slots = Slots(cfg.topn)\n            if np.isfinite(p["target"]):\n                fill_mass_nearest(slots, pool, p["target"], 25.0)\n        out[mid] = slots.padded()\n    times["pass2_s"] = round(time.time() - t, 1)\n\n    report = dict(times=times, n_molecules=len(mols), n_deep=len(deep_ids), sim_model=cfg.sim_model,\n                  n_covered_spec=int(sum(len(plans[m]["qs"]) for m in deep_ids)),\n                  ce_defaulted=int(sum(plans[m]["n_ce_default"] for m in deep_ids)), **sim_stats,\n                  cand_median=float(np.median([plans[m]["n_cand"] for m in deep_ids])) if deep_ids else float("nan"),\n                  tier1_rate=float(np.mean([d["top_is_tier1"] for d in diags.values()])) if diags else float("nan"),\n                  tier2_plumbing=tier2_plumbing(stage_a, plans, cfg),\n                  class_vote=cfg.class_vote, **vote_counters(diags),\n                  fp_channel=cfg.fp_channel, fp_banks=list(cfg.fp_kinds()), fp_weights=list(cfg.fp_w()),\n                  **fp_counters(diags, plans, deep_ids),\n                  analog_channel=cfg.analog_channel, **analog_counters(diags, plans, deep_ids),\n                  fp_bank=(banks[0].describe() if len(banks) == 1 else [b.describe() for b in banks]) if banks else None)\n    return out, report, dict(plans=plans, diags=diags, stage_a=stage_a, sim=sim)\n\n\ndef vote_counters(diags):\n    """R2 fire counters over the deep molecules (all 0 when the vote is off)."""\n    d = [x for x in diags.values() if "voted" in x]\n    return dict(n_vote_mol=int(sum(x["voted"] for x in d)), n_bonus_reorders=int(sum(x["reordered"] for x in d)),\n                n_vote_first=int(sum(x["vote_first"] for x in d)))\n\n\ndef fp_counters(diags, plans, deep_ids):\n    """R3 fire counters (0 when the channel is off): molecules with logits, with any f.z in stage A, and whose\n    stage-A top 25 differs from the FP-free order."""\n    d = [x for x in diags.values() if "fp_ok" in x]\n    zs = [plans[m].get("fpz") or [] for m in deep_ids]\n    out = dict(n_fp_z=int(sum(any(z is not None for z in zz) for zz in zs)),\n               n_fp_mol=int(sum(x["fp_ok"] for x in d)), n_fp_reorders=int(sum(x["fp_reordered"] for x in d)),\n               n_stagec_mol=int(sum(x.get("stagec_slots", 0) > 0 for x in diags.values())),\n               n_stagec_slots=int(sum(x.get("stagec_slots", 0) for x in diags.values())))\n    nb = max((len(zz) for zz in zs), default=0)\n    if nb > 1:                                          # R4: the same counters per family (order = cfg.fp_banks)\n        out.update(n_fp_z_by=[int(sum(len(zz) > i and zz[i] is not None for zz in zs)) for i in range(nb)],\n                   n_fp_mol_by=[int(sum(x["fp_ok_by"][i] for x in d if "fp_ok_by" in x)) for i in range(nb)],\n                   n_fp_reorders_by=[int(sum(x["fp_reordered_by"][i] for x in d if "fp_reordered_by" in x))\n                                     for i in range(nb)])\n    return out\n\n\ndef analog_counters(diags, plans, deep_ids):\n    """R5 fire counters (0 when the channel is off): molecules with a profile / dropped (< analog_min_n analogues),\n    with any scored candidate in stage A, whose stage-A top 25 the list changed; median analogues per profile."""\n    d = [x for x in diags.values() if "an_ok" in x]\n    profs = [plans[m].get("an_prof") for m in deep_ids if "an_prof" in plans[m]]\n    ns = [p["n"] for p in profs if p is not None]\n    return dict(n_analog_mol=len(ns), n_analog_dropped=sum(p is None for p in profs),\n                n_analog_ok=int(sum(x["an_ok"] for x in d)), n_analog_reorders=int(sum(x["an_reordered"] for x in d)),\n                analog_n_median=float(np.median(ns)) if ns else float("nan"))\n\n\ndef tier2_plumbing(stage_a, plans, cfg, n=20):\n    """Zero the tier-1 winner\'s lv (as if it had no real spectrum): does sv alone keep it in the top 25?"""\n    pos = []\n    for mid in list(stage_a)[:n]:\n        cand, lv, sv, fr, ppm = stage_a[mid]\n        top = rank_two_tier(lv, sv, fr, ppm, cfg.tau)[0]\n        if lv[top] < cfg.tau or not plans[mid]["qs"]:\n            continue\n        lv2 = lv.copy()\n        lv2[top] = 0.0\n        pos.append(int(np.flatnonzero(rank_two_tier(lv2, sv, fr, ppm, cfg.tau) == top)[0]) + 1)\n    return dict(n=len(pos), in_top25=float(np.mean(np.array(pos) <= 25)) if pos else None,\n                median_rank=float(np.median(pos)) if pos else None)\n', 'casmi_scratch/pool.py': '"""Candidate pool: structures sorted by neutral exact mass, one row per ik14."""\nimport json\nimport os\nimport pickle\nimport time\nfrom multiprocessing import Pool as MPool\n\nimport numpy as np\nimport pyarrow.parquet as pq\n\n\nclass CandidatePool:\n    def __init__(self, mass, keys, smiles, flags=None, npc=None):\n        o = np.argsort(np.asarray(mass, np.float64), kind="stable")\n        self.mass = np.asarray(mass, np.float64)[o]\n        self.keys = np.asarray(keys, dtype=object)[o]\n        self.smiles = np.asarray(smiles, dtype=object)[o]\n        self.flags = None if flags is None else np.asarray(flags)[o]   # provenance bits (R1)\n        # NPClassifier superclasses (R2): (codes (n, 3) int16, -1 = none; probs (n, 3); names)\n        self.npc = None if npc is None else (np.asarray(npc[0])[o], np.asarray(npc[1])[o], list(npc[2]))\n\n    def __len__(self):\n        return len(self.mass)\n\n    def bounds(self, target, ppm):\n        """(lo, hi) index range within +-ppm of target."""\n        return (int(np.searchsorted(self.mass, target * (1 - ppm / 1e6), "left")),\n                int(np.searchsorted(self.mass, target * (1 + ppm / 1e6), "right")))\n\n    def window(self, target, ppm):\n        return np.arange(*self.bounds(target, ppm))\n\n    def without(self, keys):\n        """A copy without the given ik14 (validation\'s Class-3 arm)."""\n        keep = ~np.isin(self.keys, np.asarray(list(keys), dtype=object))\n        return CandidatePool(self.mass[keep], self.keys[keep], self.smiles[keep],\n                             None if self.flags is None else self.flags[keep],\n                             None if self.npc is None else (self.npc[0][keep], self.npc[1][keep], self.npc[2]))\n\n\ndef _exact_mass(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.Descriptors import ExactMolWt\n    RDLogger.DisableLog("rdApp.*")\n    m = Chem.MolFromSmiles(smi)\n    return float(ExactMolWt(m)) if m is not None else np.nan\n\n\ndef build_r0_pool(coco_dir, train_path, workers=4):\n    """COCONUT + train structures (the Track 3 base pool, without the unused fingerprints).\n\n    COCONUT rows are deduplicated on ik14 (first occurrence) with their precomputed mass; train structures not in\n    COCONUT get RDKit\'s exact mass from their SMILES; rows without a finite mass are dropped.\n    """\n    t0 = time.time()\n    cm = pickle.load(open(os.path.join(coco_dir, "coco_meta.pkl"), "rb"))\n    co_mass = np.load(os.path.join(coco_dir, "coco_mass.npy")).astype(np.float64)\n    co_k14 = np.array([str(k)[:14] for k in cm["keys"]], dtype=object)\n    _, first = np.unique(co_k14, return_index=True)\n    first = np.sort(first)\n    co_k14, co_smi, co_mass = co_k14[first], np.asarray(cm["smiles"], dtype=object)[first], co_mass[first]\n\n    tr = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles"]).to_pandas()\n    tr = tr.dropna().drop_duplicates("inchikey14")\n    tr = tr[~tr.inchikey14.isin(set(co_k14))]\n    smis = list(tr.normalized_smiles)\n    if workers > 1:\n        with MPool(workers) as mp:\n            tr_mass = np.array(mp.map(_exact_mass, smis, chunksize=500), np.float64)\n    else:\n        tr_mass = np.array([_exact_mass(s) for s in smis], np.float64)\n\n    mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_k14, tr.inchikey14.to_numpy(dtype=object)])\n    smiles = np.concatenate([co_smi, tr.normalized_smiles.to_numpy(dtype=object)])\n    good = np.isfinite(mass)\n    pool = CandidatePool(mass[good], keys[good], smiles[good])\n    print(f"[pool] COCONUT {len(co_k14):,} + train-only {len(tr):,} -> {len(pool):,} structures "\n          f"({time.time() - t0:.1f}s)", flush=True)\n    return pool\n\n\n# ---------------------------------------------------------------- R1: the published unified pool\n# provenance bits of pool_flags.npy (casmi26-scratch-pool); pool_build writes them\nF_TRAIN, F_E180, F_TRAIN_OTHER, F_COCO, F_V2, F_BIO, F_NPT, F_LOTUS, F_NPATLAS = (1 << i for i in range(9))\nFLAG_NAMES = dict(train=F_TRAIN, e180=F_E180, train_other=F_TRAIN_OTHER, coco=F_COCO, v2=F_V2, bio=F_BIO,\n                  np_table=F_NPT, lotus=F_LOTUS, npatlas=F_NPATLAS)\nNP_SOURCES = F_TRAIN_OTHER | F_COCO | F_BIO | F_NPT\n\n\ndef e180_only(flags):\n    """Structures whose only provenance is enveda-180 (synthetic): no other train library, COCONUT, BIO or NP table.\n\n    v2-pool membership does not count: v2 = train + COCONUT, so every enveda-180 structure is in it.\n    """\n    flags = np.asarray(flags, np.int64)\n    return ((flags & F_E180) != 0) & ((flags & NP_SOURCES) == 0)\n\n\ndef load_pool(pool_dir, drop_e180=False, with_npc=False):\n    """casmi26-scratch-pool -> (CandidatePool keyed on scorer_key, info dict). with_npc: attach pool_npc.parquet."""\n    t0 = time.time()\n    mass = np.load(os.path.join(pool_dir, "pool_mass.npy"))\n    keys = np.load(os.path.join(pool_dir, "pool_key.npy")).astype(str).astype(object)\n    flags = np.load(os.path.join(pool_dir, "pool_flags.npy"))\n    smiles = pq.read_table(os.path.join(pool_dir, "pool_smiles.parquet")).column("smiles").to_numpy(\n        zero_copy_only=False).astype(object)\n    assert len(mass) == len(keys) == len(flags) == len(smiles) and np.all(np.diff(mass) >= 0), "pool arrays misaligned"\n    drop = e180_only(flags) if drop_e180 else np.zeros(len(mass), bool)\n    npc = None\n    if with_npc:\n        from .npc import load_npc\n        sc, sp, names = load_npc(pool_dir)\n        npc = (sc[~drop], sp[~drop], names)\n    pool = CandidatePool(mass[~drop], keys[~drop], smiles[~drop], flags[~drop], npc)\n    man = os.path.join(pool_dir, "MANIFEST.json")\n    info = dict(pool_size=len(pool), dropped_e180=int(drop.sum()),\n                pool_version=json.load(open(man)).get("version") if os.path.exists(man) else None)\n    print(f"[pool] R1 {len(pool):,} structures (dropped enveda-180-only {info[\'dropped_e180\']:,}) "\n          f"({time.time() - t0:.1f}s)", flush=True)\n    return pool, info\n\n\ndef load_keymap(pool_dir):\n    """train inchikey14 -> scorer_key (the R1 pool\'s key), for remapping the library."""\n    t = pq.read_table(os.path.join(pool_dir, "train_keymap.parquet")).to_pandas()\n    return dict(zip(t.inchikey14, t.scorer_key))\n\n\ndef load_np_labels(pool_dir):\n    """scorer_key-indexed NP side table (labels, label_source, knn_sim, metadata, np_score)."""\n    return pq.read_table(os.path.join(pool_dir, "pool_np.parquet")).to_pandas().set_index("key")\n\n\ndef npc_top1(pool_dir, keys):\n    """scorer_key -> top-1 NPClassifier superclass, for the given keys (the library\'s: the vote side). Uses the whole\n    pool file, so library structures dropped from R1b keep their label."""\n    from .npc import load_npc\n    sc, _, names = load_npc(pool_dir)\n    allk = np.load(os.path.join(pool_dir, "pool_key.npy")).astype(str)\n    want = set(map(str, keys))\n    return {k: names[c] for k, c in zip(allk, sc[:, 0]) if c >= 0 and k in want}\n', 'casmi_scratch/pool_build.py': '"""R1 unified candidate pool: v2-pool ∪ R0 (COCONUT + train) ∪ BIO (ChEBI / LIPID MAPS) ∪ NP knowledge table.\n\nEvery row is keyed on the scorer_key (InChIKey14 of the RDKit 2026.03.3 canonical tautomer, as the competition\nscorer does it). Duplicates collapse to one row: the SMILES of the highest-priority source wins (train > v2-pool >\nCOCONUT > BIO > NP table), its mass is recomputed with ExactMolWt, and the provenance bits are OR\'d.\nRows RDKit cannot parse or key are dropped; everything else is kept verbatim (no salt stripping).\n\nRuns on Colab (`drivers/r1_pool_colab.py`); the RDKit work is chunked and checkpointed so a restart resumes.\nSpec: specs/specs_scratch/r1_pool_plan.md.\n"""\nimport hashlib\nimport json\nimport os\nimport pickle\nimport time\nfrom multiprocessing import Pool as MPool\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom .pool import (F_BIO, F_COCO, F_E180, F_LOTUS, F_NPATLAS, F_NPT, F_TRAIN, F_TRAIN_OTHER, F_V2, FLAG_NAMES,\n                   e180_only)\n\nE180_LIB = "enveda-180"\nRANK = dict(train=0, v2=1, coco=2, bio=3, np_table=4)        # lower wins the SMILES of a duplicated key\nNP_META = ["np_pathway", "np_superclass", "np_class", "kingdoms", "n_organisms"]\nROW_COLS = ["smiles", "src_ik", "flags", "rank", "source"]\n\n\ndef _rows(smiles, src_ik, flags, source):\n    n = len(smiles)\n    return pd.DataFrame({"smiles": np.asarray(smiles, dtype=object),\n                         "src_ik": np.asarray(src_ik, dtype=object) if src_ik is not None else np.full(n, None, object),\n                         "flags": np.broadcast_to(np.asarray(flags, np.int64), (n,)).copy(),\n                         "rank": RANK[source], "source": source})\n\n\n# ---------------------------------------------------------------- sources -> rows (smiles, src_ik, flags, rank)\n\ndef train_rows(train_path, e180=E180_LIB):\n    """One row per train inchikey14 (first normalized_smiles, as R0), flagged by the libraries it has spectra in."""\n    t = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles", "ingest_lib"]).to_pandas()\n    t = t.dropna(subset=["inchikey14", "normalized_smiles"])\n    lib = t.ingest_lib.astype(str).to_numpy()\n    t = t.assign(e=lib == e180, o=lib != e180)\n    g = t.groupby("inchikey14", sort=False)\n    first = g.normalized_smiles.first()\n    anyf = g[["e", "o"]].any().loc[first.index]\n    flags = F_TRAIN | np.where(anyf.e, F_E180, 0) | np.where(anyf.o, F_TRAIN_OTHER, 0)\n    return _rows(first.to_numpy(dtype=object), first.index.to_numpy(dtype=object), flags, "train")\n\n\ndef v2_rows(pool_meta_path):\n    """casmi26-v2-pool pool_meta.parquet (train + COCONUT, 710,701)."""\n    cols = pq.read_schema(pool_meta_path).names\n    smi = next((c for c in ("smiles", "SMILES", "canonical_smiles", "smi") if c in cols), None)\n    assert smi is not None, f"no SMILES column in {pool_meta_path}: {cols}"\n    t = pq.read_table(pool_meta_path, columns=[smi] + (["key"] if "key" in cols else [])).to_pandas()\n    return _rows(t[smi].to_numpy(dtype=object), t["key"].astype(str).str[:14].to_numpy(dtype=object)\n                 if "key" in cols else None, F_V2, "v2")\n\n\ndef coco_rows(coco_dir):\n    """The phase1 COCONUT arrays R0 uses (coco_meta.pkl keys / smiles)."""\n    cm = pickle.load(open(os.path.join(coco_dir, "coco_meta.pkl"), "rb"))\n    return _rows(np.asarray(cm["smiles"], dtype=object), np.array([str(k)[:14] for k in cm["keys"]], dtype=object),\n                 F_COCO, "coco")\n\n\ndef bio_rows(bio_dir):\n    """chebi-lipidmaps-casmi26 bio_meta.pkl (keys / smiles)."""\n    bm = pickle.load(open(os.path.join(bio_dir, "bio_meta.pkl"), "rb"))\n    return _rows(np.asarray(bm["smiles"], dtype=object), np.array([str(k)[:14] for k in bm["keys"]], dtype=object),\n                 F_BIO, "bio")\n\n\ndef np_table_rows(np_path):\n    """NP knowledge table rows -> (rows, metadata frame aligned to rows by position)."""\n    t = pq.read_table(np_path).to_pandas()\n    flags = F_NPT | np.where(t.in_lotus.astype(bool), F_LOTUS, 0) | np.where(t.in_npatlas.astype(bool), F_NPATLAS, 0)\n    rows = _rows(t.smiles.to_numpy(dtype=object), t.scorer_key.to_numpy(dtype=object), flags, "np_table")\n    return rows, t[NP_META].reset_index(drop=True)\n\n\n# ---------------------------------------------------------------- RDKit: smiles -> key, mass, hygiene\n\n_G = {}\n\n\ndef smiles_props(smi):\n    """-> (scorer_key or None, exact mass or NaN, n_fragments, charged)."""\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.Descriptors import ExactMolWt\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog("rdApp.*")\n    if "te" not in _G:\n        _G["te"] = rdMolStandardize.TautomerEnumerator()\n    m = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None\n    if m is None:\n        return None, np.nan, 0, False\n    try:\n        key = Chem.MolToInchiKey(_G["te"].Canonicalize(m))[:14] or None\n    except Exception:\n        key = None\n    return (key, float(ExactMolWt(m)), len(Chem.GetMolFrags(m)),\n            any(a.GetFormalCharge() != 0 for a in m.GetAtoms()))\n\n\ndef _props_chunk(smis):\n    return [smiles_props(s) for s in smis]\n\n\ndef run_chunked(items, fn, ckpt_dir, name, columns, chunk=50_000, workers=8, initializer=None, initargs=(),\n                log=print):\n    """fn(list of items) -> list of tuples, run over fixed-size chunks with one checkpoint parquet per chunk.\n\n    Chunk files carry a hash of the items and the chunk size, so a rerun with the same items resumes where it\n    stopped and a changed input never mixes with stale chunks. Returns the concatenated frame, in item order.\n    """\n    os.makedirs(ckpt_dir, exist_ok=True)\n    items = list(items)\n    n_chunks = (len(items) + chunk - 1) // chunk\n    tag = hashlib.sha1((chr(10).join(map(str, items)) + f"|{chunk}").encode()).hexdigest()[:10]\n    parts, t0, mp = [], time.time(), None\n    try:\n        for i in range(n_chunks):\n            path = os.path.join(ckpt_dir, f"{name}_{tag}_{i:05d}_{n_chunks:05d}.parquet")\n            if os.path.exists(path):\n                parts.append(pd.read_parquet(path))\n                continue\n            s = items[i * chunk:(i + 1) * chunk]\n            if workers > 1:\n                if mp is None:\n                    mp = MPool(workers, initializer=initializer, initargs=initargs)\n                res = [r for part in mp.map(fn, [list(x) for x in np.array_split(np.asarray(s, dtype=object),\n                                                                                  workers * 8) if len(x)])\n                       for r in part]\n            else:\n                if initializer is not None and not _G.get("_init_" + name):\n                    initializer(*initargs)\n                    _G["_init_" + name] = True\n                res = fn(s)\n            df = pd.DataFrame(res, columns=columns)\n            df.to_parquet(path + ".tmp", index=False)\n            os.replace(path + ".tmp", path)\n            parts.append(df)\n            log(f"  {name} chunk {i + 1}/{n_chunks} | {time.time() - t0:.0f}s")\n    finally:\n        if mp is not None:\n            mp.close()\n            mp.join()\n    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=columns)\n\n\ndef compute_props(smiles, ckpt_dir, chunk=50_000, workers=8, props_fn=_props_chunk, log=print):\n    """Unique SMILES -> DataFrame(index smiles; key, mass, nfrag, charged), checkpointed per chunk."""\n    u = sorted({s for s in smiles if isinstance(s, str)})\n    df = run_chunked(u, props_fn, ckpt_dir, "props", ["key", "mass", "nfrag", "charged"], chunk, workers, log=log)\n    df.index = pd.Index(u, name="smiles")\n    return df\n\n\n# ---------------------------------------------------------------- merge\n\ndef merge_rows(rows, props):\n    """Rows of every source + their props -> (pool frame, stats).\n\n    Pool frame: key, smiles, mass, flags, nfrag, charged, source (winning source), one row per key, sorted by mass.\n    """\n    r = rows.join(props, on="smiles")\n    good = r.key.notna() & np.isfinite(r.mass.astype(np.float64)) & (r.key.astype(str).str.len() == 14)\n    stats = dict(rows_in=r.groupby("source").size().to_dict(),\n                 dropped_unkeyable=r[~good].groupby("source").size().to_dict())\n    r = r[good].copy()\n    has_src = r.src_ik.notna()\n    stats["key_disagree"] = (r[has_src].src_ik.astype(str).str[:14] != r[has_src].key).groupby(\n        r[has_src].source).sum().astype(int).to_dict()\n    r["_pos"] = np.arange(len(r))\n    r = r.sort_values(["key", "rank", "_pos"], kind="stable")\n    k = r.key.to_numpy(dtype=object)\n    starts = np.flatnonzero(np.r_[True, k[1:] != k[:-1]])\n    flags = np.bitwise_or.reduceat(r["flags"].to_numpy(np.int64), starts)\n    win = r.iloc[starts]\n    out = pd.DataFrame({"key": win.key.to_numpy(dtype=object), "smiles": win.smiles.to_numpy(dtype=object),\n                        "mass": win.mass.to_numpy(np.float64), "flags": flags.astype(np.uint16),\n                        "nfrag": win.nfrag.to_numpy(np.int16), "charged": win.charged.to_numpy(bool),\n                        "source": win.source.to_numpy(dtype=object)})\n    out = out.sort_values(["mass", "key"], kind="stable").reset_index(drop=True)\n    stats.update(pool_size=len(out), multi_fragment=int((out.nfrag > 1).sum()), charged=int(out.charged.sum()),\n                 per_flag={n: int(((out["flags"].to_numpy(np.int64) & b) != 0).sum()) for n, b in FLAG_NAMES.items()},\n                 winner_source=out.source.value_counts().to_dict(),\n                 e180_only=int(e180_only(out["flags"]).sum()))\n    return out, stats\n\n\ndef keymap_from_rows(rows, props):\n    """train inchikey14 -> scorer_key (from the train rows\' SMILES)."""\n    t = rows[rows.source == "train"].join(props, on="smiles")\n    t = t[t.key.notna()]\n    return pd.DataFrame({"inchikey14": t.src_ik.astype(str).to_numpy(), "scorer_key": t.key.astype(str).to_numpy()})\n\n\ndef np_meta_by_key(np_rows, np_meta, props):\n    """NP-table metadata per recomputed key (first table row per key) + parity of the table\'s own scorer_key."""\n    k = np_rows.smiles.map(props.key)\n    ok = k.notna().to_numpy()\n    meta = np_meta[ok].assign(key=k[ok].to_numpy(), smiles=np_rows.smiles[ok].to_numpy())\n    parity = dict(n=int(ok.sum()), table_key_differs=int((np_rows.src_ik[ok].to_numpy() != k[ok].to_numpy()).sum()))\n    return meta.drop_duplicates("key").set_index("key"), parity\n\n\ndef superset_check(r0_smiles, props, r1_keys):\n    """R1.1: R0 pool rows whose structure is not in R1 (expected: none).\n\n    An R0 row is lost if RDKit parses it (R0 had a mass for it) but gives no scorer key, or if its key is not in R1.\n    """\n    p = props.reindex(pd.Index(np.asarray(r0_smiles, dtype=object)))\n    parsed = np.isfinite(p.mass.to_numpy(np.float64))\n    keyed = p.key.notna().to_numpy()\n    lost_unkeyable = parsed & ~keyed\n    miss = sorted(set(p.key[keyed]) - set(r1_keys))\n    return dict(r0_rows=int(len(p)), r0_parsed=int(parsed.sum()), r0_keys=int(p.key[keyed].nunique()),\n                lost_unkeyable=int(lost_unkeyable.sum()), missing_keys=len(miss),\n                missing=int(lost_unkeyable.sum()) + len(miss),\n                examples=list(p.index[lost_unkeyable][:5]) + miss[:5])\n\n\n# ---------------------------------------------------------------- write\n\ndef _sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_pool(pool, np_side, keymap, out_dir, manifest):\n    """pool frame (mass-sorted) + NP side table (key-indexed) + keymap -> casmi26-scratch-pool files + MANIFEST."""\n    os.makedirs(out_dir, exist_ok=True)\n    assert np.all(np.diff(pool.mass.to_numpy()) >= 0) and not pool.key.duplicated().any()\n    np.save(os.path.join(out_dir, "pool_mass.npy"), pool.mass.to_numpy(np.float64))\n    np.save(os.path.join(out_dir, "pool_key.npy"), pool.key.to_numpy().astype("S14"))\n    np.save(os.path.join(out_dir, "pool_flags.npy"), pool["flags"].to_numpy(np.uint16))\n    pq.write_table(pa.table({"smiles": pool.smiles.astype(str).to_numpy()}), os.path.join(out_dir, "pool_smiles.parquet"))\n    side = np_side.reindex(pool.key.to_numpy())\n    side.index.name = "key"\n    pq.write_table(pa.Table.from_pandas(side.reset_index(), preserve_index=False),\n                   os.path.join(out_dir, "pool_np.parquet"))\n    pq.write_table(pa.Table.from_pandas(keymap.drop_duplicates("inchikey14"), preserve_index=False),\n                   os.path.join(out_dir, "train_keymap.parquet"))\n    files = sorted(f for f in os.listdir(out_dir) if f != "MANIFEST.json")\n    manifest = dict(manifest, files={f: dict(bytes=os.path.getsize(os.path.join(out_dir, f)),\n                                             sha1=_sha1(os.path.join(out_dir, f))) for f in files},\n                    flag_bits={n: int(b) for n, b in FLAG_NAMES.items()})\n    with open(os.path.join(out_dir, "MANIFEST.json"), "w") as f:\n        json.dump(manifest, f, indent=1, default=str)\n    return manifest\n\n\n# ---------------------------------------------------------------- the whole build (drivers/r1_pool_colab.py)\n\ndef build(inputs, out_dir, ckpt_dir, version, workers=8, log=print):\n    """inputs: dict(train=train.parquet, v2=pool_meta.parquet, coco=dir, bio=dir, np_table=natural_products.parquet).\n\n    Every RDKit stage is checkpointed under ckpt_dir; re-running resumes. -> MANIFEST dict.\n    """\n    from rdkit import __version__ as rdkit_version\n\n    from . import np_labels\n    t0 = time.time()\n    os.makedirs(ckpt_dir, exist_ok=True)\n    rows_path, meta_path = os.path.join(ckpt_dir, "rows.parquet"), os.path.join(ckpt_dir, "np_meta_rows.parquet")\n    if os.path.exists(rows_path) and os.path.exists(meta_path):\n        rows, np_meta = pd.read_parquet(rows_path), pd.read_parquet(meta_path)\n    else:\n        npr, np_meta = np_table_rows(inputs["np_table"])\n        rows = pd.concat([train_rows(inputs["train"]), v2_rows(inputs["v2"]), coco_rows(inputs["coco"]),\n                          bio_rows(inputs["bio"]), npr], ignore_index=True)\n        rows.to_parquet(rows_path, index=False)\n        np_meta.to_parquet(meta_path, index=False)\n    log(f"[1 rows] {rows.groupby(\'source\').size().to_dict()} | {rows.smiles.nunique():,} unique SMILES "\n        f"| {time.time() - t0:.0f}s")\n\n    props = compute_props(rows.smiles, os.path.join(ckpt_dir, "props"), workers=workers, log=log)\n    log(f"[2 keys] {len(props):,} SMILES, unkeyable {int(props.key.isna().sum()):,} | {time.time() - t0:.0f}s")\n\n    pool, stats = merge_rows(rows, props)\n    keymap = keymap_from_rows(rows, props)\n    np_tab, parity = np_meta_by_key(rows[rows.source == "np_table"].reset_index(drop=True), np_meta, props)\n    r1_1 = superset_check(rows.smiles[rows.source.isin(["coco", "train"])], props, pool.key)\n    log(f"[3 merge] {json.dumps(dict(stats, np_parity=parity, R1_1=r1_1), default=str)}")\n    log(f"R1.1 no loss: {\'PASS\' if r1_1[\'missing\'] == 0 else \'FAIL\'} ({r1_1[\'missing\']} R0 rows lost)")\n\n    labels = np_labels.assign_labels(\n        pool.key, pool.smiles, np_tab,\n        nearest=lambda q, b: np_labels.knn_nearest(q, b, os.path.join(ckpt_dir, "knn"), workers=workers, log=log))\n    side = labels.join(np_tab[["kingdoms", "n_organisms"]], how="left")\n    side["np_score"] = np_labels.np_scores(pool.smiles, os.path.join(ckpt_dir, "npscore"), workers=workers,\n                                           log=log).astype(np.float16)\n    side["knn_sim"] = side.knn_sim.astype(np.float16)\n    lstats = dict(label_source={int(k): int(v) for k, v in side.label_source.value_counts().sort_index().items()},\n                  superclass_top=side.np_superclass.value_counts().head(10).to_dict(),\n                  np_score_nan=int(side.np_score.isna().sum()))\n    log(f"[4 labels] {json.dumps(lstats, default=str)} | {time.time() - t0:.0f}s")\n\n    man = write_pool(pool, side, keymap, out_dir, dict(\n        version=version, built=time.strftime("%Y-%m-%d %H:%M"), rdkit=rdkit_version,\n        inputs={k: str(v) for k, v in inputs.items()}, stats=stats, np_parity=parity, R1_1=r1_1, labels=lstats,\n        build_s=round(time.time() - t0, 1)))\n    log(f"[5 write] {out_dir}: {len(pool):,} structures | {time.time() - t0:.0f}s")\n    return man\n', 'casmi_scratch/rank.py': '"""Two-tier ranking and slot filling (stage A -> B -> C), strictly deduplicated on ik14."""\nimport numpy as np\n\n\ndef avg_rank_desc(x):\n    """1-based ranks of x in descending order, ties share their average rank (NaN ranks last, tied)."""\n    x = np.where(np.isnan(x), -np.inf, np.asarray(x, np.float64))\n    o = np.argsort(-x, kind="mergesort")\n    xs = -x[o]\n    r = np.empty(len(x), np.float64)\n    i = 0\n    while i < len(xs):\n        j = i\n        while j + 1 < len(xs) and xs[j + 1] == xs[i]:\n            j += 1\n        r[o[i:j + 1]] = (i + j) / 2.0 + 1.0\n        i = j + 1\n    return r\n\n\ndef fp_rrf(fps, weights, k, idx=None):\n    """Sum of w_i / (k + rank(fp_i)) over the FP channels (R4), restricted to idx. fps: one array or a list of them."""\n    fps = [fps] if isinstance(fps, np.ndarray) else list(fps)\n    weights = [1.0] * len(fps) if weights is None else list(weights)\n    out = 0.0\n    for f, w in zip(fps, weights):\n        f = np.asarray(f, np.float64)\n        out = out + w / (k + avg_rank_desc(f if idx is None else f[idx]))\n    return out\n\n\ndef analog_rrf(score, k, w=1.0):\n    """R5 RRF term w / (k + rank) of the scored candidates (rank among them, ties averaged). Unscored (NaN)\n    candidates get the mean term of the scored ones, so the list only reorders what it scores; none scored -> 0."""\n    score = np.asarray(score, np.float64)\n    ok = np.isfinite(score)\n    out = np.zeros(len(score), np.float64)\n    if ok.any():\n        out[ok] = w / (k + avg_rank_desc(score[ok]))\n        out[~ok] = out[ok].mean()\n    return out\n\n\ndef rank_two_tier(lv, sv, fr, ppm, tau, bonus=None, fp=None, rrf_k=3.0, fp_w=None, an=None, an_w=1.0):\n    """Order (indices) of one candidate set.\n\n    Tier 1 = real-library match lv >= tau, by lv desc (then sv, fr, |ppm|). Tier 2 = the rest by sv (+ bonus) desc,\n    fr desc, |ppm| asc. A missing sv / fr (NaN) counts 0; a missing ppm sorts last. bonus (R2 class vote) only\n    touches tier 2; None = the R1 order. fp (R3, f.z per candidate, NaN = no fingerprint) replaces the tier-2 key\n    by RRF: 1/(k + rank(sv + bonus)) + 1/(k + rank(f.z)), average ranks for ties, within tier 2 only.\n    R4: fp may be a list of f.z arrays (one per FPNet family), each adding fp_w[i] / (k + rank(f.z_i)).\n    R5: an (analogue score per candidate, NaN = not scored) adds analog_rrf(an, k, an_w) to the same tier-2 key.\n    """\n    lv, sv, fr = (np.nan_to_num(np.asarray(x, np.float64), nan=0.0) for x in (lv, sv, fr))\n    ppm = np.nan_to_num(np.asarray(ppm, np.float64), nan=np.inf)\n    s2 = sv if bonus is None else sv + np.nan_to_num(np.asarray(bonus, np.float64), nan=0.0)\n    t1, t2 = np.flatnonzero(lv >= tau), np.flatnonzero(lv < tau)\n    o1 = t1[np.lexsort((ppm[t1], -fr[t1], -sv[t1], -lv[t1]))]\n    if (fp is not None or an is not None) and len(t2):\n        key = 1.0 / (rrf_k + avg_rank_desc(s2[t2]))\n        if fp is not None:\n            key = key + fp_rrf(fp, fp_w, rrf_k, t2)\n        if an is not None:\n            key = key + analog_rrf(np.asarray(an, np.float64)[t2], rrf_k, an_w)\n        o2 = t2[np.lexsort((ppm[t2], -fr[t2], -key))]\n    else:\n        o2 = t2[np.lexsort((ppm[t2], -fr[t2], -s2[t2]))]\n    return np.concatenate([o1, o2]).astype(np.int64)\n\n\nclass Slots:\n    """Up to `topn` (ik14, smiles) picks, never two with the same ik14."""\n\n    def __init__(self, topn):\n        self.topn, self.keys, self.smiles, self.seen = topn, [], [], set()\n\n    @property\n    def full(self):\n        return len(self.smiles) >= self.topn\n\n    def add(self, key, smi):\n        if self.full or key is None:\n            return False\n        k14 = str(key)[:14]\n        if len(k14) != 14 or k14 in self.seen:\n            return False\n        self.seen.add(k14)\n        self.keys.append(k14)\n        self.smiles.append(smi)\n        return True\n\n    def take(self, pool, idx_in_order):\n        for c in idx_in_order:\n            if self.full:\n                break\n            self.add(pool.keys[c], pool.smiles[c])\n\n    def padded(self, pad="CCO"):\n        return self.smiles[:self.topn] + [pad] * max(0, self.topn - len(self.smiles))\n', 'casmi_scratch/resources/__init__.py': '', 'casmi_scratch/resources/glacier_worker.py': '"""GLACIER forward-simulation worker: the iceberg_worker.py protocol on top of the casmi26-glacier gl_runner.py.\n\nRuns in a subprocess (never in the notebook\'s interpreter, which keeps RDKit 2026.03.3) with PYTHONPATH =\n<glacier>/shim, <glacier>/src, the RDKit 2025.03 site built from the casmi26-iceberg wheels. Model loading, SMILES\ncanonicalisation, featurisation, batching (B * N_atoms^2 budget, OOM split, fail-fast) and the post-processing (merge\nidentical m/z by MAX, top 100, max 1) are gl_runner\'s own functions; only the I/O is ours:\n\n    request tsv  : key, smiles, adduct (CASMI spelling), precursor_mz, ce (eV), instrument\n    response npz : off (n+1,) int64, mz / it float32 flat, ok bool, err str, timing json, above_n / above_share /\n                   above_off (peaks dropped above precursor_mz + --prec-margin), as iceberg_worker.py\n\nGLACIER predicts [M+H]+ / [M+Na]+ only (gl_runner.COV); the CE is bucketed to 5 eV (min 5) as the checkpoint was used\nin the A1 panel. Rejected rows get err "rdkit_parse:<why>", "adduct_unsupported:<adduct>", "ce_nan"; a failed\nprediction "predict:none". A systematic failure (gl_runner fail-fast) exits non-zero, so the scheduler retries / degrades.\n\n    python glacier_worker.py --pkg /kaggle/input/.../casmi26-glacier --inp req.tsv --out resp.npz\n"""\nimport argparse\nimport json\nimport os\nimport sys\nimport time\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport pandas as pd\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--pkg", required=True, help="casmi26-glacier dir (gl_runner.py, ckpt/glacier.pt, shim/, src/)")\n    p.add_argument("--inp", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--device", default="cuda")\n    p.add_argument("--batch-size", type=int, default=32)\n    p.add_argument("--sparse-k", type=int, default=100)\n    p.add_argument("--prec-margin", type=float, default=2.0)\n    p.add_argument("--nondeterministic", dest="deterministic", action="store_false")\n    return p.parse_args()\n\n\ndef to_peaks(s, k, prec, margin):\n    """(mz, it) -> (mz, it, above_n, above_share, above_off): iceberg_worker.to_peaks on gl_runner\'s post() output.\n\n    Drops peaks above prec + margin, then keeps the top-k by intensity, sorted by m/z, base = 1.\n    """\n    s = np.stack([np.asarray(s[0], np.float64), np.asarray(s[1], np.float64)], axis=1).reshape(-1, 2)\n    s = s[np.isfinite(s).all(axis=1) & (s[:, 1] > 0)]\n    above = s[:, 0] > prec + margin\n    above_n = int(above.sum())\n    above_share = float(s[above, 1].sum() / s[:, 1].sum()) if len(s) else 0.0\n    above_off = float(s[above, 0].max() - prec) if above_n else np.nan\n    s = s[~above]\n    if len(s) == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32), above_n, above_share, above_off\n    s = s[np.argsort(s[:, 1])[::-1][:k]]\n    s = s[np.argsort(s[:, 0])]\n    return (s[:, 0].astype(np.float32), (s[:, 1] / s[:, 1].max()).astype(np.float32),\n            above_n, above_share, above_off)\n\n\ndef main():\n    args = get_args()\n    if args.deterministic:                                     # before torch / CUDA initialise (gl_runner.main)\n        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    t0 = time.time()\n    pkg = os.path.abspath(args.pkg)\n    sys.path.insert(0, pkg)\n    import gl_runner as gl                                     # noqa: E402  (the casmi26-glacier runner, as a library)\n    log = gl.Log()\n    # site \'none\': the RDKit 2025.03 site is already on PYTHONPATH (sim.setup_glacier); gl_runner\'s own install_site\n    # would look for its .ice_site_ok marker and reinstall the package\'s cp312 RDKit\n    rdkit_v = gl.setup_env(SimpleNamespace(pkg=pkg, site="none", wheels=None, any_rdkit=False), log)\n    import torch                                               # noqa: E402\n    dev = args.device\n    if dev.startswith("cuda") and not torch.cuda.is_available():\n        log("cuda unavailable -> cpu")\n        dev = "cpu"\n    if args.deterministic:\n        torch.use_deterministic_algorithms(True, warn_only=True)\n        torch.backends.cudnn.benchmark = False\n    model = gl.load_model(pkg, dev, log)\n    t_load = time.time() - t0\n\n    df = pd.read_csv(args.inp, sep="\\t")\n    n = len(df)\n    err = [""] * n\n    jobs = [None] * n\n    for i, r in enumerate(df.to_dict("records")):\n        smi = r["smiles"] if isinstance(r["smiles"], str) else ""\n        ok, why = gl.smiles_ok(smi)\n        ce = float(r["ce"])\n        if not ok:\n            err[i] = f"rdkit_parse:{why}"\n        elif r["adduct"] not in gl.COV:\n            err[i] = f"adduct_unsupported:{r[\'adduct\']}"\n        elif not np.isfinite(ce):\n            err[i] = "ce_nan"\n        else:\n            jobs[i] = (smi, r["adduct"], gl.instr_token({"instr": r["instrument"]}), gl.ce_bucket(ce))\n\n    pred = gl.Predictor(model, dev, args.batch_size, log)\n    t1 = time.time()\n    pred.predict({j for j in jobs if j is not None}, float("inf"))\n    if dev.startswith("cuda"):\n        torch.cuda.synchronize()\n    t_pred = time.time() - t1\n\n    empty = (np.empty(0, np.float32), np.empty(0, np.float32))\n    peaks = [empty] * n\n    above_n = np.zeros(n, np.int32)\n    above_share = np.zeros(n, np.float32)\n    above_off = np.full(n, np.nan, np.float32)\n    for i, j in enumerate(jobs):\n        if j is None:\n            continue\n        s = pred.cache.get(j)\n        if s is None:\n            err[i] = "predict:none"\n            continue\n        mz_, it_, above_n[i], above_share[i], above_off[i] = to_peaks(s, args.sparse_k, float(df.precursor_mz.iloc[i]),\n                                                                      args.prec_margin)\n        peaks[i] = (mz_, it_)\n        if len(mz_) == 0:\n            err[i] = "empty_after_prec_cut" if above_n[i] else "empty_spectrum"\n    ok = np.array([len(p[0]) > 0 for p in peaks], bool)\n    lens = np.array([len(p[0]) for p in peaks], dtype=np.int64)\n    off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in peaks]) if n else np.empty(0, np.float32)\n    it = np.concatenate([p[1] for p in peaks]) if n else np.empty(0, np.float32)\n    n_rows = sum(j is not None for j in jobs)\n    timing = dict(model="glacier", n=n, n_predicted=int(ok.sum()), t_load=t_load, t_pred=t_pred,\n                  mol_per_s=float(n_rows / t_pred) if t_pred > 0 and n_rows else None,\n                  n_pred=int(pred.n_pred), n_pred_fail=int(pred.n_fail), bs_final=int(pred.bs), rdkit=rdkit_v,\n                  device=dev, batch_size=args.batch_size, prec_margin=args.prec_margin,\n                  gpu=torch.cuda.get_device_name(0) if dev.startswith("cuda") else "cpu")\n    np.savez(args.out, off=off, mz=mz, it=it, ok=ok, err=np.array(err, dtype="U200"),\n             above_n=above_n, above_share=above_share, above_off=above_off, timing=json.dumps(timing))\n    print(json.dumps(timing))\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_scratch/resources/iceberg_worker.py': '"""ICEBERG forward-simulation worker for specs_simple_ms_simulator.\n\nRuns INSIDE the isolated ms-pred environment (its own torch 2.6 / DGL / rdkit 2025.03), never in\nthe notebook\'s main interpreter -- that keeps RDKit 2026.03.3 (the metric\'s version) untouched.\nThe notebook talks to it through files only:\n\n    request tsv      : key, smiles, adduct (CASMI spelling), precursor_mz, ce, instrument\n                       (TSV, not parquet: the ms-pred environment ships no pyarrow)\n    response npz     : off (n+1,) int64, mz / it float32 flat, ok bool, err str, timing json,\n                       above_n int32 / above_share float32 / above_off float32 (per row, see below)\n\nPeaks are the top `--sparse-k` by intensity, sorted by m/z, base peak = 1.0. m/z values are the\ncharged fragment masses (ICEBERG applies the adduct shift itself).\n\nPeaks above precursor_mz + `--prec-margin` (2 Da) are dropped BEFORE the top-k, the same cleaning the\norganisers applied to the test spectra. They exist because the MassSpecGym inten checkpoints have\n`include_unshifted_mz=True`: every fragment is also emitted without the adduct shift, and for\nwater-loss adducts the unshifted root lands 17-35 Da above the precursor. What was dropped is\nrecorded per row: above_n = peak count, above_share = share of raw intensity, above_off = the\nlargest dropped m/z minus the precursor (NaN when nothing was dropped). A row left with no peaks\ngets err "empty_after_prec_cut".\n\nReused unchanged by Phase 0 (gate), Phase 2 (precompute) and Phase 4 (Kaggle on-the-fly).\n\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --inp req.tsv --out resp.npz\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --hparams-only --out hp.json\n"""\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom rdkit import Chem, RDLogger\n\nimport importlib.abc  # noqa: E402\nimport importlib.machinery  # noqa: E402\nimport os  # noqa: E402\nimport sys  # noqa: E402\nimport types  # noqa: E402\n\nSTUBBED = []\n\n\nclass _Missing:\n    """Stands in for a name from an ms_pred module the package does not ship; fails only if actually used."""\n\n    def __init__(self, name):\n        self._name = name\n\n    def __call__(self, *a, **k):\n        raise RuntimeError(f"{self._name} is not shipped with this ms_pred copy")\n\n    def __getattr__(self, attr):\n        raise RuntimeError(f"{self._name}.{attr} is not shipped with this ms_pred copy")\n\n\nclass _StubMissingMsPred(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    """The casmi26-iceberg package ships a trimmed ms_pred (inference modules only) whose package __init__s still\n    import training-side modules (e.g. nn_utils.tune_utils). Last on sys.meta_path, so it only answers for an\n    ms_pred submodule no real finder can find, with an empty module."""\n\n    def find_spec(self, name, path=None, target=None):\n        return importlib.machinery.ModuleSpec(name, self) if name.startswith("ms_pred.") else None\n\n    def create_module(self, spec):\n        m = types.ModuleType(spec.name)\n        m.__all__ = []\n\n        def __getattr__(attr, _n=spec.name):\n            if attr.startswith("__"):\n                raise AttributeError(attr)\n            return _Missing(f"{_n}.{attr}")\n        m.__getattr__ = __getattr__\n        return m\n\n    def exec_module(self, module):\n        STUBBED.append(module.__name__)\n\n\nclass _Anything:\n    """Permissive placeholder served by an absent third-party module (import-time use only; ice_runner.py\'s)."""\n\n    def __init__(self, *a, **k):\n        pass\n\n    def __call__(self, *a, **k):\n        return _Anything()\n\n    def __getattr__(self, k):\n        if k.startswith("__"):\n            raise AttributeError(k)\n        return _Anything()\n\n    def __getitem__(self, k):\n        return _Anything()\n\n    def __setitem__(self, k, v):\n        pass\n\n    def __iter__(self):\n        return iter(())\n\n    def __bool__(self):\n        return False\n\n\nclass _StubModule(types.ModuleType):\n    def __getattr__(self, k):\n        if k.startswith("__"):\n            raise AttributeError(k)\n        v = type(k, (_Anything,), {}) if k[:1].isupper() else _Anything()\n        setattr(self, k, v)\n        return v\n\n\nclass _StubThirdParty(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    """Serves stub modules (and any submodule) for third-party roots that ms_pred imports at module level but never\n    uses on the predict_mol path, when the image lacks them (multiprocess, h5py, pygmtools on the cp313 image)."""\n\n    def __init__(self, roots):\n        self.roots = set(roots)\n\n    def find_spec(self, name, path=None, target=None):\n        if name.split(".")[0] not in self.roots:\n            return None\n        return importlib.machinery.ModuleSpec(name, self, is_package=True)\n\n    def create_module(self, spec):\n        m = _StubModule(spec.name)\n        m.__path__ = []\n        return m\n\n    def exec_module(self, module):\n        pass\n\n\n# the casmi26-iceberg ice_runner.py list: imported at ms_pred module level, unused by ICEBERG inference\nOPTIONAL = ("pygmtools", "h5py", "multiprocess", "platformdirs")\n\nif os.environ.get("CASMI_ICE_PKG") == "1":\n    sys.meta_path.append(_StubMissingMsPred())\n    _absent = []\n    for _m in OPTIONAL:\n        try:\n            importlib.import_module(_m)\n        except BaseException:\n            for _k in [k for k in sys.modules if k.split(".")[0] == _m]:\n                del sys.modules[_k]\n            _absent.append(_m)\n    if _absent:\n        sys.meta_path.insert(0, _StubThirdParty(_absent))\n        print(f"[worker] absent third-party modules, stubbed: {_absent}", file=sys.stderr, flush=True)\n\nimport ms_pred.common as common  # noqa: E402\nimport ms_pred.common.chem_utils as _chem_utils  # noqa: E402\nfrom ms_pred.iceberg import gen_model, inten_model, joint_model  # noqa: E402\n\nif STUBBED:\n    print(f"[worker] ms_pred modules not shipped, stubbed: {STUBBED}", file=sys.stderr, flush=True)\n\n# predict_mol\'s InChI round trip writes a constant tautomer-transform file under platformdirs\' user_cache_dir:\n# point it at a writable temp dir (also needed when platformdirs itself is stubbed)\nimport tempfile  # noqa: E402\n_CACHE = os.path.join(tempfile.gettempdir(), "ice_ms_pred_cache")\n_chem_utils.user_cache_dir = lambda name, *a, **k: os.path.join(_CACHE, name)\n\nimport rdkit  # noqa: E402\nprint(f"[worker] rdkit {rdkit.__version__} at {os.path.dirname(rdkit.__file__)} | torch {torch.__version__}",\n      file=sys.stderr, flush=True)\nif os.environ.get("CASMI_ICE_PKG") == "1" and not rdkit.__version__.startswith("2025.03"):\n    raise RuntimeError(f"rdkit {rdkit.__version__}: the casmi26-iceberg checkpoints need 2025.03 "\n                       "(2026.03 kekulizes differently) -- the ice site is not first on PYTHONPATH")\n\nRDLogger.DisableLog("rdApp.*")\n\n# CASMI adduct spelling -> ms-pred spelling (only where they differ)\nADDUCT_MAP = {"[M-2H2O+H]+": "[M+H-2H2O]+", "[M+CH2O2-H]-": "[M+CHO2]-"}\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--gen", required=True)\n    p.add_argument("--inten", required=True)\n    p.add_argument("--inp")\n    p.add_argument("--out", required=True)\n    p.add_argument("--device", default="cuda")\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--max-nodes", type=int, default=100)\n    p.add_argument("--sparse-k", type=int, default=100)\n    p.add_argument("--threshold", type=float, default=0.0)\n    p.add_argument("--prec-margin", type=float, default=2.0)\n    p.add_argument("--hparams-only", action="store_true")\n    return p.parse_args()\n\n\ndef _jsonable(d):\n    out = {}\n    for k, v in dict(d).items():\n        try:\n            json.dumps(v)\n            out[k] = v\n        except TypeError:\n            out[k] = str(v)\n    return out\n\n\n# PyTorch 2.6 defaults weights_only=True, which breaks legacy Lightning checkpoints containing PosixPath/hparams.\ntry:\n    from pathlib import PosixPath, WindowsPath\n    torch.serialization.add_safe_globals([PosixPath, WindowsPath])\nexcept Exception:\n    pass\n\n_orig_torch_load = torch.load\ndef _safe_torch_load(*load_args, **load_kwargs):\n    if "weights_only" not in load_kwargs:\n        load_kwargs["weights_only"] = False\n    return _orig_torch_load(*load_args, **load_kwargs)\ntorch.load = _safe_torch_load\n\n\nHP_KEYS = ("hyper_parameters", "hparams", "hp", "config", "kwargs", "args")\nSD_KEYS = ("state_dict", "model_state_dict", "model", "weights")\n\n\ndef load_model(cls, path):\n    """A Lightning checkpoint (.ckpt), or a plain torch file (.pt, e.g. the casmi26-iceberg package): a pickled\n    module, or a dict holding hyper-parameters + state_dict."""\n    if str(path).endswith(".pt"):   # the casmi26-iceberg layout: {hyper_parameters, state_dict}, as ice_runner.py loads it\n        obj = torch.load(path, map_location="cpu")\n        if isinstance(obj, dict) and "hyper_parameters" in obj and "state_dict" in obj:\n            m = cls(**obj["hyper_parameters"])\n            m.load_state_dict(obj["state_dict"], strict=True)\n            return m\n    try:\n        try:\n            return cls.load_from_checkpoint(path, map_location="cpu", weights_only=False)\n        except TypeError:\n            return cls.load_from_checkpoint(path, map_location="cpu")\n    except Exception as e:\n        lightning_err = e\n    obj = torch.load(path, map_location="cpu")\n    if isinstance(obj, torch.nn.Module):\n        return obj\n    if isinstance(obj, dict):\n        hp = next((obj[k] for k in HP_KEYS if k in obj), None)\n        sd = next((obj[k] for k in SD_KEYS if k in obj), None)\n        if isinstance(sd, torch.nn.Module):\n            return sd\n        if hp is not None and sd is not None:\n            m = cls(**dict(vars(hp) if hasattr(hp, "__dict__") and not isinstance(hp, dict) else hp))\n            m.load_state_dict(sd)\n            return m\n    raise RuntimeError(f"cannot load {path} as {cls.__name__}: Lightning said {lightning_err!r}; file holds "\n                       f"{sorted(obj)[:30] if isinstance(obj, dict) else type(obj)}")\n\n\ndef load(args):\n    return load_model(gen_model.FragGNN, args.gen), load_model(inten_model.IntenGNN, args.inten)\n\n\ndef to_peaks(s, k, prec, margin):\n    """(n, 2) array [mz, inten] -> (mz, it, above_n, above_share, above_off).\n\n    Drops peaks above prec + margin, then keeps the top-k by intensity, sorted by m/z, base = 1.\n    """\n    s = np.asarray(s, dtype=np.float64).reshape(-1, 2)\n    s = s[np.isfinite(s).all(axis=1) & (s[:, 1] > 0)]\n    above = s[:, 0] > prec + margin\n    above_n = int(above.sum())\n    above_share = float(s[above, 1].sum() / s[:, 1].sum()) if len(s) else 0.0\n    above_off = float(s[above, 0].max() - prec) if above_n else np.nan\n    s = s[~above]\n    if len(s) == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32), above_n, above_share, above_off\n    s = s[np.argsort(s[:, 1])[::-1][:k]]\n    s = s[np.argsort(s[:, 0])]\n    return (s[:, 0].astype(np.float32), (s[:, 1] / s[:, 1].max()).astype(np.float32),\n            above_n, above_share, above_off)\n\n\ndef predict_batch(model, rows, args):\n    return model.predict_mol(\n        [r["smiles"] for r in rows],\n        collision_eng=[float(r["ce"]) for r in rows],\n        precursor_mz=[float(r["precursor_mz"]) for r in rows],\n        adduct=[r["adduct"] for r in rows],\n        instrument=[r["instrument"] for r in rows],\n        threshold=args.threshold,\n        device=args.device,\n        max_nodes=args.max_nodes,\n        adduct_shift=False,\n        canonical_root_smi=True,\n        binned_out=False,\n    )["spec"]\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    gen, inten = load(args)\n\n    if args.hparams_only:\n        with open(args.out, "w") as f:\n            json.dump({"gen": _jsonable(gen.hparams), "inten": _jsonable(inten.hparams),\n                       "ion2onehot_pos": common.ion2onehot_pos,\n                       "instrument2onehot_pos": common.instrument2onehot_pos}, f, indent=1)\n        return\n\n    model = joint_model.JointModel(gen_model_obj=gen, inten_model_obj=inten).to(args.device)\n    model.eval()\n    model.freeze()\n    t_load = time.time() - t0\n\n    df = pd.read_csv(args.inp, sep="\t")\n    n = len(df)\n    peaks = [None] * n\n    err = [""] * n\n    above_n = np.zeros(n, np.int32)\n    above_share = np.zeros(n, np.float32)\n    above_off = np.full(n, np.nan, np.float32)\n\n    # canonicalise exactly as predict_smis.py does; reject what ms-pred cannot take\n    rows = []\n    for i, r in enumerate(df.to_dict("records")):\n        m = Chem.MolFromSmiles(r["smiles"]) if isinstance(r["smiles"], str) else None\n        ad = ADDUCT_MAP.get(r["adduct"], r["adduct"])\n        if m is None:\n            err[i] = "rdkit_parse"\n        elif ad not in common.ion2onehot_pos:\n            err[i] = f"adduct_unsupported:{r[\'adduct\']}"\n        elif r["instrument"] not in common.instrument2onehot_pos:\n            err[i] = f"instrument_unsupported:{r[\'instrument\']}"\n        elif not np.isfinite(r["ce"]):\n            err[i] = "ce_nan"\n        else:\n            rows.append(dict(r, i=i, smiles=Chem.MolToSmiles(m), adduct=ad))\n\n    t1 = time.time()\n    with torch.no_grad():\n        for b in range(0, len(rows), args.batch_size):\n            batch = rows[b:b + args.batch_size]\n            try:\n                specs = predict_batch(model, batch, args)\n            except Exception:\n                # isolate the failing molecule(s); keep the rest of the batch\n                specs = []\n                for r in batch:\n                    try:\n                        specs.append(predict_batch(model, [r], args)[0])\n                    except Exception as e:  # noqa: BLE001\n                        specs.append(None)\n                        err[r["i"]] = f"predict:{type(e).__name__}:{str(e)[:120]}"\n            for r, s in zip(batch, specs):\n                if s is not None:\n                    i = r["i"]\n                    mz_, it_, above_n[i], above_share[i], above_off[i] = to_peaks(\n                        s.detach().cpu().numpy(), args.sparse_k, float(r["precursor_mz"]), args.prec_margin)\n                    peaks[i] = (mz_, it_)\n    if args.device.startswith("cuda"):\n        torch.cuda.synchronize()\n    t_pred = time.time() - t1\n\n    empty = (np.empty(0, np.float32), np.empty(0, np.float32))\n    ok = np.array([p is not None and len(p[0]) > 0 for p in peaks])\n    for i in range(n):\n        if peaks[i] is not None and len(peaks[i][0]) == 0 and not err[i]:\n            err[i] = "empty_after_prec_cut" if above_n[i] else "empty_spectrum"\n    peaks = [p if p is not None else empty for p in peaks]\n    lens = np.array([len(p[0]) for p in peaks], dtype=np.int64)\n    off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in peaks]) if n else np.empty(0, np.float32)\n    it = np.concatenate([p[1] for p in peaks]) if n else np.empty(0, np.float32)\n    timing = dict(n=n, n_predicted=int(ok.sum()), t_load=t_load, t_pred=t_pred,\n                  mol_per_s=float(len(rows) / t_pred) if t_pred > 0 else None,\n                  device=args.device, batch_size=args.batch_size, prec_margin=args.prec_margin,\n                  gpu=torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu")\n    np.savez(args.out, off=off, mz=mz, it=it, ok=ok, err=np.array(err, dtype="U200"),\n             above_n=above_n, above_share=above_share, above_off=above_off, timing=json.dumps(timing))\n    print(json.dumps(timing))\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_scratch/resources/npc_fp_worker.py': '"""NPClassifier fingerprints under the RDKit the network was trained with.\n\nRDKit changed the Morgan bit assignment between 2021.03.5 and 2021.09.5; NP-Classifier\'s V1 weights expect the old\nbits (R2.0, gate_report). So this runs in its own env (never in the notebook\'s interpreter, which keeps RDKit\n2026.03.3): Python 3.9 + rdkit-pypi 2021.3.5 + numpy, started by `casmi_scratch.npc.fp_subprocess`.\nStandalone: numpy + rdkit only (plus the repo\'s fingerprint_handler when --ref is given).\n\n    in.txt  : one SMILES per line\n    out.npz : F (n, 2048) int16 radius-0 counts, B (n, 4096) int16 radius-1/2 counts, ok (n,) bool, rdkit str,\n              parity int (rows where the repo\'s calculate_fingerprint gives the same arrays; -1 without --ref)\n\n    python npc_fp_worker.py --inp in.txt --out out.npz --workers 8 [--ref DIR_WITH_fingerprint_handler] [--any-rdkit]\n"""\nimport argparse\nimport sys\nfrom multiprocessing import Pool\n\nimport numpy as np\n\nOLD_MAX = (2021, 3)            # last RDKit release with the bits NP-Classifier V1 was trained on\n\n\ndef fingerprint(smiles, radi=2):\n    """NP-Classifier\'s calculate_fingerprint arithmetic -> (formula (2048,), binary (4096,)) or None."""\n    from rdkit import Chem\n    from rdkit.Chem import rdMolDescriptors\n    mol = Chem.MolFromSmiles(smiles) if smiles else None\n    if mol is None or mol.GetNumAtoms() == 0:\n        return None\n    mol = Chem.AddHs(mol)\n    formula, binary = np.zeros(2048, np.int16), np.zeros(2048 * radi, np.int16)\n    for r in range(radi + 1):\n        bi = {}\n        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=r, bitInfo=bi, nBits=2048)\n        for i in fp.GetOnBits():\n            n = sum(1 for e in bi[i] if e[1] == r)\n            if n == 0:\n                continue\n            if r == 0:\n                formula[i] = n\n            else:\n                binary[2048 * (r - 1) + i] = n\n    return formula, binary\n\n\ndef block(smiles):\n    from rdkit import RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    n = len(smiles)\n    F, B, ok = np.zeros((n, 2048), np.int16), np.zeros((n, 4096), np.int16), np.zeros(n, bool)\n    for j, s in enumerate(smiles):\n        fp = fingerprint(s)\n        if fp is not None:\n            F[j], B[j], ok[j] = fp[0], fp[1], True\n    return F, B, ok\n\n\ndef parity(smiles, F, B, ok, ref_dir):\n    sys.path.insert(0, ref_dir)\n    import fingerprint_handler as REF          # the repo\'s own code\n    same = 0\n    for j, s in enumerate(smiles):\n        if ok[j]:\n            a, b = REF.calculate_fingerprint(s, 2)\n            same += bool(np.array_equal(a[0], F[j]) and np.array_equal(b[0], B[j]))\n    return same\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--inp", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--workers", type=int, default=1)\n    ap.add_argument("--ref", default=None)\n    ap.add_argument("--any-rdkit", action="store_true", help="tests only: skip the RDKit version check")\n    a = ap.parse_args()\n\n    import rdkit\n    ver = tuple(int(x) for x in rdkit.__version__.split(".")[:2])\n    if ver > OLD_MAX and not a.any_rdkit:\n        sys.exit(f"npc_fp_worker: RDKit {rdkit.__version__} gives the post-2021.09 Morgan bits; need <= 2021.03")\n    with open(a.inp, encoding="utf-8") as fh:\n        smiles = fh.read().split("\\n")\n    if smiles and smiles[-1] == "":\n        smiles = smiles[:-1]\n    parts = [smiles[i::max(1, a.workers)] for i in range(max(1, a.workers))]\n    if a.workers > 1 and len(smiles) > 1000:\n        with Pool(a.workers) as p:\n            res = p.map(block, parts)\n    else:\n        res = [block(x) for x in parts]\n    n = len(smiles)\n    F, B, ok = np.zeros((n, 2048), np.int16), np.zeros((n, 4096), np.int16), np.zeros(n, bool)\n    for i, (f, b, o) in enumerate(res):                    # undo the strided split\n        F[i::len(parts)], B[i::len(parts)], ok[i::len(parts)] = f, b, o\n    par = parity(smiles, F, B, ok, a.ref) if a.ref else -1\n    np.savez(a.out, F=F, B=B, ok=ok, rdkit=rdkit.__version__, parity=par)\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_scratch/sim.py': '"""Forward simulation (ICEBERG, or GLACIER with `sim_model="glacier"`): request building, the worker subprocess, and\nthe deadline-watchdog scheduler.\n\nThe model runs in its own environment (torch / DGL / RDKit 2025.03: the casmi26-iceberg site, or a venv) through\n`resources/iceberg_worker.py` or `resources/glacier_worker.py`; both speak the same TSV -> NPZ protocol, and this module\nonly writes request TSVs and reads response NPZs.\n"""\nimport glob\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .chem import precursor_of\n\nWORKER_SRC = Path(__file__).resolve().parent / "resources" / "iceberg_worker.py"\nGL_WORKER_SRC = Path(__file__).resolve().parent / "resources" / "glacier_worker.py"\nREQ_COLS = ["key", "smiles", "adduct", "precursor_mz", "ce_ev", "prio", "ce", "instrument"]\nSIM_ADDUCTS = {"glacier": ("[M+H]+", "[M+Na]+")}     # the GLACIER MassSpecGym checkpoint\'s adducts (gl_runner.COV)\n\n\ndef sim_adducts(cfg):\n    """Query adducts that get simulation requests: cfg.covered, restricted to what the model predicts."""\n    allowed = SIM_ADDUCTS.get(cfg.sim_model)\n    return tuple(a for a in cfg.covered if allowed is None or a in allowed)\n\n\ndef to_model_ce(ce_ev, prec, mode):\n    return ce_ev if mode == "ev" else ce_ev * 500.0 / prec\n\n\ndef sim_priority(lv, fr, ppm):\n    """Per-candidate simulation priority (0 = first): lv desc, fr desc, |ppm| asc."""\n    o = np.lexsort((np.asarray(ppm, np.float64), -np.asarray(fr, np.float64), -np.asarray(lv, np.float64)))\n    r = np.empty(len(o), np.int64)\n    r[o] = np.arange(len(o))\n    return r\n\n\ndef build_requests(parts, instrument="QTOF", ce_mode="ev"):\n    """Molecule parts -> one request per distinct (key, adduct, ce_ev), sorted by priority.\n\n    Each part: dict with arrays key / smiles / mass / prio and conds = [(adduct, ce_ev)]. The precursor is the\n    candidate\'s theoretical m/z, so a request is shared by every molecule that needs it (best priority kept).\n    """\n    frames = []\n    for p in parts:\n        if not len(p["key"]) or not p["conds"]:\n            continue\n        mass = np.asarray(p["mass"], np.float64)\n        for adduct, ce in p["conds"]:\n            frames.append(pd.DataFrame({"key": np.asarray(p["key"], dtype=object),\n                                        "smiles": np.asarray(p["smiles"], dtype=object), "adduct": adduct,\n                                        "precursor_mz": precursor_of(mass, adduct), "ce_ev": float(ce),\n                                        "prio": np.asarray(p["prio"], np.int64)}))\n    if not frames:\n        return pd.DataFrame({c: [] for c in REQ_COLS})\n    req = (pd.concat(frames, ignore_index=True)\n           .sort_values(["prio", "key", "adduct", "ce_ev"], kind="stable")\n           .drop_duplicates(["key", "adduct", "ce_ev"], keep="first").reset_index(drop=True))\n    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), ce_mode)\n    req["instrument"] = instrument\n    return req[REQ_COLS]\n\n\ndef rows_that_fit(remaining_s, rate, startup_s):\n    return max(0, int((remaining_s - startup_s) * rate))\n\n\ndef err_counts(err):\n    return pd.Series([str(e).split(":")[0] for e in err if e], dtype=object).value_counts().to_dict()\n\n\ndef sha1(path, chunk=1 << 20, algo="sha1"):\n    h = hashlib.new(algo)\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --- the worker ----------------------------------------------------------------------------------------\nclass IcebergWorker:\n    """Callable(req, tag, batch_size) -> (spectra [(mz, it)], ok, err, timing, wall_s)."""\n    src = WORKER_SRC\n    batch_retry = None              # None: the scheduler retries with cfg.batch_retry\n\n    def __init__(self, py, gen, inten, env, work, device="cuda", sparse_k=100, prec_margin=2.0, batch=128):\n        self.py, self.gen, self.inten, self.env = str(py), str(gen), str(inten), env\n        self._init_common(work, device, sparse_k, prec_margin, batch)\n\n    def _init_common(self, work, device, sparse_k, prec_margin, batch):\n        self.work = Path(work)\n        self.work.mkdir(parents=True, exist_ok=True)\n        self.device, self.sparse_k, self.prec_margin, self.batch = device, sparse_k, prec_margin, batch\n        self.worker = self.work / self.src.name\n        shutil.copyfile(self.src, self.worker)\n\n    def _model_args(self):\n        return ["--gen", self.gen, "--inten", self.inten]\n\n    def _cmd(self, inp, out, batch_size):\n        return [self.py, str(self.worker), *self._model_args(), "--inp", str(inp), "--out", str(out),\n                "--device", self.device, "--batch-size", str(batch_size or self.batch),\n                "--sparse-k", str(self.sparse_k), "--prec-margin", str(self.prec_margin)]\n\n    def __call__(self, req, tag, batch_size=None):\n        inp, out = self.work / f"req_{tag}.tsv", self.work / f"resp_{tag}.npz"\n        req[["key", "smiles", "adduct", "precursor_mz", "ce", "instrument"]].to_csv(inp, sep="\\t", index=False)\n        cmd = self._cmd(inp, out, batch_size)\n        t = time.time()\n        r = subprocess.run(cmd, text=True, capture_output=True, env=self.env, timeout=12 * 3600)\n        if r.returncode:\n            print(r.stdout[-2000:], r.stderr[-4000:])\n            raise RuntimeError(f"worker failed on {tag} (exit {r.returncode}):\\n{r.stderr[-1500:]}")\n        with np.load(out) as z:     # closed before the unlink below (Windows cannot delete an open file)\n            # read each array once: NpzFile re-reads an array on every access (the Phase-1 Colab OOM)\n            off, mz, it = z["off"], z["mz"], z["it"]\n            ok, err, timing = z["ok"], z["err"], json.loads(str(z["timing"]))\n        assert len(off) == len(req) + 1, f"{out}: {len(off) - 1} rows, expected {len(req)}"\n        spectra = [(mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]) for i in range(len(req))]\n        for f in (inp, out):\n            f.unlink(missing_ok=True)\n        return spectra, ok, err, timing, time.time() - t\n\n\nclass GlacierWorker(IcebergWorker):\n    """The same protocol, served by resources/glacier_worker.py on top of the casmi26-glacier gl_runner.py."""\n    src = GL_WORKER_SRC\n\n    def __init__(self, py, pkg, env, work, device="cuda", sparse_k=100, prec_margin=2.0, batch=32, batch_retry=8):\n        self.py, self.pkg, self.env, self.batch_retry = str(py), str(pkg), env, batch_retry\n        self._init_common(work, device, sparse_k, prec_margin, batch)\n\n    def _model_args(self):\n        return ["--pkg", self.pkg]\n\n\nICE_RDKIT = "2025.3.6"     # the RDKit release the casmi26-iceberg package pins (2026.03 kekulizes differently)\n\n\ndef py_tag():\n    return f"cp{sys.version_info.major}{sys.version_info.minor}"\n\n\ndef rdkit_wheels(input_root, prefer=None):\n    """Attached rdkit-*.whl built for this interpreter, the `prefer` release first (sorted otherwise)."""\n    hits = sorted(w for w in glob.glob(f"{input_root}/**/rdkit-*.whl", recursive=True)\n                  if f"-{py_tag()}-" in os.path.basename(w))\n    if prefer:\n        hits.sort(key=lambda w: not os.path.basename(w).startswith(f"rdkit-{prefer}-"))\n    return hits\n\n\ndef _pick_ckpt(pkg, word):\n    hits = sorted(p for p in glob.glob(f"{pkg}/ckpt/**/*", recursive=True)\n                  if p.endswith((".ckpt", ".pt")) and word in os.path.basename(p).lower())\n    if len(hits) != 1:\n        raise FileNotFoundError(f"expected one \'{word}\' checkpoint under {pkg}/ckpt, found {hits} "\n                                f"(all: {sorted(glob.glob(f\'{pkg}/ckpt/**/*\', recursive=True))})")\n    return hits[0]\n\n\ndef preseed_ice_site(pkg, site, input_root):\n    """Install the casmi26-iceberg wheels into `site` (pip --target), swapping the package\'s RDKit wheel for the\n    same release built for this interpreter (the v18 zenith notebook\'s fix for the cp313 Kaggle image)."""\n    srcs = sorted(glob.glob(os.path.join(pkg, "wheels", "*.whl.ice")) + glob.glob(os.path.join(pkg, "wheels", "*.whl")))\n    if not srcs:\n        raise FileNotFoundError(f"no wheels under {pkg}/wheels -- casmi26-iceberg uploaded without sub-directories?")\n    keep = [w for w in srcs if not os.path.basename(w).lower().startswith("rdkit")]\n    new = rdkit_wheels(input_root, prefer=ICE_RDKIT)\n    if not new:\n        raise FileNotFoundError(f"no {py_tag()} rdkit wheel attached (eveliaveldrine/rdkit-wheels)")\n    want = "\\n".join(os.path.basename(w) for w in keep + new[:1])\n    marker = os.path.join(site, ".casmi_site_ok")\n    if os.path.exists(marker) and open(marker).read() == want:\n        return new[0]\n    shutil.rmtree(site, ignore_errors=True)\n    wdir = os.path.join(site, ".wheels")\n    os.makedirs(wdir, exist_ok=True)\n    whl = []\n    for w in keep + new[:1]:\n        dst = os.path.join(wdir, os.path.basename(w)[:-4] if w.endswith(".ice") else os.path.basename(w))\n        shutil.copyfile(w, dst)\n        whl.append(dst)\n    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--no-compile",\n                        "--disable-pip-version-check", "--upgrade", "--target", site, *whl],\n                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    if r.returncode != 0:\n        raise RuntimeError("ICEBERG site install failed:\\n" + (r.stdout or "")[-3000:])\n    shutil.rmtree(wdir, ignore_errors=True)\n    with open(marker, "w") as f:\n        f.write(want)\n    print(f"[ICEBERG] site ready: {len(keep)} package wheels + {os.path.basename(new[0])}", flush=True)\n    return new[0]\n\n\ndef _offline_manifest(input_root):\n    hits = sorted(glob.glob(f"{input_root}/**/wheelhouse", recursive=True), key=len)\n    p = os.path.join(os.path.dirname(hits[0]), "manifest.json") if hits else None\n    return json.load(open(p)) if p and os.path.exists(p) else None\n\n\ndef setup_ice_pkg(pkg, input_root, work, cfg):\n    """ICEBERG from the casmi26-iceberg package (ckpt/ gen + inten .pt or .ckpt, src/, shim/, wheels/*.whl.ice), run by our own worker under\n    the notebook\'s interpreter with shim/, src/ and the preseeded site first on its PYTHONPATH (no venv)."""\n    site = os.path.join(str(work), "ice_site")\n    whl = preseed_ice_site(pkg, site, input_root)\n    gen, inten = _pick_ckpt(pkg, "gen"), _pick_ckpt(pkg, "inten")\n    man = _offline_manifest(input_root)\n    if man is not None:             # casmi26-iceberg-offline attached as well: CE convention must agree\n        assert man["ce_mode"] == cfg.ce_mode and man["instrument"] == cfg.instrument, man\n    print(f"[ICEBERG] casmi26-iceberg {pkg} | rdkit {os.path.basename(whl)} | gen {os.path.basename(gen)} "\n          f"{sha1(gen)[:10]} | inten {os.path.basename(inten)} {sha1(inten)[:10]} | offline manifest "\n          f"{\'CE convention ok\' if man else \'not attached\'}", flush=True)\n    path = [p for p in (os.path.join(pkg, "shim"), os.path.join(pkg, "src"), site) if os.path.isdir(p)]\n    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}\n    env.update(PYTHONPATH=os.pathsep.join(path), DGLBACKEND="pytorch", CUDA_VISIBLE_DEVICES="0", CASMI_ICE_PKG="1")\n    device = "cuda" if shutil.which("nvidia-smi") else "cpu"\n    if device == "cpu":\n        print("[WARNING] no GPU -- ICEBERG on CPU is ~15x slower; the watchdog will cap the simulation", flush=True)\n    return IcebergWorker(sys.executable, gen, inten, env, work, device=device, sparse_k=cfg.sparse_k,\n                         prec_margin=cfg.prec_margin, batch=cfg.batch)\n\n\ndef setup_iceberg(input_root, work, cfg, venv="/kaggle/working/iceberg_venv"):\n    """casmi26-iceberg package if attached (ice_runner.py found), else the casmi26-iceberg-offline wheelhouse in\n    its own venv. Verifies checkpoints, returns an IcebergWorker."""\n    runner = sorted(glob.glob(f"{input_root}/**/ice_runner.py", recursive=True), key=len)\n    if runner:\n        return setup_ice_pkg(os.path.dirname(runner[0]), input_root, work, cfg)\n    hits = [h for h in glob.glob(f"{input_root}/**/wheelhouse", recursive=True) if os.path.isdir(h)]\n    if not hits:\n        raise FileNotFoundError("neither ice_runner.py nor wheelhouse found -- attach casmi26-iceberg (or casmi26-iceberg-offline)")\n    up = os.path.dirname(sorted(hits, key=len)[0])\n    man = json.load(open(os.path.join(up, "manifest.json")))\n    assert man["ce_mode"] == cfg.ce_mode and man["instrument"] == cfg.instrument, man\n    py = f"{venv}/bin/python"\n    site = os.path.join(venv, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")\n    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}\n    env.update(PYTHONPATH=site, DGLBACKEND="pytorch", CUDA_VISIBLE_DEVICES="0")\n    if not os.path.exists(py):\n        shutil.rmtree(venv, ignore_errors=True)\n        subprocess.run([sys.executable, "-m", "venv", "--without-pip", venv], check=True)\n    os.makedirs(site, exist_ok=True)\n    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", site,\n                        *sorted(glob.glob(f"{up}/wheelhouse/*.whl"))], capture_output=True, text=True, env=env)\n    assert r.returncode == 0, r.stderr[-3000:]\n    gen, inten = os.path.join(up, "ckpt", "gen.ckpt"), os.path.join(up, "ckpt", "inten.ckpt")\n    assert sha1(gen) == man["gen_sha1"] and sha1(inten) == man["inten_sha1"], "checkpoint SHA1 mismatch"\n    device = "cuda" if shutil.which("nvidia-smi") else "cpu"\n    if device == "cpu":\n        print("[WARNING] no GPU -- ICEBERG on CPU is ~15x slower; the watchdog will cap the simulation", flush=True)\n    return IcebergWorker(py, gen, inten, env, work, device=device, sparse_k=cfg.sparse_k,\n                         prec_margin=cfg.prec_margin, batch=cfg.batch)\n\n\ndef _find_pkg(input_root, marker, dataset):\n    hits = sorted(glob.glob(f"{input_root}/**/{marker}", recursive=True), key=len)\n    if not hits:\n        raise FileNotFoundError(f"{marker} not found under {input_root} -- attach {dataset}")\n    return os.path.dirname(hits[0])\n\n\ndef setup_glacier(input_root, work, cfg):\n    """GLACIER from casmi26-glacier (gl_runner.py, ckpt/glacier.pt, shim/, src/; no wheels), run by our own worker under\n    the notebook\'s interpreter. The RDKit 2025.03 site comes from the casmi26-iceberg wheels (preseed_ice_site, cp-tag\n    RDKit swap), so casmi26-iceberg and the rdkit wheels stay attached; the ICEBERG checkpoints are not loaded."""\n    assert cfg.ce_mode == "ev", f"GLACIER takes the CE in eV (5 eV buckets), not ce_mode={cfg.ce_mode!r}"\n    pkg = _find_pkg(input_root, "gl_runner.py", "casmi26-glacier")\n    ice = _find_pkg(input_root, "ice_runner.py", "casmi26-iceberg (its wheels build GLACIER\'s RDKit 2025.03 site)")\n    ckpt = os.path.join(pkg, "ckpt", "glacier.pt")\n    if not os.path.exists(ckpt):\n        raise FileNotFoundError(f"{ckpt} missing -- casmi26-glacier uploaded without sub-directories?")\n    site = os.path.join(str(work), "ice_site")\n    whl = preseed_ice_site(ice, site, input_root)\n    man_p = os.path.join(pkg, "MANIFEST.json")\n    want = json.load(open(man_p))["files"]["ckpt/glacier.pt"]["sha256"] if os.path.exists(man_p) else None\n    got = sha1(ckpt, algo="sha256")\n    assert want is None or got == want, f"glacier.pt sha256 {got[:10]} != MANIFEST {want[:10]}"\n    print(f"[GLACIER] casmi26-glacier {pkg} | rdkit {os.path.basename(whl)} | glacier.pt {got[:10]} "\n          f"({\'MANIFEST ok\' if want else \'no MANIFEST\'}) | site from {ice}", flush=True)\n    path = [p for p in (os.path.join(pkg, "shim"), os.path.join(pkg, "src"), site) if os.path.isdir(p)]\n    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}\n    env.update(PYTHONPATH=os.pathsep.join(path), DGLBACKEND="pytorch", CUDA_VISIBLE_DEVICES="0")\n    device = "cuda" if shutil.which("nvidia-smi") else "cpu"\n    if device == "cpu":\n        print("[WARNING] no GPU -- GLACIER on CPU is slow; the watchdog will cap the simulation", flush=True)\n    return GlacierWorker(sys.executable, pkg, env, work, device=device, sparse_k=cfg.sparse_k,\n                         prec_margin=cfg.prec_margin, batch=cfg.gl_batch, batch_retry=cfg.gl_batch_retry)\n\n\ndef setup_sim(input_root, work, cfg, venv="/kaggle/working/iceberg_venv"):\n    """The worker for cfg.sim_model."""\n    if cfg.sim_model == "glacier":\n        return setup_glacier(input_root, work, cfg)\n    assert cfg.sim_model == "iceberg", cfg.sim_model\n    return setup_iceberg(input_root, work, cfg, venv=venv)\n\n\ndef smoke(worker, cfg):\n    """Caffeine [M+H]+ at 40 eV must predict; a 120 eV off-ladder probe is reported."""\n    m = 194.0803756\n    req = pd.DataFrame({"key": ["RYYVLZVUVIJVGH"] * 2, "smiles": ["Cn1cnc2c1c(=O)n(C)c(=O)n2C"] * 2,\n                        "adduct": "[M+H]+", "precursor_mz": precursor_of(np.array([m, m]), "[M+H]+"),\n                        "ce_ev": [40.0, 120.0]})\n    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), cfg.ce_mode)\n    req["instrument"] = cfg.instrument\n    sp, ok, err, timing, _ = worker(req, "smoke")\n    assert ok[0], f"{cfg.sim_model} smoke failed: {err[0]}"\n    return dict(model=cfg.sim_model, gpu=timing.get("gpu"), n_peaks_40=int(len(sp[0][0])),\n                probe_120=dict(ok=bool(ok[1]), n_peaks=int(len(sp[1][0])), err=str(err[1])))\n\n\n# --- the scheduler -------------------------------------------------------------------------------------\ndef simulate(req, call, deadline, cfg, strict=False, log=print):\n    """Run requests in priority-ordered chunks until done or the deadline.\n\n    call: IcebergWorker-like callable. strict: re-raise a chunk that fails twice (fast commit) instead of\n    degrading. Returns (SIM {(key, adduct, ce_ev): (mz, it, prec)}, stats dict).\n    """\n    sim, errs = {}, {}\n    failed = capped = done = ci = 0\n    rate, startup = cfg.rate, cfg.startup_s\n    n_chunks = math.ceil(len(req) / cfg.chunk) if len(req) else 0\n    while done < len(req):\n        fit = rows_that_fit(deadline - time.time(), rate, startup)\n        n = min(cfg.chunk, len(req) - done, fit)\n        if fit < min(500, len(req) - done):\n            capped = len(req) - done\n            log(f"[watchdog] deadline: {capped:,} lowest-priority requests left unsimulated")\n            break\n        part = req.iloc[done:done + n]\n        try:\n            spectra, ok, err, timing, wall = call(part, f"c{ci}")\n        except RuntimeError:\n            try:\n                retry = getattr(call, "batch_retry", None) or cfg.batch_retry\n                spectra, ok, err, timing, wall = call(part, f"c{ci}r", retry)\n            except RuntimeError:\n                if strict:\n                    raise\n                failed += n\n                log(f"[WARNING] chunk {ci} ({n:,} rows) failed twice -- those candidates rank by fr/ppm")\n                done, ci = done + n, ci + 1\n                continue\n        for row, s, o in zip(part.itertuples(), spectra, ok):\n            if o:\n                sim[(row.key, row.adduct, row.ce_ev)] = (s[0], s[1], row.precursor_mz)\n        for k, v in err_counts(err).items():\n            errs[k] = errs.get(k, 0) + v\n        if timing.get("mol_per_s"):\n            rate, startup = float(timing["mol_per_s"]), max(float(wall - timing["t_pred"]), 1.0)\n        done, ci = done + n, ci + 1\n        log(f"  sim chunk {ci}/{n_chunks}: {done:,}/{len(req):,} rows | {rate:.1f}/s | ok {len(sim):,}")\n    attempted = len(req) - capped\n    ok_share = len(sim) / (attempted - failed) if attempted > failed else float("nan")\n    return sim, dict(n_requests=len(req), n_predicted=len(sim), sim_ok_share=ok_share, sim_errors=errs,\n                     failed_rows=failed, capped_rows=capped, rate=rate, startup_s=startup)\n', 'casmi_scratch/spectra.py': '"""Spectrum cleaning and similarity with the config\'s single max_peaks."""\nimport numpy as np\n\nfrom .kernels import clean, entropy_sim\n\nEMPTY = (np.empty(0, np.float32), np.empty(0, np.float32))\n\n\ndef clean_spectrum(mz, it, cfg):\n    return clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                 cfg.int_floor, cfg.max_peaks, cfg.int_power, cfg.ent_weight)\n\n\ndef prep(mz, it, prec, cfg):\n    """Clean a spectrum after dropping peaks above precursor + prec_margin (the organisers\' cleaning)."""\n    mz = np.asarray(mz, np.float32)\n    it = np.asarray(it, np.float32)\n    keep = mz <= prec + cfg.prec_margin\n    return clean(mz[keep], it[keep], cfg.int_floor, cfg.max_peaks, cfg.int_power, cfg.ent_weight)\n\n\ndef esim(a, b, tol):\n    if len(a[0]) == 0 or len(b[0]) == 0:\n        return 0.0\n    return float(entropy_sim(a[0], a[1], b[0], b[1], tol))\n\n\ndef sum_merge(specs, decimals=4):\n    """Sum per-CE spectra (base peak 1 each) the way a merged acquisition does; renormalise to base 1."""\n    parts = [(np.asarray(m, np.float64), np.asarray(i, np.float64)) for m, i in specs if len(m)]\n    if not parts:\n        return EMPTY\n    mz = np.concatenate([m for m, _ in parts])\n    it = np.concatenate([i / i.max() for _, i in parts])\n    _, inv = np.unique(np.round(mz, decimals), return_inverse=True)\n    s = np.bincount(inv, weights=it)\n    m = np.bincount(inv, weights=mz) / np.bincount(inv)\n    o = np.argsort(m)\n    return m[o].astype(np.float32), (s[o] / s.max()).astype(np.float32)\n\n\ndef spectrum_sv(q, sims, cfg):\n    """Simulated-library score of one prepped query spectrum against one candidate.\n\n    sims: the candidate\'s raw predictions (mz, it, prec) at each of the spectrum\'s CEs, None where not simulated.\n    max / mid: best similarity over the CEs; sum: similarity to the renormalised sum. Nothing available -> 0.\n    """\n    have = [s for s in sims if s is not None and len(s[0])]\n    if not have or len(q[0]) == 0:\n        return 0.0\n    if cfg.merge_rule == "sum":\n        mz, it = sum_merge([(m, i) for m, i, _ in have])\n        return esim(q, prep(mz, it, have[0][2], cfg), cfg.mz_tol)\n    return max(esim(q, prep(m, i, p, cfg), cfg.mz_tol) for m, i, p in have)\n\n\nMERGE_RULES = ("max", "sum", "mid")\n\n\ndef ce_tuple(v):\n    """collision_energy_ev cell (list / array / None) -> tuple of floats rounded to 0.1 eV."""\n    if v is None:\n        return ()\n    try:\n        return tuple(round(float(x), 1) for x in v)\n    except TypeError:\n        return ()\n\n\ndef spectrum_ces(ces, rule, default):\n    """One spectrum\'s CE list -> (CEs to simulate, defaulted). CE <= 0 dropped; `mid` simulates the mean once."""\n    assert rule in MERGE_RULES, rule\n    v = [round(float(c), 1) for c in ces if np.isfinite(c) and c > 0]\n    if not v:\n        return [float(default)], True\n    if rule == "mid":\n        return [round(float(np.mean(v)), 1)], False\n    return sorted(set(v)), False\n', 'casmi_scratch/submit.py': '"""Run-mode detection and submission writing."""\nimport os\n\nimport pandas as pd\n\n\ndef run_mode(n_mol, n_spec, cfg, env=None):\n    """-> (fast, info). Fail-safe: full run unless the PUBLIC test file is recognised and this is not the rerun."""\n    env = os.environ if env is None else env\n    raw = env.get("KAGGLE_IS_COMPETITION_RERUN")\n    looks_public = n_mol == cfg.public_n_mol and n_spec == cfg.public_n_spec\n    fast = bool(cfg.allow_fast) and looks_public and not bool(raw) and not cfg.force_full\n    return fast, dict(rerun_env=raw, looks_public=looks_public, mode="fast" if fast else "full")\n\n\ndef make_submission(ranked, sample, topn=25, pad="CCO"):\n    """{molecule_id: [smiles]} aligned to sample_submission, exactly topn per row, audited."""\n    sub = sample[["molecule_id"]].copy()\n    sub["smiles"] = [";".join((list(ranked.get(m, [])) + [pad] * topn)[:topn]) for m in sub.molecule_id]\n    assert len(sub) == len(sample) and (sub.molecule_id.values == sample.molecule_id.values).all()\n    assert not sub.molecule_id.duplicated().any() and sub.smiles.notna().all()\n    assert (sub.smiles.str.split(";").map(len) == topn).all()\n    return sub\n\n\ndef write_submission(sub, path="submission.csv"):\n    sub.to_csv(path, index=False)\n    return path\n\n\ndef molecules(test_df):\n    return [(m, g) for m, g in test_df.groupby("molecule_id")]\n\n\ndef read_sample(path):\n    return pd.read_csv(path)\n'}
PKG_ROOT = WORK / "pkg"
for rel, src in PKG_FILES.items():
    p = PKG_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(src, encoding="utf-8")
sys.path.insert(0, str(PKG_ROOT))


def find_file(name):
    hits = glob.glob(f"{INPUT}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found in {INPUT} -- is the dataset attached?")
    return sorted(hits, key=len)[0]


# RDKit as in the v18 zenith notebook: the attached wheel built for this interpreter (cp tag), 2026.03.3 first if
# one exists, else eveliaveldrine/rdkit-wheels' 2025.3.6. casmi26-iceberg ships its wheels as *.whl.ice, so this
# glob never picks them up.
from casmi_scratch.sim import py_tag, rdkit_wheels  # noqa: E402
whl = rdkit_wheels(INPUT, prefer="2026.3.3")
print(f"[INFO] {py_tag()} rdkit wheels: {whl}")
if whl:
    r_ = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", whl[0]],
                        capture_output=True, text=True)
    print(r_.stdout[-500:], r_.stderr[-500:])
import rdkit  # noqa: E402  (fails loudly: the pool and MetFrag-lite need RDKit)
print(f"[INFO] RDKit {rdkit.__version__}")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import pyarrow.parquet as pq  # noqa: E402

import casmi_scratch  # noqa: E402
from casmi_scratch import pipeline, submit  # noqa: E402
from casmi_scratch.config import Config  # noqa: E402
from casmi_scratch.library import load_library, remap_keys  # noqa: E402
from casmi_scratch.pool import build_r0_pool, load_keymap, load_pool, npc_top1  # noqa: E402
from casmi_scratch.sim import setup_sim, smoke  # noqa: E402

CFG = Config().with_(**CONFIG_OVERRIDES)
COMP = os.path.dirname(find_file("test.parquet"))
TIMES = {}
print(f"[INFO] casmi_scratch {casmi_scratch.__version__} | {RUN_TAG} | {CFG.as_dict()}")

In [ ]:
t = time.time()
WORKER = setup_sim(INPUT, WORK / "sim", CFG, venv=str(WORK / "iceberg_venv")) if CFG.use_sim else None
SMOKE = smoke(WORKER, CFG) if WORKER else None
TIMES["sim_setup_s"] = round(time.time() - t, 1)
print(f"[{CFG.sim_model}] {SMOKE} in {TIMES['sim_setup_s']}s")

In [ ]:
t = time.time()
if CFG.pool_source == "r1":
    POOL_DIR = os.path.dirname(find_file("pool_mass.npy"))
    POOL, POOL_INFO = load_pool(POOL_DIR, drop_e180=CFG.pool_drop_e180, with_npc=CFG.class_vote)
else:
    POOL = build_r0_pool(os.path.dirname(find_file("coco_meta.pkl")), os.path.join(COMP, "train.parquet"),
                         workers=CFG.frag_workers)
    POOL_INFO = dict(pool_size=len(POOL), dropped_e180=0, pool_version="r0-runtime")
TIMES["pool_s"] = round(time.time() - t, 1)
t = time.time()
LIB = load_library(os.path.join(COMP, "train.parquet"), struct_mass=CFG.struct_mass)
if CFG.pool_source == "r1":                 # the R1 pool is keyed on scorer keys: lv must look up the same keys
    LIB = remap_keys(LIB, load_keymap(POOL_DIR))
    print(f"[library] remapped to scorer keys: {len(set(LIB.ik)):,} structures")
VOTE_LABEL = None
if CFG.class_vote:                          # R2: the library side of the vote uses the same NPClassifier labels
    assert CFG.pool_source == "r1", "class_vote needs casmi26-scratch-pool v2"
    VOTE_LABEL = npc_top1(POOL_DIR, set(LIB.ik))
    print(f"[vote] NPClassifier superclass for {len(VOTE_LABEL):,} of {len(set(LIB.ik)):,} library structures")
TIMES["library_s"] = round(time.time() - t, 1)

FP_BANKS, FP_CHECKS = [], []
if CFG.fp_kinds():                          # R3: FPNet channel, fused with sv by RRF inside tier 2; R4: one per family
    from casmi_scratch import audit, fpnet  # noqa: E402
    t = time.time()
    _sl = pq.read_table(os.path.join(COMP, "train.parquet"), columns=audit.SLICE_COLS,
                        filters=[("ingest_lib", "==", audit.SLICE_I_LIB)]).to_pandas()
    _sl = [(k, g.reset_index(drop=True)) for k, g in _sl.groupby("inchikey14")]
    _km = load_keymap(POOL_DIR) if CFG.pool_source == "r1" else {}
    for _kind in CFG.fp_kinds():
        _paths, _bits, _ref = fpnet.find_bank_files(_kind, INPUT)
        _bank = fpnet.load_bank(_kind, _paths, _bits)
        print(f"[fp] {_bank.describe()} | fp_bits {_bits}")
        _r = fpnet.parity_reference(_kind, _ref)
        _c = dict(kind=_kind, parity=None if _r is None else fpnet.featuriser_parity(_bank.feat, *_r),
                  parity_n=0 if _r is None else len(_r[0]))
        _c["self_rank_hit"], _c["self_rank"] = audit.fp_self_rank(_bank, _sl[::max(1, len(_sl) // 40)], POOL,
                                                                  lambda k: _km.get(k, k), CFG)
        print(f"[fp] featuriser parity {_c['parity']}/{_c['parity_n']} ({_kind}) | self-rank top-5 "
              f"{_c['self_rank_hit']}/{len(_c['self_rank'])} {_c['self_rank']}")
        FP_BANKS.append(_bank)
        FP_CHECKS.append(_c)
    TIMES["fp_setup_s"] = round(time.time() - t, 1)
    print(f"[fp] {len(FP_BANKS)} bank(s), weights {CFG.fp_w()} | {TIMES['fp_setup_s']}s")
FP_BANK = FP_BANKS[0] if len(FP_BANKS) == 1 else (FP_BANKS or None)
FP_CHECK = FP_CHECKS[0] if len(FP_CHECKS) == 1 else dict(banks=FP_CHECKS)

AN_BANK, AN_INFO = None, None
if CFG.analog_channel:                      # R5: open analogue hits -> PSSM / Tanimoto kernel, a 3rd tier-2 RRF list
    from casmi_scratch.analog import AnalogBank, estimate_q, train_smiles  # noqa: E402
    from casmi_scratch.pool import F_TRAIN  # noqa: E402
    t = time.time()
    AN_BANK = AnalogBank(train_smiles(os.path.join(COMP, "train.parquet"),
                                      load_keymap(POOL_DIR) if CFG.pool_source == "r1" else None), None, CFG)
    if CFG.analog_channel == "pssm":        # background q over the structures the list scores (non-train pool rows
        _qf = glob.glob(f"{INPUT}/**/analog_q.npy", recursive=True)    # when excluding train, else the whole pool)
        _nontrain = (None if POOL.flags is None or not CFG.analog_exclude_train
                     else (np.asarray(POOL.flags, np.int64) & F_TRAIN) == 0)
        AN_BANK.q = np.load(sorted(_qf, key=len)[0]) if _qf else estimate_q(POOL, AN_BANK.feat, CFG.analog_q_n,
                                                                           mask=_nontrain)
        assert AN_BANK.q.shape == (AN_BANK.feat.raw_bits,), AN_BANK.q.shape
    AN_INFO = dict(n_structures=len(AN_BANK.smiles_of), exclude_train=CFG.analog_exclude_train,
                   q_source=("file" if _qf else "sample") if CFG.analog_channel == "pssm" else None,
                   q_mean=float(AN_BANK.q.mean()) if AN_BANK.q is not None else None)
    TIMES["analog_setup_s"] = round(time.time() - t, 1)
    print(f"[analog] {CFG.analog_channel} | {AN_INFO} | {TIMES['analog_setup_s']}s")

TE = pq.read_table(os.path.join(COMP, "test.parquet")).to_pandas()
SAMPLE = pd.read_csv(os.path.join(COMP, "sample_submission.csv"))
MOLS = submit.molecules(TE)
FAST, MODE = submit.run_mode(len(MOLS), len(TE), CFG)
DEEP = {m for m, _ in MOLS[:CFG.fast_dev_molecules]} if FAST else None
print("=" * 78)
print(f"  {MODE} | {len(TE):,} spectra | {len(MOLS):,} molecules")
print("  *** FAST COMMIT: NOT A REAL SUBMISSION ***" if FAST else "  FULL RUN: submittable")
print("=" * 78)

In [ ]:
RANKED, REPORT, AUX = pipeline.run(MOLS, LIB, POOL, CFG, call=WORKER, deep=DEEP, t0=T0, strict=FAST,
                                   vote_label=VOTE_LABEL, fp_bank=FP_BANK, analog_bank=AN_BANK)

In [ ]:
SUB = submit.make_submission(RANKED, SAMPLE, CFG.topn)
submit.write_submission(SUB, WORK / "submission.csv")
REPORT.update(run_tag=RUN_TAG, mode=MODE["mode"], run_mode=MODE, config=CFG.as_dict(), smoke=SMOKE, setup_times=TIMES,
              total_s=round(time.time() - T0, 1), library_size=len(LIB), rdkit=rdkit.__version__, fp_check=FP_CHECK,
              analog_info=AN_INFO, **POOL_INFO)
(WORK / "run_report.json").write_text(json.dumps(REPORT, indent=1, default=str))
print(json.dumps({k: REPORT[k] for k in ("mode", "sim_model", "n_deep", "n_covered_spec", "times", "total_s",
                                          "n_requests", "sim_ok_share",
                                          "capped_rows", "failed_rows", "tier1_rate", "tier2_plumbing",
                                          "pool_size", "dropped_e180", "cand_median", "class_vote", "n_vote_mol",
                                          "n_bonus_reorders", "n_vote_first", "fp_channel", "n_fp_z", "n_fp_mol",
                                          "n_fp_reorders", "fp_banks", "n_stagec_mol", "n_stagec_slots",
                                          "analog_channel", "n_analog_mol", "n_analog_dropped", "n_analog_ok",
                                          "n_analog_reorders", "analog_n_median")}
                  | {k: REPORT[k] for k in ("n_fp_z_by", "n_fp_mol_by", "n_fp_reorders_by") if k in REPORT},
                 indent=1, default=str))
checks = {
    "sim ok share >= 0.99": (REPORT["sim_ok_share"] >= CFG.min_ok_fast) if CFG.use_sim else None,
    "no capped / failed rows": REPORT["capped_rows"] == 0 and REPORT["failed_rows"] == 0,
    "tier-1 rate >= 0.9 (public file only)": (REPORT["tier1_rate"] >= 0.9) if FAST else None,
    "class vote fires (n_vote_mol, n_bonus_reorders > 0)":
        (REPORT["n_vote_mol"] > 0 and REPORT["n_bonus_reorders"] > 0) if CFG.class_vote else None,
    "R3.run (c) FP fires (n_fp_mol, n_fp_reorders > 0)":
        (REPORT["n_fp_mol"] > 0 and REPORT["n_fp_reorders"] > 0) if CFG.fp_kinds() else None,
    "R4.run (c) every family fires (n_fp_mol_by, n_fp_reorders_by > 0)":
        (min(REPORT["n_fp_mol_by"]) > 0 and min(REPORT["n_fp_reorders_by"]) > 0) if len(FP_BANKS) > 1 else None,
    "R5.0 analogue list on >= 50% of deep molecules":
        (REPORT["n_analog_mol"] >= 0.5 * REPORT["n_deep"]) if CFG.analog_channel else None,
    "R5.0 analogue list fires (n_analog_ok, n_analog_reorders > 0)":
        (REPORT["n_analog_ok"] > 0 and REPORT["n_analog_reorders"] > 0) if CFG.analog_channel else None,
}
for _c in FP_CHECKS:
    checks[f"R3/R4.run (a) {_c['kind']}: featuriser parity >= 45/50 (asserted for ahmed, prv; reported for own)"] = \
        _c["parity"] is not None and _c["parity"] >= 45
    checks[f"R3/R4.run (b) {_c['kind']}: self-rank, truth f.z in the window top 5 for >= 4/5"] = \
        _c["self_rank_hit"] >= 4 and len(_c["self_rank"]) >= 5
for k, v in checks.items():
    print(f"  {'PASS ' if v else ('n/a  ' if v is None else 'CHECK')} {k}")
if FAST and CFG.fp_kinds():
    assert REPORT["n_fp_mol"] > 0 and REPORT["n_fp_reorders"] > 0, REPORT
    if len(FP_BANKS) > 1:
        assert min(REPORT["n_fp_mol_by"]) > 0 and min(REPORT["n_fp_reorders_by"]) > 0, REPORT
    for _c in FP_CHECKS:
        assert _c["self_rank_hit"] >= 4, _c
        if _c["kind"] != "own":
            assert _c["parity"] is not None and _c["parity"] >= 45, _c
if FAST and CFG.analog_channel:
    assert REPORT["n_analog_mol"] > 0, REPORT
if FAST and CFG.use_sim:
    assert REPORT["sim_ok_share"] >= CFG.min_ok_fast, REPORT["sim_errors"]
SUB.head()